# DERM-Net benchmark: Diverse Dermatology Images (DDI)

Benchmarks the DERM-Net architecture — EfficientNet-B4 + ViT-B/16 with multi-scale
channel-attention (MSCA) fusion — against its own ablations and against standard
single-backbone baselines, all trained and evaluated under one protocol on this dataset.

## What is being compared

| Model | Parameters | Role |
|---|---|---|
| **DERM-Net** | ~107M | proposed architecture |
| DERM-Net (no MSCA) | ~105M | ablation: same two backbones, plain concatenation |
| DERM-Net (Eff only) | ~19M | ablation: EfficientNet-B4 alone |
| DERM-Net (ViT only) | ~86M | ablation: ViT-B/16 alone |
| ResNet-50 | ~24M | standard baseline |
| DenseNet-121 | ~7M | standard baseline |

**The row that matters most is "no MSCA".** If DERM-Net beats ResNet-50, that may simply
be two large pretrained backbones rather than the fusion block. Only the comparison
against plain concatenation isolates the contribution the architecture actually claims.

## Why this design

A published number from another paper is not a fair comparator: it differs in dataset,
splits, augmentation, training budget and tuning effort as much as in architecture. Here
every model sees identical splits, identical augmentation, identical schedule and an
identical epoch budget, so differences are attributable to the architecture. Differences
are then tested rather than eyeballed.

## About this dataset

Biopsy-confirmed labels and a deliberately skin-tone-balanced design make this the most trustworthy comparator, despite its small size.

## How to run

1. Attach the dataset (the loader cell names the source).
2. **Settings → Internet → On** — required, for pretrained weights and for `timm`.
3. **Accelerator → GPU T4** — required; on CPU a single fold takes hours.
4. **Run All.** Expect roughly **1-2 hours** on a T4.

The final cells write `benchmark_row_*.json`. Gather those from all four notebooks to
build the cross-dataset table.

## 0. Pre-flight checks

In [ ]:
import sys, subprocess
from pathlib import Path

print("Python:", sys.version.split()[0])
try:
    import timm
    print("  timm        ", timm.__version__)
except ImportError:
    print("  timm         MISSING - installing (needed for EfficientNet-B4 and ViT-B/16)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm
    print("  timm        ", timm.__version__, "(installed)")

import torch, torchvision
print("  torch       ", torch.__version__)
print("  torchvision ", torchvision.__version__)
GPU = torch.cuda.is_available()
print("\nGPU:", torch.cuda.get_device_name(0) if GPU else "NONE - this benchmark needs a GPU")
if not GPU:
    print("\n" + "!" * 72)
    print("No GPU. DERM-Net is ~107M parameters; on CPU a single fold takes hours.")
    print("Kaggle: Settings -> Accelerator -> GPU T4 x2, then Run All again.")
    print("!" * 72)

inp = Path("/kaggle/input")
if inp.exists():
    print("\n/kaggle/input:")
    for p in sorted(inp.iterdir()):
        print("   ", p)

print("\nChecking pretrained weights ...")
WEIGHTS_OK = False
try:
    _ = timm.create_model("efficientnet_b4", pretrained=True, num_classes=0)
    WEIGHTS_OK = True
    print("  OK - pretrained weights available.")
except Exception as exc:
    print(f"  FAILED: {exc}")
    print("\n" + "!" * 72)
    print("STOP. Kaggle notebooks have Internet OFF by default.")
    print("Settings -> Internet -> On, then Run All again.")
    print("Training from random initialisation on a few hundred images produces")
    print("numbers that mean nothing.")
    print("!" * 72)

## 1. Analysis pipeline

Both modules are embedded verbatim. These cells only define functions.

In [ ]:
from __future__ import annotations

import argparse
import json
import logging
import math
import os
import random
import sys
import time
import warnings
import zipfile
from collections import Counter, OrderedDict, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

# Matplotlib must be configured before pyplot import in headless environments.
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import scipy.stats as st
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC

# Tolerate slightly corrupt JPEGs rather than crashing mid-run.
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --------------------------------------------------------------------------- #
# Optional dependencies - every one of these is strictly optional.             #
# --------------------------------------------------------------------------- #
try:  # progress bars
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover - trivial fallback

    def tqdm(iterable=None, **kwargs):  # type: ignore
        return iterable if iterable is not None else []


try:  # torchvision backbones
    import torchvision
    from torchvision import transforms as T

    _HAS_TORCHVISION = True
except Exception:  # pragma: no cover
    torchvision = None  # type: ignore
    T = None  # type: ignore
    _HAS_TORCHVISION = False

try:
    import umap  # type: ignore

    _HAS_UMAP = True
except Exception:
    _HAS_UMAP = False

try:
    import kagglehub  # type: ignore

    _HAS_KAGGLEHUB = True
except Exception:
    _HAS_KAGGLEHUB = False


LOGGER = logging.getLogger("dermgnn")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

# Canonical class vocabulary for this dataset. Folder names on Kaggle vary in
# spacing / underscores / case, so we normalise aggressively rather than
# assuming an exact string match.
CANONICAL_CLASSES: "OrderedDict[str, List[str]]" = OrderedDict(
    [
        (
            "Epidermolysis Bullosa",
            ["epidermolysisbullosa", "epidermolysis", "eb", "bullosa"],
        ),
        ("Ichthyosis", ["ichthyosis", "ichthyoses", "ichtyosis"]),
        ("Hemangiomas", ["hemangiomas", "hemangioma", "haemangioma", "haemangiomas"]),
        (
            "Port-wine stain",
            ["portwinestain", "portwine", "pws", "portwinestains", "nevusflammeus"],
        ),
        (
            "Healthy Skin",
            ["healthyskin", "healthy", "normal", "normalskin", "healthyskins"],
        ),
    ]
)

SPLIT_DIR_NAMES = {
    "train",
    "training",
    "test",
    "testing",
    "val",
    "valid",
    "validation",
    "dev",
    "holdout",
}

### 1.1 CONFIGURATION

In [ ]:
# =========================================================================== #
# SECTION 1 - CONFIGURATION                                                   #
# =========================================================================== #
@dataclass
class Config:
    """Every tunable knob of the study, serialised into results.json."""

    # --- data -------------------------------------------------------------
    data_root: Optional[str] = None
    # CSV manifest with columns path,label[,<group column>]. Used for datasets
    # that are not laid out as one directory per class.
    manifest: Optional[str] = None
    # Column naming the unit that must not straddle a split (lesion, patient).
    group_column: Optional[str] = None
    # Manifest column holding a *recorded* skin-tone measure (Fitzpatrick type,
    # DDI skin_tone group). When present it replaces the ITA proxy for the
    # fairness audit, which is a real measurement rather than an estimate
    # derived from uncalibrated pixels.
    skin_tone_column: Optional[str] = None
    # Cap images per class; 0 disables. Keeps dominant classes from defining the graph.
    max_per_class: int = 0
    download: bool = False
    synthetic: bool = False
    synthetic_n: int = 320
    image_size: int = 224
    # Crop to the central fraction of each image before resizing. Below 1.0
    # this discards background, the material the shortcut probe flags.
    center_crop_frac: float = 1.0
    dedupe: bool = True
    dedupe_hamming: int = 4  # <=4 bits on a 64-bit aHash => near-duplicate
    min_class_size: int = 10

    # --- representation ---------------------------------------------------
    backbone: str = "resnet50"
    feature_tta: bool = True  # horizontal-flip test-time augmentation
    pca_dim: int = 128  # 0 disables PCA whitening
    batch_size: int = 32
    num_workers: int = 2
    # "foldwise" refits the scaler and PCA on each fold's training partition
    # only. "global" fits once on the whole cohort: faster, and label-free, but
    # it lets the test partition's distribution inform the projection, which a
    # careful reviewer will (rightly) call a leak.
    feature_fit: str = "foldwise"
    # Fine-tune the encoder on each fold's training images instead of using
    # frozen ImageNet features. This is the largest legitimate accuracy lever
    # available, and it must run inside the fold or the estimates are invalid.
    finetune: bool = False
    finetune_epochs: int = 30
    finetune_patience: int = 8
    finetune_lr: float = 3e-4
    finetune_batch: int = 32

    # --- graph ------------------------------------------------------------
    knn_k: int = 10
    graph_mode: str = "inductive"  # {"inductive", "transductive"}
    graph_metric: str = "cosine"
    mutual_knn: bool = False
    edge_sim_threshold: float = 0.0

    # --- model ------------------------------------------------------------
    hidden_dim: int = 128
    num_layers: int = 2
    heads: int = 4
    dropout: float = 0.5
    drop_edge: float = 0.10

    # --- optimisation -----------------------------------------------------
    lr: float = 5e-3
    weight_decay: float = 5e-4
    epochs: int = 300
    patience: int = 50
    focal_gamma: float = 2.0
    label_smoothing: float = 0.05
    use_class_weights: bool = True

    # --- evaluation -------------------------------------------------------
    folds: int = 5
    repeats: int = 5
    val_fraction: float = 0.15  # carved out of the training portion
    bootstrap: int = 2000
    alpha: float = 0.05
    calibrate: bool = True

    # --- study scope ------------------------------------------------------
    models: List[str] = field(
        default_factory=lambda: [
            "LogisticRegression",
            "SVM-RBF",
            "RandomForest",
            "MLP",
            "GCN",
            "GraphSAGE",
            "GAT",
            "DERM-GNN",
        ]
    )
    reference_model: str = "DERM-GNN"
    ablations: bool = False
    fairness: bool = True
    explain: bool = True
    # Shortcut-learning probe: re-runs the reference model on images whose
    # centre has been masked out, leaving only background/periphery. Accuracy
    # meaningfully above chance means the model can exploit acquisition context
    # rather than the lesion itself.
    shortcut_test: bool = True
    shortcut_mask_fraction: float = 0.60  # side length of the masked centre square

    # --- infrastructure ---------------------------------------------------
    outdir: str = "results"
    seed: int = 42
    device: str = "auto"
    quick: bool = False
    dpi: int = 600

    def resolve(self) -> "Config":
        """Apply interdependent defaults (e.g. --quick) and validate."""
        if self.quick:
            self.repeats = 1
            self.folds = 3
            self.epochs = 40
            self.patience = 15
            self.bootstrap = 200
            self.ablations = False
        if self.graph_mode not in {"inductive", "transductive"}:
            raise ValueError(f"invalid --graph-mode: {self.graph_mode}")
        if self.feature_fit not in {"foldwise", "global"}:
            raise ValueError(f"invalid --feature-fit: {self.feature_fit}")
        if not 0.0 < self.center_crop_frac <= 1.0:
            raise ValueError("--center-crop-frac must lie in (0, 1]")
        if not 0.0 < self.shortcut_mask_fraction < 1.0:
            raise ValueError("--shortcut-mask-fraction must lie in (0, 1)")
        if self.knn_k < 1:
            raise ValueError("--knn-k must be >= 1")
        if not 0.0 < self.val_fraction < 0.5:
            raise ValueError("--val-fraction must lie in (0, 0.5)")
        if self.folds < 2:
            raise ValueError("--folds must be >= 2")
        bad_bb = [n.strip() for n in self.backbone.split(",")
                  if n.strip() and n.strip() not in BACKBONE_REGISTRY]
        if bad_bb:
            raise ValueError(f"unknown backbone(s): {bad_bb}; choose from {list(BACKBONE_REGISTRY)}")
        unknown = [m for m in self.models if m not in ALL_MODELS]
        if unknown:
            raise ValueError(f"unknown model(s): {unknown}; choose from {ALL_MODELS}")
        if self.reference_model not in self.models:
            self.reference_model = self.models[-1]
        return self


ALL_MODELS = [
    "LogisticRegression",
    "SVM-RBF",
    "RandomForest",
    "MLP",
    "GCN",
    "GraphSAGE",
    "GAT",
    "DERM-GNN",
]
GRAPH_MODELS = {"GCN", "GraphSAGE", "GAT", "DERM-GNN"}
TORCH_MODELS = GRAPH_MODELS | {"MLP"}

### 1.2 REPRODUCIBILITY, LOGGING, SMALL UTILITIES

In [ ]:
# =========================================================================== #
# SECTION 2 - REPRODUCIBILITY, LOGGING, SMALL UTILITIES                       #
# =========================================================================== #
def set_seed(seed: int) -> None:
    """Seed every RNG we touch and request deterministic cuDNN kernels."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def setup_logging(outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)
    fmt = "%(asctime)s | %(levelname)-7s | %(message)s"
    datefmt = "%H:%M:%S"
    LOGGER.setLevel(logging.INFO)
    LOGGER.handlers.clear()
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(logging.Formatter(fmt, datefmt))
    LOGGER.addHandler(sh)
    fh = logging.FileHandler(outdir / "run.log", mode="w", encoding="utf-8")
    fh.setFormatter(logging.Formatter(fmt, datefmt))
    LOGGER.addHandler(fh)


def resolve_device(spec: str) -> torch.device:
    if spec == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(spec)


def banner(title: str) -> None:
    LOGGER.info("=" * 78)
    LOGGER.info(title)
    LOGGER.info("=" * 78)


def json_safe(obj: Any) -> Any:
    """Recursively coerce numpy/torch scalars so json.dump never fails."""
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if (math.isnan(val) or math.isinf(val)) else val
    if isinstance(obj, np.ndarray):
        return json_safe(obj.tolist())
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, torch.Tensor):
        return json_safe(obj.detach().cpu().numpy())
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, float):
        return None if (math.isnan(obj) or math.isinf(obj)) else obj
    return obj


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(json_safe(obj), fh, indent=2, ensure_ascii=False)


def savefig(fig: plt.Figure, outdir: Path, name: str, dpi: int = 600) -> None:
    """Persist every figure twice: raster for preview, vector for submission."""
    figdir = outdir / "figures"
    figdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figdir / f"{name}.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(figdir / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    LOGGER.info("  figure saved: figures/%s.{png,pdf}", name)


def set_publication_style() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
            "font.family": "sans-serif",
            "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": True,
            "grid.alpha": 0.25,
            "grid.linewidth": 0.5,
            "legend.frameon": False,
            "legend.fontsize": 8,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "lines.linewidth": 1.6,
            "pdf.fonttype": 42,  # editable text in Illustrator
            "ps.fonttype": 42,
        }
    )


# Colour-blind-safe qualitative palette (Okabe-Ito), used consistently.
PALETTE = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
    "#56B4E9",
    "#F0E442",
    "#000000",
]


def model_color(name: str) -> str:
    return PALETTE[ALL_MODELS.index(name) % len(PALETTE)] if name in ALL_MODELS else "#666666"


def fmt_ci(mean: float, lo: float, hi: float, pct: bool = True) -> str:
    """Render 'estimate (low-high)'. Percentages get one decimal; unit-scale
    quantities (Brier, ECE, MCC, kappa) get three, or they round away."""
    if mean is None or (isinstance(mean, float) and math.isnan(mean)):
        return "n/a"
    scale, dp = (100.0, 1) if pct else (1.0, 3)
    if any(v is None or (isinstance(v, float) and math.isnan(v)) for v in (lo, hi)):
        return f"{mean * scale:.{dp}f}"
    # "to" rather than a hyphen: metrics such as MCC and kappa can be negative,
    # where "-0.076--0.060" would be unreadable.
    return f"{mean * scale:.{dp}f} ({lo * scale:.{dp}f} to {hi * scale:.{dp}f})"

### 1.3 DATA ACQUISITION, DISCOVERY AND QUALITY CONTROL

In [ ]:
# =========================================================================== #
# SECTION 3 - DATA ACQUISITION, DISCOVERY AND QUALITY CONTROL                 #
# =========================================================================== #
def normalise_label(raw: str) -> Optional[str]:
    """Map an arbitrary folder name onto the canonical class vocabulary."""
    key = "".join(ch for ch in raw.lower() if ch.isalnum())
    if not key:
        return None
    for canonical, aliases in CANONICAL_CLASSES.items():
        canon_key = "".join(ch for ch in canonical.lower() if ch.isalnum())
        if key == canon_key or key in aliases:
            return canonical
    # Substring fallback, longest alias first to avoid spurious short matches.
    for canonical, aliases in CANONICAL_CLASSES.items():
        for alias in sorted(aliases, key=len, reverse=True):
            if len(alias) >= 4 and alias in key:
                return canonical
    return None


def download_dataset(dest: Path) -> Path:
    """Fetch the Kaggle dataset via kagglehub, with a clear error if unavailable."""
    if not _HAS_KAGGLEHUB:
        raise RuntimeError(
            "--download requires kagglehub. Install it with `pip install kagglehub` "
            "and configure Kaggle credentials (~/.kaggle/kaggle.json or the "
            "KAGGLE_USERNAME / KAGGLE_KEY environment variables). Alternatively, "
            "download archive.zip manually and pass --data-root."
        )
    LOGGER.info("Downloading roshni2404/rare-skin-disease-dataset via kagglehub ...")
    path = Path(kagglehub.dataset_download("roshni2404/rare-skin-disease-dataset"))
    LOGGER.info("Dataset cached at: %s", path)
    return path


def maybe_extract_zip(root: Path) -> Path:
    """If --data-root points at (or contains) archive.zip, extract it once."""
    if root.is_file() and root.suffix.lower() == ".zip":
        target = root.parent / (root.stem + "_extracted")
        if not target.exists():
            LOGGER.info("Extracting %s -> %s", root.name, target)
            with zipfile.ZipFile(root) as zf:
                zf.extractall(target)
        return target
    if root.is_dir():
        zips = sorted(root.glob("*.zip"))
        has_images = any(
            p.suffix.lower() in IMAGE_EXTENSIONS for p in root.rglob("*") if p.is_file()
        )
        if zips and not has_images:
            target = root / (zips[0].stem + "_extracted")
            if not target.exists():
                LOGGER.info("Extracting %s -> %s", zips[0].name, target)
                with zipfile.ZipFile(zips[0]) as zf:
                    zf.extractall(target)
            return target
    return root


def make_synthetic_dataset(dest: Path, n_total: int, seed: int) -> Path:
    """
    Build a small, deliberately *learnable* stand-in corpus so the entire
    pipeline (features -> graph -> GNN -> statistics -> figures -> report) can
    be validated without the real data. Class-imbalanced on purpose, mirroring
    the real cohort's rare-disease skew.
    """
    rng = np.random.default_rng(seed)
    if dest.exists() and any(dest.rglob("*.png")):
        return dest
    LOGGER.info("Generating synthetic smoke-test corpus at %s", dest)
    weights = np.array([0.14, 0.18, 0.26, 0.16, 0.26])
    counts = np.maximum((weights * n_total).astype(int), 12)
    size = 96
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32) / size

    for ci, (cls, n) in enumerate(zip(CANONICAL_CLASSES.keys(), counts)):
        cdir = dest / cls.replace(" ", "_")
        cdir.mkdir(parents=True, exist_ok=True)
        # Each class gets a distinct hue + texture signature.
        base_hue = np.array([[0.75, 0.45, 0.40], [0.80, 0.72, 0.60], [0.85, 0.30, 0.32],
                             [0.62, 0.35, 0.55], [0.70, 0.55, 0.45]])[ci]
        for i in range(int(n)):
            # Constitutive pigmentation varies per image (skin-tone diversity).
            tone = rng.uniform(0.35, 1.0)
            img = np.ones((size, size, 3), np.float32) * base_hue * tone
            freq = 4.0 + 3.0 * ci
            texture = 0.12 * np.sin(2 * np.pi * freq * xx + ci) * np.cos(
                2 * np.pi * freq * yy
            )
            img += texture[..., None]
            # Class-specific lesion geometry.
            cx, cy = rng.uniform(0.3, 0.7, 2)
            rad = 0.10 + 0.03 * ci
            mask = ((xx - cx) ** 2 + (yy - cy) ** 2) < rad ** 2
            if ci != 4:  # "Healthy Skin" gets no lesion
                img[mask] = img[mask] * 0.45 + base_hue[::-1] * 0.55
            img += rng.normal(0, 0.035, img.shape).astype(np.float32)
            arr = (np.clip(img, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(arr).save(cdir / f"{cls.replace(' ', '_')}_{i:04d}.png")
    return dest


def average_hash(path: Path, hash_size: int = 8) -> Optional[int]:
    """
    64-bit perceptual average hash. Used only for near-duplicate detection;
    deliberately cheap so it can run over the whole corpus.
    """
    try:
        with Image.open(path) as im:
            im = im.convert("L").resize((hash_size, hash_size), Image.BILINEAR)
            arr = np.asarray(im, dtype=np.float32)
    except Exception:
        return None
    bits = (arr > arr.mean()).flatten()
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


def load_manifest_dataset(cfg: Config, manifest_path: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Build the cohort from an explicit CSV manifest instead of folder structure.

    Required columns: `path` and `label`. An optional grouping column (set with
    --group-column) names the unit that must not be split across folds - the
    lesion in HAM10000, the patient elsewhere. Honouring it is what separates a
    defensible estimate from one inflated by multiple images of the same lesion
    appearing on both sides of a split.

    This is the entry point used for HAM10000, Fitzpatrick17k and DDI, none of
    which are laid out as one directory per class.
    """
    flow: Dict[str, Any] = {"source": "manifest", "manifest": str(manifest_path)}
    df = pd.read_csv(manifest_path)
    missing = [c for c in ("path", "label") if c not in df.columns]
    if missing:
        raise ValueError(f"manifest {manifest_path} is missing column(s): {missing}")

    df["path"] = df["path"].astype(str)
    df["label"] = df["label"].astype(str).str.strip()
    flow["rows_in_manifest"] = len(df)

    group_col = cfg.group_column if cfg.group_column and cfg.group_column in df.columns else None
    if cfg.group_column and group_col is None:
        LOGGER.warning("Group column '%s' not present in the manifest; falling back to "
                       "image-level splitting.", cfg.group_column)
    if group_col:
        df["group"] = df[group_col].astype(str)
        LOGGER.info("Grouping by '%s': %d unique groups over %d images",
                    group_col, df["group"].nunique(), len(df))
    else:
        df["group"] = [f"img{i}" for i in range(len(df))]

    # astype(bool) matters: on an empty frame the mapped Series has object dtype,
    # and ~obj.sum() yields '' rather than 0, which then fails to cast to int.
    exists = df["path"].map(lambda p: Path(p).is_file()).astype(bool)
    flow["missing_files"] = int(len(df) - int(exists.sum()))
    if flow["missing_files"] == len(df):
        raise FileNotFoundError(
            f"None of the {len(df)} paths in {manifest_path} resolve to a file. The usual "
            f"cause is a join key that pandas parsed as a number, dropping leading zeros - "
            f"read the metadata CSV with dtype=str. First path tried: "
            f"{df['path'].iloc[0] if len(df) else 'n/a'}"
        )
    if flow["missing_files"]:
        LOGGER.warning("%d manifest row(s) point at files that do not exist; dropping them.",
                       flow["missing_files"])
    df = df[exists].reset_index(drop=True)
    if df.empty:
        raise FileNotFoundError(f"No readable images referenced by {manifest_path}.")

    df["filename"] = df["path"].map(lambda p: Path(p).name)
    df["source_split"] = df.get("source_split", "all")
    flow["files_found"] = len(df)
    flow["files_unmapped"] = 0
    return df, flow


def _finalise_cohort(cfg: Config, df: pd.DataFrame, flow: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """QC, optional subsampling and class indexing shared by both entry points."""
    # ---- QC 1: exact path duplicates -------------------------------------
    before = len(df)
    df = df.drop_duplicates(subset="path").reset_index(drop=True)
    flow["removed_exact_path_duplicates"] = before - len(df)

    # ---- QC 2: unreadable / degenerate images ----------------------------
    keep, widths, heights = [], [], []
    for path in tqdm(df["path"].tolist(), desc="QC: verifying images", leave=False):
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                w, h = im.size
                im.convert("RGB")
            ok = w >= 24 and h >= 24
        except Exception:
            ok, w, h = False, 0, 0
        keep.append(ok)
        widths.append(w)
        heights.append(h)
    df["width"], df["height"] = widths, heights
    flow["removed_unreadable"] = int((~np.array(keep)).sum())
    df = df[np.array(keep)].reset_index(drop=True)

    # ---- QC 3: perceptual near-duplicate removal -------------------------
    if cfg.dedupe and len(df) > 1:
        hashes = [
            average_hash(Path(p))
            for p in tqdm(df["path"].tolist(), desc="QC: perceptual hashing", leave=False)
        ]
        df["ahash"] = hashes
        keep_mask = np.ones(len(df), dtype=bool)
        groups_arr = df["group"].to_numpy() if "group" in df else np.arange(len(df)).astype(str)
        has_groups = "group" in df and df["group"].nunique() < len(df)
        kept: List[Tuple[int, int]] = []
        for i, h in enumerate(hashes):
            if h is None:
                continue
            dup = False
            for kh, ki in kept:
                if bin(kh ^ h).count("1") > cfg.dedupe_hamming:
                    continue
                # Several views of the same lesion are legitimate data, not
                # redundancy: HAM10000 ships multiple dermoscopic frames per
                # lesion. Only drop a near-duplicate that belongs to a
                # *different* group, which is the case that leaks across folds.
                if has_groups and groups_arr[ki] == groups_arr[i]:
                    continue
                dup = True
                break
            if dup:
                keep_mask[i] = False
            else:
                kept.append((h, i))
        if has_groups:
            LOGGER.info("Near-duplicate removal restricted to cross-group pairs "
                        "(repeat views of one group are retained).")
        flow["removed_near_duplicates"] = int((~keep_mask).sum())
        if flow["removed_near_duplicates"]:
            LOGGER.info("Removed %d near-duplicate image(s) (aHash Hamming <= %d)",
                        flow["removed_near_duplicates"], cfg.dedupe_hamming)
        df = df[keep_mask].reset_index(drop=True)
    else:
        flow["removed_near_duplicates"] = 0

    # ---- QC 4: optional class subsampling --------------------------------
    # HAM10000 is dominated by one class (nv, ~67%). Capping per class keeps
    # runtime tractable and stops a single class defining the graph.
    if cfg.max_per_class and cfg.max_per_class > 0:
        rng = np.random.default_rng(cfg.seed)
        keep_idx: List[int] = []
        for cls, sub in df.groupby("label"):
            idx = sub.index.to_numpy()
            if len(idx) > cfg.max_per_class:
                # Sample whole groups so a lesion is never partially retained.
                groups = sub["group"].unique()
                rng.shuffle(groups)
                chosen, n = [], 0
                for g in groups:
                    gi = sub.index[sub["group"] == g].to_numpy()
                    if n + len(gi) > cfg.max_per_class and n > 0:
                        continue
                    chosen.extend(gi.tolist())
                    n += len(gi)
                    if n >= cfg.max_per_class:
                        break
                idx = np.array(chosen)
            keep_idx.extend(idx.tolist())
        flow["removed_subsampling"] = int(len(df) - len(keep_idx))
        if flow["removed_subsampling"]:
            LOGGER.info("Subsampled to at most %d image(s) per class (removed %d)",
                        cfg.max_per_class, flow["removed_subsampling"])
        df = df.loc[sorted(keep_idx)].reset_index(drop=True)
    else:
        flow["removed_subsampling"] = 0

    # ---- QC 5: drop classes too small to cross-validate ------------------
    counts = df["label"].value_counts()
    min_needed = max(cfg.min_class_size, cfg.folds)
    too_small = counts[counts < min_needed].index.tolist()
    if too_small:
        LOGGER.warning("Dropping %d class(es) with < %d images after QC: %s",
                       len(too_small), min_needed,
                       too_small[:10] + (["..."] if len(too_small) > 10 else []))
        df = df[~df["label"].isin(too_small)].reset_index(drop=True)
    flow["removed_small_classes"] = too_small
    if df.empty or df["label"].nunique() < 2:
        raise RuntimeError("Fewer than two classes survived quality control.")

    # ---- finalise ---------------------------------------------------------
    known = [c for c in CANONICAL_CLASSES if c in set(df["label"])]
    classes = known + sorted(set(df["label"]) - set(known))
    df["y"] = df["label"].map({c: i for i, c in enumerate(classes)}).astype(int)
    df = df.sort_values(["label", "filename"]).reset_index(drop=True)

    flow["analysed"] = len(df)
    flow["classes"] = classes
    flow["class_counts"] = {c: int((df["label"] == c).sum()) for c in classes}
    flow["n_groups"] = int(df["group"].nunique()) if "group" in df else len(df)
    flow["imbalance_ratio"] = round(
        max(flow["class_counts"].values()) / max(1, min(flow["class_counts"].values())), 2
    )

    LOGGER.info("Analysis cohort: %d images, %d classes, %d groups",
                len(df), len(classes), flow["n_groups"])
    for c in classes[:15]:
        LOGGER.info("    %-34s %5d", c[:34], flow["class_counts"][c])
    if len(classes) > 15:
        LOGGER.info("    ... and %d further class(es)", len(classes) - 15)
    LOGGER.info("Imbalance ratio (max/min): %.2f", flow["imbalance_ratio"])
    return df, flow


def discover_dataset(cfg: Config, root: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Walk the extracted dataset and build a tidy manifest.

    Robust to the layout variations seen in the wild:
      root/<class>/*.jpg
      root/Rare_Skin_Disease_Dataset/<class>/*.jpg
      root/{train,test,val}/<class>/*.jpg
    Any directory whose name resolves to a canonical class is treated as a
    class directory, regardless of depth.
    """
    flow: Dict[str, Any] = {}
    all_files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    flow["files_found"] = len(all_files)
    if not all_files:
        raise FileNotFoundError(
            f"No image files under {root}. Pass --data-root pointing at the "
            f"extracted Rare_Skin_Disease_Dataset folder (or archive.zip), or "
            f"use --download / --synthetic."
        )

    records: List[Dict[str, Any]] = []
    unmatched_dirs: Counter = Counter()
    for path in all_files:
        label: Optional[str] = None
        split = "all"
        # Search upward from the file for the nearest directory that names a class.
        for parent in path.parents:
            if parent == root.parent:
                break
            if parent.name.lower() in SPLIT_DIR_NAMES:
                split = parent.name.lower()
                continue
            cand = normalise_label(parent.name)
            if cand is not None:
                label = cand
                break
        if label is None:
            unmatched_dirs[path.parent.name] += 1
            continue
        records.append(
            {"path": str(path), "label": label, "source_split": split, "filename": path.name}
        )

    if unmatched_dirs:
        LOGGER.warning(
            "Ignored %d image(s) in directories that do not map to a known class: %s",
            sum(unmatched_dirs.values()),
            dict(unmatched_dirs.most_common(8)),
        )
    flow["files_unmapped"] = int(sum(unmatched_dirs.values()))

    df = pd.DataFrame.from_records(records)
    if df.empty:
        raise RuntimeError(
            "Images were found but none could be assigned to a class. Expected "
            "class sub-directories named after: " + ", ".join(CANONICAL_CLASSES)
        )
    flow["files_labelled"] = len(df)
    df["group"] = [f"img{i}" for i in range(len(df))]
    return _finalise_cohort(cfg, df, flow)


# --------------------------------------------------------------------------- #
# Individual Typology Angle (ITA) - objective skin pigmentation proxy          #
# --------------------------------------------------------------------------- #
def srgb_to_lab(rgb: np.ndarray) -> np.ndarray:
    """
    Convert sRGB in [0,1] (..., 3) to CIE L*a*b* under D65.
    Implemented in numpy so scikit-image is not a dependency.
    """
    rgb = np.clip(rgb.astype(np.float64), 0.0, 1.0)
    # sRGB -> linear RGB
    lin = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    m = np.array(
        [
            [0.4124564, 0.3575761, 0.1804375],
            [0.2126729, 0.7151522, 0.0721750],
            [0.0193339, 0.1191920, 0.9503041],
        ]
    )
    xyz = lin @ m.T
    white = np.array([0.95047, 1.00000, 1.08883])
    xyz = xyz / white
    eps, kappa = 216.0 / 24389.0, 24389.0 / 27.0
    f = np.where(xyz > eps, np.cbrt(xyz), (kappa * xyz + 16.0) / 116.0)
    L = 116.0 * f[..., 1] - 16.0
    a = 500.0 * (f[..., 0] - f[..., 1])
    b = 200.0 * (f[..., 1] - f[..., 2])
    return np.stack([L, a, b], axis=-1)


def compute_ita(path: str, size: int = 128) -> float:
    """
    ITA = arctan((L* - 50) / b*) * 180 / pi, computed over the pixels most
    likely to be perilesional/normal skin. We take the inter-quartile band of
    luminance to exclude specular highlights, deep shadow, and the (typically
    erythematous, high-chroma) lesion itself.

    Fitzpatrick-aligned ITA bands (Chardon et al.):
        > 55 very light | 41-55 light | 28-41 intermediate
        10-28 tan       | -30-10 brown | < -30 dark
    """
    try:
        with Image.open(path) as im:
            im = im.convert("RGB").resize((size, size), Image.BILINEAR)
            arr = np.asarray(im, dtype=np.float32) / 255.0
    except Exception:
        return float("nan")
    lab = srgb_to_lab(arr.reshape(-1, 3))
    L, a, b = lab[:, 0], lab[:, 1], lab[:, 2]
    chroma = np.sqrt(a ** 2 + b ** 2)
    lo, hi = np.percentile(L, [25, 75])
    sel = (L >= lo) & (L <= hi) & (chroma < np.percentile(chroma, 70)) & (b > 1e-3)
    if sel.sum() < 32:
        sel = (L >= lo) & (L <= hi) & (b > 1e-3)
    if sel.sum() < 8:
        return float("nan")
    return float(np.degrees(np.arctan((np.median(L[sel]) - 50.0) / np.median(b[sel]))))


def ita_band(ita: float) -> str:
    if ita is None or (isinstance(ita, float) and math.isnan(ita)):
        return "Unknown"
    if ita > 41:
        return "Light (ITA > 41 deg)"
    if ita > 10:
        return "Intermediate/Tan (10-41 deg)"
    return "Brown/Dark (ITA <= 10 deg)"


def annotate_skin_tone(df: pd.DataFrame) -> pd.DataFrame:
    LOGGER.info("Computing Individual Typology Angle (skin-tone proxy) ...")
    df = df.copy()
    df["ita"] = [
        compute_ita(p) for p in tqdm(df["path"].tolist(), desc="ITA", leave=False)
    ]
    df["ita_band"] = [ita_band(v) for v in df["ita"]]
    counts = df["ita_band"].value_counts().to_dict()
    LOGGER.info("Skin-tone strata: %s", counts)
    return df

### 1.4 IMAGE REPRESENTATION (frozen CNN encoder)

In [ ]:
# =========================================================================== #
# SECTION 4 - IMAGE REPRESENTATION (frozen CNN encoder)                       #
# =========================================================================== #
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

BACKBONE_REGISTRY: Dict[str, Dict[str, Any]] = {
    "resnet18": {"ctor": "resnet18", "weights": "ResNet18_Weights", "dim": 512},
    "resnet50": {"ctor": "resnet50", "weights": "ResNet50_Weights", "dim": 2048},
    "densenet121": {"ctor": "densenet121", "weights": "DenseNet121_Weights", "dim": 1024},
    "efficientnet_b0": {
        "ctor": "efficientnet_b0",
        "weights": "EfficientNet_B0_Weights",
        "dim": 1280,
    },
    "convnext_tiny": {"ctor": "convnext_tiny", "weights": "ConvNeXt_Tiny_Weights", "dim": 768},
}


class ImageListDataset(torch.utils.data.Dataset):
    """Minimal, dependency-light image dataset over a list of file paths."""

    def __init__(self, paths: Sequence[str], transform) -> None:
        self.paths = list(paths)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        path = self.paths[idx]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            # QC should have removed these; fall back to grey rather than crash.
            img = Image.new("RGB", (256, 256), (128, 128, 128))
        return self.transform(img), idx


class FractionalCentreCrop:
    """
    Crop to the central `frac` of the shorter side before resizing.

    Clinical photographs of a lesion are usually framed with the lesion in the
    middle, so tightening the crop discards background, framing and setting -
    exactly the material the shortcut probe shows the model exploiting. Values
    around 0.6-0.8 remove most context while keeping the lesion.
    """

    def __init__(self, frac: float) -> None:
        self.frac = float(frac)

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        side = max(int(round(min(w, h) * self.frac)), 8)
        return T.functional.center_crop(img, [side, side])


def build_eval_transform(image_size: int, crop_frac: float = 1.0):
    if not _HAS_TORCHVISION:
        raise RuntimeError("torchvision is required for image feature extraction.")
    resize = int(round(image_size * 1.14))
    steps = []
    if crop_frac < 1.0:
        steps.append(FractionalCentreCrop(crop_frac))
    steps += [
        T.Resize(resize),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
    return T.Compose(steps)


def load_backbone(name: str, trainable: bool = False) -> Tuple[nn.Module, int]:
    """
    Instantiate a pretrained CNN with its classifier head removed.

    Handles both the modern `weights=` enum API and the legacy
    `pretrained=True` argument, and degrades to random initialisation if no
    network access is available (with a loud warning).
    """
    if not _HAS_TORCHVISION:
        raise RuntimeError("torchvision is required for image feature extraction.")
    if name not in BACKBONE_REGISTRY:
        raise ValueError(f"unknown backbone '{name}'; choose from {list(BACKBONE_REGISTRY)}")
    spec = BACKBONE_REGISTRY[name]
    ctor = getattr(torchvision.models, spec["ctor"])

    model = None
    try:
        weights_enum = getattr(torchvision.models, spec["weights"], None)
        if weights_enum is not None:
            model = ctor(weights=weights_enum.DEFAULT)
        else:
            raise AttributeError
    except Exception:
        try:
            model = ctor(pretrained=True)  # legacy torchvision
        except Exception as exc:
            LOGGER.warning(
                "Could not download pretrained weights for %s (%s). Falling back to "
                "RANDOM initialisation - results will NOT be publication-valid.",
                name,
                exc,
            )
            model = ctor(weights=None) if "weights" in ctor.__code__.co_varnames else ctor()

    dim = spec["dim"]
    if name.startswith("resnet"):
        model.fc = nn.Identity()
    elif name.startswith("densenet"):
        model.classifier = nn.Identity()
    elif name.startswith("efficientnet"):
        model.classifier = nn.Identity()
    elif name.startswith("convnext"):
        model.classifier = nn.Sequential(*list(model.classifier.children())[:-1], nn.Flatten(1))
    if trainable:
        model.train()
        for p in model.parameters():
            p.requires_grad_(True)
    else:
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
    return model, dim


def build_train_transform(image_size: int, crop_frac: float = 1.0):
    """
    Augmentation for backbone fine-tuning.

    Deliberately aggressive on colour and geometry: with a few hundred training
    images the backbone will otherwise memorise them, and colour jitter also
    weakens any reliance on device-specific white balance.
    """
    if not _HAS_TORCHVISION:
        raise RuntimeError("torchvision is required for fine-tuning.")
    pre = [FractionalCentreCrop(crop_frac)] if crop_frac < 1.0 else []
    return T.Compose(
        pre + [
            T.RandomResizedCrop(image_size, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(p=0.2),
            T.RandomApply([T.RandomRotation(20)], p=0.5),
            T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.03),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            T.RandomErasing(p=0.25, scale=(0.02, 0.10)),
        ]
    )


class LabelledImageDataset(torch.utils.data.Dataset):
    """Image dataset that also yields labels, for supervised fine-tuning."""

    def __init__(self, paths: Sequence[str], labels: Sequence[int], transform) -> None:
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        try:
            with Image.open(self.paths[idx]) as im:
                img = im.convert("RGB")
        except Exception:
            img = Image.new("RGB", (256, 256), (128, 128, 128))
        return self.transform(img), int(self.labels[idx])


class FinetuneNet(nn.Module):
    """Backbone plus a linear head, used only to adapt the encoder."""

    def __init__(self, backbone: nn.Module, dim: int, n_cls: int, dropout: float = 0.3):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, n_cls))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x).flatten(1))

    @torch.no_grad()
    def embed(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x).flatten(1)


def finetune_backbone(
    cfg: Config, paths: Sequence[str], y: np.ndarray,
    train_idx: np.ndarray, val_idx: np.ndarray, n_cls: int,
    device: torch.device, seed: int,
) -> Tuple[nn.Module, Dict[str, Any]]:
    """
    Adapt the imaging encoder to this fold's training images.

    Frozen ImageNet features are a weak representation for dermatology: they
    were never trained to separate bullae from ichthyotic scale. Fine-tuning is
    the largest legitimate source of additional accuracy available here.

    It must happen inside the fold. Fine-tuning once on the whole cohort would
    let every test image influence the encoder, and the resulting numbers would
    not survive replication.
    """
    set_seed(seed)
    name = cfg.backbone.split(",")[0].strip()
    backbone, dim = load_backbone(name, trainable=True)
    model = FinetuneNet(backbone, dim, n_cls, cfg.dropout * 0.6).to(device)

    train_ds = LabelledImageDataset(
        [paths[i] for i in train_idx], y[train_idx], build_train_transform(cfg.image_size, cfg.center_crop_frac)
    )
    val_ds = LabelledImageDataset(
        [paths[i] for i in val_idx], y[val_idx], build_eval_transform(cfg.image_size, cfg.center_crop_frac)
    )
    common = dict(num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"))
    train_ld = torch.utils.data.DataLoader(
        train_ds, batch_size=cfg.finetune_batch, shuffle=True, drop_last=len(train_ds) > cfg.finetune_batch, **common
    )
    val_ld = torch.utils.data.DataLoader(val_ds, batch_size=cfg.finetune_batch, shuffle=False, **common)

    weights = class_weights_from(y[train_idx], n_cls).to(device) if cfg.use_class_weights else None
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=cfg.label_smoothing)
    # Discriminative learning rates: the head must move faster than the trunk,
    # whose ImageNet features are already useful and easily destroyed.
    opt = torch.optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": cfg.finetune_lr},
            {"params": model.head.parameters(), "lr": cfg.finetune_lr * 10},
        ],
        weight_decay=1e-4,
    )
    _steps = max(cfg.finetune_epochs * max(len(train_ld), 1), 1)
    # See note in dermnet_benchmark: OneCycle divides by zero when the schedule
    # is too short for a warm-up phase.
    if _steps >= 8:
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=[cfg.finetune_lr, cfg.finetune_lr * 10],
            total_steps=_steps, pct_start=0.25,
        )
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_score, best_state, stale = -np.inf, None, 0
    history = {"train_loss": [], "val_bacc": []}
    for epoch in range(cfg.finetune_epochs):
        model.train()
        losses = []
        for xb, yb in train_ld:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(float(loss.detach().item()))

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in val_ld:
                out = model(xb.to(device, non_blocking=True))
                preds.append(out.argmax(1).cpu().numpy())
                trues.append(yb.numpy())
        vb = balanced_accuracy_score(np.concatenate(trues), np.concatenate(preds)) if preds else 0.0
        history["train_loss"].append(float(np.mean(losses)) if losses else float("nan"))
        history["val_bacc"].append(float(vb))

        if vb > best_score + 1e-6:
            best_score, stale = vb, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= max(cfg.finetune_patience, 1):
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, {"val_bacc": float(best_score), "epochs": len(history["train_loss"]),
                   "dim": dim, "history": history}


@torch.no_grad()
def embed_with_model(
    model: FinetuneNet, cfg: Config, paths: Sequence[str], device: torch.device
) -> np.ndarray:
    """Embed every image with a fine-tuned encoder (flip-TTA if enabled)."""
    ds = ImageListDataset(list(paths), build_eval_transform(cfg.image_size, cfg.center_crop_frac))
    loader = torch.utils.data.DataLoader(
        ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"),
    )
    out = None
    for batch, idx in loader:
        batch = batch.to(device, non_blocking=True)
        emb = model.embed(batch).float()
        if cfg.feature_tta:
            emb = 0.5 * (emb + model.embed(torch.flip(batch, dims=[3])).float())
        if out is None:
            out = np.zeros((len(ds), emb.shape[1]), dtype=np.float32)
        out[idx.numpy()] = emb.cpu().numpy()
    return out if out is not None else np.zeros((len(ds), 1), dtype=np.float32)


@torch.no_grad()
def extract_features(
    cfg: Config, df: pd.DataFrame, device: torch.device
) -> Tuple[np.ndarray, str]:
    """
    Map every image to a fixed embedding using a *frozen*, label-agnostic
    encoder. Because no labels enter this step, computing embeddings over the
    full cohort once (rather than per fold) introduces no optimistic bias.
    """
    names = [n.strip() for n in cfg.backbone.split(",") if n.strip()]
    banner(f"Feature extraction - backbone(s) = {', '.join(names)}")
    tf = build_eval_transform(cfg.image_size, cfg.center_crop_frac)
    ds = ImageListDataset(df["path"].tolist(), tf)
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
    )

    # Several backbones can be fused by concatenating their (separately
    # L2-normalised) embeddings. Different architectures fail on different
    # images, so the union is usually a few points stronger than any one.
    parts: List[np.ndarray] = []
    for name in names:
        model, dim = load_backbone(name)
        model = model.to(device)
        feats = np.zeros((len(ds), dim), dtype=np.float32)
        for batch, idx in tqdm(loader, desc=f"embedding [{name}]", leave=False):
            batch = batch.to(device, non_blocking=True)
            out = model(batch).float()
            if cfg.feature_tta:  # horizontal-flip TTA halves embedding variance
                out = 0.5 * (out + model(torch.flip(batch, dims=[3])).float())
            feats[idx.numpy()] = out.flatten(1).cpu().numpy()
        parts.append(_l2_normalise(feats) if len(names) > 1 else feats)
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    feats = np.concatenate(parts, axis=1) if len(parts) > 1 else parts[0]
    LOGGER.info("Raw embeddings: %s", feats.shape)
    desc = (f"{'+'.join(names)} (frozen, ImageNet-pretrained, d={feats.shape[1]})"
            if not cfg.finetune else
            f"{names[0]} (fine-tuned per fold, d={feats.shape[1]})")
    return feats, desc


def fit_reducer(train_feats: np.ndarray, pca_dim: int, seed: int):
    """
    Fit standardisation (+ optional PCA whitening) on *training rows only* and
    return a callable that projects any feature matrix into that space.

    Fitting the projection on the whole cohort would be label-free but still
    lets the held-out partition's distribution shape the representation. Refit
    per fold, the pipeline is clean end to end.
    """
    scaler = StandardScaler().fit(train_feats.astype(np.float64))
    Z = scaler.transform(train_feats.astype(np.float64))
    info: Dict[str, Any] = {"input_dim": int(train_feats.shape[1])}

    pca = None
    if pca_dim and 0 < pca_dim < min(Z.shape):
        pca = PCA(n_components=pca_dim, whiten=True, random_state=seed).fit(Z)
        info["pca_dim"] = int(pca_dim)
        info["explained_variance"] = float(np.sum(pca.explained_variance_ratio_))
    else:
        info["pca_dim"] = int(Z.shape[1])
        info["explained_variance"] = 1.0

    def transform(feats: np.ndarray) -> np.ndarray:
        out = scaler.transform(feats.astype(np.float64))
        if pca is not None:
            out = pca.transform(out)
        return out.astype(np.float32)

    info["output_dim"] = int(transform(train_feats[:1]).shape[1])
    return transform, info


def reduce_features(feats: np.ndarray, pca_dim: int, seed: int) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Standardise then (optionally) PCA-whiten. With n = 733 and d = 2048 the
    raw embedding space is badly over-parameterised for a k-NN graph; PCA to
    ~128 dimensions stabilises the similarity metric and suppresses noise
    directions. PCA is unsupervised, so this remains leakage-free.
    """
    info: Dict[str, Any] = {"input_dim": int(feats.shape[1])}
    X = StandardScaler().fit_transform(feats.astype(np.float64))
    if pca_dim and 0 < pca_dim < min(X.shape):
        pca = PCA(n_components=pca_dim, whiten=True, random_state=seed)
        X = pca.fit_transform(X)
        info["pca_dim"] = int(pca_dim)
        info["explained_variance"] = float(np.sum(pca.explained_variance_ratio_))
        LOGGER.info(
            "PCA -> %d dims, cumulative explained variance = %.1f%%",
            pca_dim,
            100 * info["explained_variance"],
        )
    else:
        info["pca_dim"] = int(X.shape[1])
        info["explained_variance"] = 1.0
    X = X.astype(np.float32)
    info["output_dim"] = int(X.shape[1])
    return X, info

### 1.5 POPULATION GRAPH CONSTRUCTION

In [ ]:
# =========================================================================== #
# SECTION 5 - POPULATION GRAPH CONSTRUCTION                                   #
# =========================================================================== #
def _l2_normalise(X: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norm, 1e-12)


def similarity_matrix(X: np.ndarray, metric: str) -> np.ndarray:
    """Dense pairwise similarity. n <= a few thousand, so dense is fine."""
    if metric == "cosine":
        Z = _l2_normalise(X.astype(np.float32))
        S = Z @ Z.T
    elif metric == "euclidean":
        sq = np.sum(X ** 2, axis=1)
        d2 = np.maximum(sq[:, None] + sq[None, :] - 2.0 * (X @ X.T), 0.0)
        sigma = np.median(np.sqrt(d2)) + 1e-12
        S = np.exp(-d2 / (2.0 * sigma ** 2))
    else:
        raise ValueError(f"unknown graph metric '{metric}'")
    return np.clip(S, -1.0, 1.0).astype(np.float32)


def build_knn_edges(
    S: np.ndarray,
    k: int,
    source_mask: np.ndarray,
    target_mask: np.ndarray,
    mutual: bool = False,
    threshold: float = 0.0,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Directed k-NN edges j -> i, where i ranges over `target_mask` (nodes that
    may *receive* messages) and j is drawn only from `source_mask` (nodes that
    may *send* them).

    This asymmetry is what makes the inductive protocol possible: at training
    time both masks are the training subgraph; at inference the test nodes are
    added as receivers only, so a test image can look at labelled neighbours
    but can never leak information into another test node's prediction.

    Returns (edge_index[2, E], edge_weight[E]) with self-loops excluded here;
    the layers add them explicitly.
    """
    n = S.shape[0]
    src_idx = np.flatnonzero(source_mask)
    tgt_idx = np.flatnonzero(target_mask)
    if src_idx.size == 0 or tgt_idx.size == 0:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    sub = S[np.ix_(tgt_idx, src_idx)].copy()
    # Forbid self-selection where a node is both source and target.
    same = tgt_idx[:, None] == src_idx[None, :]
    sub[same] = -np.inf

    kk = int(min(k, src_idx.size - 1 if src_idx.size > 1 else 1))
    kk = max(kk, 1)
    part = np.argpartition(-sub, kth=kk - 1, axis=1)[:, :kk]

    rows = np.repeat(np.arange(len(tgt_idx)), kk)
    cols = part.ravel()
    weights = sub[rows, cols]
    valid = np.isfinite(weights) & (weights > threshold)

    dst = tgt_idx[rows[valid]]
    src = src_idx[cols[valid]]
    w = weights[valid].astype(np.float32)

    if mutual:
        # Keep j -> i only if i is also among j's k nearest neighbours.
        rank = np.argsort(-S, axis=1)[:, : kk + 1]
        keep = np.array([dst[e] in rank[src[e]] for e in range(len(src))], dtype=bool)
        dst, src, w = dst[keep], src[keep], w[keep]

    edge_index = np.stack([src, dst], axis=0).astype(np.int64)  # [source, target]
    return edge_index, np.clip(w, 0.0, None)


def graph_statistics(edge_index: np.ndarray, n: int, y: np.ndarray, labelled: np.ndarray) -> Dict[str, Any]:
    """Descriptive graph metrics, including label homophily (the key sanity check)."""
    if edge_index.shape[1] == 0:
        return {"num_edges": 0, "mean_degree": 0.0, "homophily": float("nan"), "isolated_nodes": n}
    deg = np.bincount(edge_index[1], minlength=n)
    src, dst = edge_index[0], edge_index[1]
    both = labelled[src] & labelled[dst]
    homo = float(np.mean(y[src[both]] == y[dst[both]])) if both.any() else float("nan")
    return {
        "num_edges": int(edge_index.shape[1]),
        "mean_degree": float(deg.mean()),
        "median_degree": float(np.median(deg)),
        "isolated_nodes": int((deg == 0).sum()),
        "homophily": homo,
    }

### 1.6 GRAPH NEURAL NETWORK OPERATORS (pure PyTorch, no PyG needed)

In [ ]:
# =========================================================================== #
# SECTION 6 - GRAPH NEURAL NETWORK OPERATORS (pure PyTorch, no PyG needed)    #
# =========================================================================== #
def add_self_loops(
    edge_index: torch.Tensor, edge_weight: torch.Tensor, num_nodes: int
) -> Tuple[torch.Tensor, torch.Tensor]:
    device = edge_index.device
    loop = torch.arange(num_nodes, device=device, dtype=edge_index.dtype)
    loop_index = torch.stack([loop, loop], dim=0)
    loop_weight = torch.ones(num_nodes, device=device, dtype=edge_weight.dtype)
    return (
        torch.cat([edge_index, loop_index], dim=1),
        torch.cat([edge_weight, loop_weight], dim=0),
    )


def drop_edge(
    edge_index: torch.Tensor, edge_weight: torch.Tensor, p: float, training: bool
) -> Tuple[torch.Tensor, torch.Tensor]:
    """DropEdge regularisation (Rong et al., ICLR 2020) - vital at n < 1000."""
    if not training or p <= 0.0 or edge_index.numel() == 0:
        return edge_index, edge_weight
    keep = torch.rand(edge_index.size(1), device=edge_index.device) >= p
    if not bool(keep.any()):
        return edge_index, edge_weight
    return edge_index[:, keep], edge_weight[keep]


def scatter_softmax(src: torch.Tensor, index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """
    Numerically stable softmax over edges grouped by their target node.
    src: [E, H]; index: [E] target node ids.
    """
    idx = index.unsqueeze(-1).expand_as(src)
    try:
        maxes = torch.full(
            (num_nodes, src.size(1)), float("-inf"), dtype=src.dtype, device=src.device
        )
        maxes = maxes.scatter_reduce(0, idx, src, reduce="amax", include_self=True)
        maxes = torch.nan_to_num(maxes, neginf=0.0, posinf=0.0)
    except (RuntimeError, TypeError):  # torch < 1.12
        maxes = torch.zeros(num_nodes, src.size(1), dtype=src.dtype, device=src.device)
        maxes = maxes + src.max().detach()
    centred = (src - maxes.index_select(0, index)).clamp(min=-30.0, max=30.0)
    expo = centred.exp()
    denom = torch.zeros(num_nodes, src.size(1), dtype=src.dtype, device=src.device)
    denom = denom.index_add(0, index, expo)
    return expo / (denom.index_select(0, index) + 1e-16)


class GCNLayer(nn.Module):
    """Kipf & Welling (2017), symmetric-normalised propagation with edge weights."""

    def __init__(self, in_dim: int, out_dim: int, bias: bool = True) -> None:
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)
        nn.init.xavier_uniform_(self.lin.weight)
        if bias:
            nn.init.zeros_(self.lin.bias)

    def forward(
        self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor
    ) -> torch.Tensor:
        n = x.size(0)
        ei, ew = add_self_loops(edge_index, edge_weight, n)
        deg_in = torch.zeros(n, device=x.device, dtype=x.dtype).index_add(0, ei[1], ew)
        deg_out = torch.zeros(n, device=x.device, dtype=x.dtype).index_add(0, ei[0], ew)
        norm = (
            deg_out.clamp(min=1e-12).pow(-0.5)[ei[0]]
            * ew
            * deg_in.clamp(min=1e-12).pow(-0.5)[ei[1]]
        )
        h = self.lin(x)
        msg = h.index_select(0, ei[0]) * norm.unsqueeze(-1)
        out = torch.zeros_like(h).index_add(0, ei[1], msg)
        return out


class SAGELayer(nn.Module):
    """GraphSAGE with mean aggregation (Hamilton et al., 2017)."""

    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim, bias=False)
        nn.init.xavier_uniform_(self.lin_self.weight)
        nn.init.xavier_uniform_(self.lin_neigh.weight)
        nn.init.zeros_(self.lin_self.bias)

    def forward(
        self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor
    ) -> torch.Tensor:
        n = x.size(0)
        if edge_index.numel() == 0:
            return self.lin_self(x)
        msg = x.index_select(0, edge_index[0]) * edge_weight.unsqueeze(-1)
        agg = torch.zeros_like(x).index_add(0, edge_index[1], msg)
        deg = torch.zeros(n, device=x.device, dtype=x.dtype).index_add(
            0, edge_index[1], edge_weight
        )
        agg = agg / deg.clamp(min=1e-12).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg)


class GATLayer(nn.Module):
    """
    Multi-head graph attention (Velickovic et al., 2018).

    The learned attention coefficients are retained after the forward pass so
    that they can be exported for explainability (which visually similar
    training cases drove a given prediction).
    """

    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        heads: int = 4,
        concat: bool = True,
        dropout: float = 0.0,
        negative_slope: float = 0.2,
    ) -> None:
        super().__init__()
        self.heads, self.out_dim, self.concat = heads, out_dim, concat
        self.dropout, self.negative_slope = dropout, negative_slope
        self.lin = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_dim))
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_dim))
        self.bias = nn.Parameter(torch.zeros(heads * out_dim if concat else out_dim))
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        self.last_attention: Optional[torch.Tensor] = None

    def forward(
        self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor
    ) -> torch.Tensor:
        n = x.size(0)
        ei, ew = add_self_loops(edge_index, edge_weight, n)
        h = self.lin(x).view(n, self.heads, self.out_dim)

        a_src = (h * self.att_src).sum(-1)  # [N, H]
        a_dst = (h * self.att_dst).sum(-1)  # [N, H]
        logits = a_src.index_select(0, ei[0]) + a_dst.index_select(0, ei[1])
        logits = F.leaky_relu(logits, self.negative_slope)
        # Similarity acts as a soft prior on the attention logits.
        logits = logits + torch.log(ew.clamp(min=1e-6)).unsqueeze(-1)
        alpha = scatter_softmax(logits, ei[1], n)
        self.last_attention = alpha.detach()
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        msg = h.index_select(0, ei[0]) * alpha.unsqueeze(-1)
        out = torch.zeros_like(h).index_add(0, ei[1], msg)
        out = out.reshape(n, -1) if self.concat else out.mean(dim=1)
        return out + self.bias

### 1.7 MODELS

In [ ]:
# =========================================================================== #
# SECTION 7 - MODELS                                                          #
# =========================================================================== #
class MLPNet(nn.Module):
    """Graph-free control: identical capacity/optimisation, no message passing.
    Any gain of the GNNs over this model is attributable to the graph alone."""

    def __init__(self, in_dim: int, hidden: int, n_cls: int, layers: int = 2, dropout: float = 0.5):
        super().__init__()
        dims = [in_dim] + [hidden] * max(layers - 1, 1)
        blocks: List[nn.Module] = []
        for a, b in zip(dims[:-1], dims[1:]):
            blocks += [nn.Linear(a, b), nn.BatchNorm1d(b), nn.ELU(), nn.Dropout(dropout)]
        self.body = nn.Sequential(*blocks)
        self.head = nn.Linear(dims[-1], n_cls)

    def forward(self, x, edge_index=None, edge_weight=None):
        return self.head(self.body(x))

    def embed(self, x, edge_index=None, edge_weight=None):
        return self.body(x)


class GNNStack(nn.Module):
    """Homogeneous stack of GCN / GraphSAGE / GAT layers."""

    def __init__(
        self,
        kind: str,
        in_dim: int,
        hidden: int,
        n_cls: int,
        layers: int = 2,
        heads: int = 4,
        dropout: float = 0.5,
        drop_edge: float = 0.0,
    ):
        super().__init__()
        self.kind, self.dropout, self.drop_edge = kind, dropout, drop_edge
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        d = in_dim
        for _ in range(layers):
            if kind == "GCN":
                conv, out = GCNLayer(d, hidden), hidden
            elif kind == "GraphSAGE":
                conv, out = SAGELayer(d, hidden), hidden
            elif kind == "GAT":
                conv, out = GATLayer(d, hidden // heads, heads, True, dropout), (hidden // heads) * heads
            else:
                raise ValueError(f"unknown GNN kind '{kind}'")
            self.convs.append(conv)
            self.norms.append(nn.BatchNorm1d(out))
            d = out
        self.head = nn.Linear(d, n_cls)
        self.out_dim = d

    def embed(self, x, edge_index, edge_weight):
        ei, ew = drop_edge(edge_index, edge_weight, self.drop_edge, self.training)
        h = x
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, ei, ew)
            h = norm(h)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_weight):
        return self.head(self.embed(x, edge_index, edge_weight))


class DermGNN(nn.Module):
    """
    Proposed architecture.

    Design choices, each motivated by the n = 733, 5-class, imbalanced regime:
      * a linear stem projects the (PCA-whitened) embedding into the GNN width,
        decoupling backbone dimensionality from graph capacity;
      * multi-head graph attention lets each node weight its neighbours, which
        matters because k-NN neighbourhoods in a rare-disease cohort are
        inevitably impure;
      * residual connections + BatchNorm prevent the over-smoothing that
        otherwise destroys performance beyond two propagation steps;
      * jumping knowledge (concatenating every layer's output with the stem)
        preserves the purely visual signal, so the model can fall back to
        image evidence when the local graph neighbourhood is unhelpful;
      * DropEdge + feature dropout provide the strong regularisation a
        700-node graph demands.
    """

    def __init__(
        self,
        in_dim: int,
        hidden: int,
        n_cls: int,
        layers: int = 2,
        heads: int = 4,
        dropout: float = 0.5,
        drop_edge: float = 0.1,
    ):
        super().__init__()
        self.dropout, self.drop_edge = dropout, drop_edge
        per_head = max(hidden // heads, 4)
        width = per_head * heads
        self.stem = nn.Sequential(nn.Linear(in_dim, width), nn.BatchNorm1d(width), nn.ELU())
        self.convs = nn.ModuleList(
            [GATLayer(width, per_head, heads, True, dropout) for _ in range(layers)]
        )
        self.norms = nn.ModuleList([nn.BatchNorm1d(width) for _ in range(layers)])
        self.out_dim = width * (layers + 1)  # jumping knowledge
        self.head = nn.Sequential(
            nn.Linear(self.out_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_cls),
        )

    def embed(self, x, edge_index, edge_weight):
        ei, ew = drop_edge(edge_index, edge_weight, self.drop_edge, self.training)
        h = self.stem(x)
        states = [h]
        for conv, norm in zip(self.convs, self.norms):
            m = conv(h, ei, ew)
            m = norm(m)
            m = F.elu(m)
            m = F.dropout(m, p=self.dropout, training=self.training)
            h = h + m  # residual
            states.append(h)
        return torch.cat(states, dim=1)

    def forward(self, x, edge_index, edge_weight):
        return self.head(self.embed(x, edge_index, edge_weight))

    def attention_weights(self) -> List[Optional[torch.Tensor]]:
        return [c.last_attention for c in self.convs]


def build_model(name: str, cfg: Config, in_dim: int, n_cls: int) -> nn.Module:
    if name == "MLP":
        return MLPNet(in_dim, cfg.hidden_dim, n_cls, cfg.num_layers, cfg.dropout)
    if name in {"GCN", "GraphSAGE", "GAT"}:
        return GNNStack(
            name, in_dim, cfg.hidden_dim, n_cls, cfg.num_layers, cfg.heads, cfg.dropout, cfg.drop_edge
        )
    if name == "DERM-GNN":
        return DermGNN(
            in_dim, cfg.hidden_dim, n_cls, cfg.num_layers, cfg.heads, cfg.dropout, cfg.drop_edge
        )
    raise ValueError(f"'{name}' is not a torch model")

### 1.8 LOSS, CALIBRATION AND THE TRAINING LOOP

In [ ]:
# =========================================================================== #
# SECTION 8 - LOSS, CALIBRATION AND THE TRAINING LOOP                         #
# =========================================================================== #
class FocalLoss(nn.Module):
    """
    Class-weighted focal loss with label smoothing.

    Focal down-weighting (gamma) concentrates gradient on hard examples, while
    the inverse-frequency class weights counteract the prevalence skew. Label
    smoothing tempers the over-confidence that small-sample training induces
    and measurably improves calibration.
    """

    def __init__(self, weight: Optional[torch.Tensor] = None, gamma: float = 2.0, label_smoothing: float = 0.0):
        super().__init__()
        self.register_buffer("weight", weight if weight is not None else torch.tensor([]))
        self.gamma = float(gamma)
        self.ls = float(label_smoothing)

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        n_cls = logits.size(-1)
        logp = F.log_softmax(logits, dim=-1)
        if self.ls > 0.0 and n_cls > 1:
            off = self.ls / (n_cls - 1)
            dist = torch.full_like(logp, off)
            dist.scatter_(1, target.unsqueeze(1), 1.0 - self.ls)
        else:
            dist = torch.zeros_like(logp).scatter_(1, target.unsqueeze(1), 1.0)
        ce = -(dist * logp).sum(dim=1)
        pt = logp.gather(1, target.unsqueeze(1)).squeeze(1).exp().clamp(1e-6, 1.0)
        loss = ((1.0 - pt) ** self.gamma) * ce
        if self.weight is not None and self.weight.numel() == n_cls:
            loss = loss * self.weight.index_select(0, target)
        return loss.mean()


def class_weights_from(y: np.ndarray, n_cls: int) -> torch.Tensor:
    """Inverse-frequency weights, normalised to mean 1 so the LR stays comparable."""
    counts = np.bincount(y, minlength=n_cls).astype(np.float64)
    counts[counts == 0] = 1.0
    w = counts.sum() / (n_cls * counts)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)


def fit_temperature(logits: torch.Tensor, labels: torch.Tensor, max_iter: int = 200) -> float:
    """
    Single-parameter temperature scaling (Guo et al., 2017), fitted on the
    held-out validation split only. Improves probability calibration without
    changing the ranking, hence leaves AUC and accuracy untouched.
    """
    logits = logits.detach().double()
    labels = labels.detach().long()
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=max_iter)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    try:
        opt.step(closure)
        temp = float(log_t.detach().exp().item())
    except Exception:
        temp = 1.0
    return float(np.clip(temp, 0.05, 20.0))


@dataclass
class FoldGraphs:
    """Edge sets for the three phases of one cross-validation fold."""

    train: Tuple[torch.Tensor, torch.Tensor]
    val: Tuple[torch.Tensor, torch.Tensor]
    test: Tuple[torch.Tensor, torch.Tensor]
    stats: Dict[str, Any]


def build_fold_graphs(
    S: np.ndarray,
    cfg: Config,
    y: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    device: torch.device,
) -> FoldGraphs:
    """
    Construct the fold's graphs under the chosen protocol.

    inductive (default, and the clinically defensible option)
        train phase : edges train -> train
        val phase   : edges train -> (train u val)
        test phase  : edges (train u val) -> (train u val u test)
        A test node therefore receives messages only from *seen* cases; no
        test-test edge exists, so predictions are mutually independent, exactly
        as they would be for a prospectively presented patient.

    transductive
        one graph over all nodes; reported for comparison only, and flagged as
        such in the results, because it lets unlabelled test images influence
        one another.
    """
    n = S.shape[0]
    m_train = np.zeros(n, bool)
    m_train[train_idx] = True
    m_val = np.zeros(n, bool)
    m_val[val_idx] = True
    m_test = np.zeros(n, bool)
    m_test[test_idx] = True

    def to_torch(ei: np.ndarray, ew: np.ndarray):
        return (
            torch.as_tensor(ei, dtype=torch.long, device=device),
            torch.as_tensor(ew, dtype=torch.float32, device=device),
        )

    if cfg.graph_mode == "transductive":
        all_mask = np.ones(n, bool)
        ei, ew = build_knn_edges(S, cfg.knn_k, all_mask, all_mask, cfg.mutual_knn, cfg.edge_sim_threshold)
        g = to_torch(ei, ew)
        stats = graph_statistics(ei, n, y, m_train | m_val | m_test)
        return FoldGraphs(train=g, val=g, test=g, stats=stats)

    ei_tr, ew_tr = build_knn_edges(S, cfg.knn_k, m_train, m_train, cfg.mutual_knn, cfg.edge_sim_threshold)
    ei_va, ew_va = build_knn_edges(S, cfg.knn_k, m_train, m_train | m_val, cfg.mutual_knn, cfg.edge_sim_threshold)
    seen = m_train | m_val
    ei_te, ew_te = build_knn_edges(S, cfg.knn_k, seen, seen | m_test, cfg.mutual_knn, cfg.edge_sim_threshold)
    stats = graph_statistics(ei_te, n, y, seen)
    stats["train_edges"] = int(ei_tr.shape[1])
    return FoldGraphs(
        train=to_torch(ei_tr, ew_tr),
        val=to_torch(ei_va, ew_va),
        test=to_torch(ei_te, ew_te),
        stats=stats,
    )


def train_torch_model(
    name: str,
    cfg: Config,
    X: torch.Tensor,
    y: torch.Tensor,
    graphs: FoldGraphs,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    n_cls: int,
    device: torch.device,
    seed: int,
) -> Dict[str, Any]:
    """
    Train one neural model for a single fold and return calibrated
    probabilities for the validation and test nodes, plus the loss history.
    """
    set_seed(seed)
    model = build_model(name, cfg, X.size(1), n_cls).to(device)

    y_train_np = y[train_idx].cpu().numpy()
    weights = class_weights_from(y_train_np, n_cls).to(device) if cfg.use_class_weights else None
    criterion = FocalLoss(weights, cfg.focal_gamma, cfg.label_smoothing).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(cfg.epochs, 1), eta_min=cfg.lr * 0.02)

    tr_t = torch.as_tensor(train_idx, dtype=torch.long, device=device)
    va_t = torch.as_tensor(val_idx, dtype=torch.long, device=device)
    te_t = torch.as_tensor(test_idx, dtype=torch.long, device=device)

    ei_tr, ew_tr = graphs.train
    ei_va, ew_va = graphs.val
    ei_te, ew_te = graphs.test

    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history: Dict[str, List[float]] = {"train_loss": [], "val_loss": [], "val_bacc": []}

    for epoch in range(cfg.epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(X, ei_tr, ew_tr)
        loss = criterion(logits.index_select(0, tr_t), y.index_select(0, tr_t))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            vlogits = model(X, ei_va, ew_va).index_select(0, va_t)
            vy = y.index_select(0, va_t)
            vloss = F.cross_entropy(vlogits, vy).item()
            vpred = vlogits.argmax(1).cpu().numpy()
            vbacc = balanced_accuracy_score(vy.cpu().numpy(), vpred)

        history["train_loss"].append(float(loss.item()))
        history["val_loss"].append(float(vloss))
        history["val_bacc"].append(float(vbacc))

        # Model selection on balanced accuracy: the primary endpoint, and the
        # only sensible criterion under this degree of class imbalance.
        score = vbacc - 1e-4 * vloss
        if score > best_score + 1e-6:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        val_logits = model(X, ei_va, ew_va).index_select(0, va_t).float().cpu()
        test_logits = model(X, ei_te, ew_te).index_select(0, te_t).float().cpu()

    temperature = 1.0
    if cfg.calibrate and len(val_idx) >= max(2 * n_cls, 10):
        temperature = fit_temperature(val_logits, y.index_select(0, va_t).cpu())

    val_prob = F.softmax(val_logits / temperature, dim=1).numpy()
    test_prob = F.softmax(test_logits / temperature, dim=1).numpy()
    test_prob_uncal = F.softmax(test_logits, dim=1).numpy()

    attn = None
    if isinstance(model, DermGNN):
        try:
            with torch.no_grad():
                model(X, ei_te, ew_te)
            atts = [a for a in model.attention_weights() if a is not None]
            if atts:
                attn = atts[-1].mean(dim=1).cpu().numpy()
        except Exception:
            attn = None

    return {
        "val_prob": val_prob,
        "test_prob": test_prob,
        "test_prob_uncal": test_prob_uncal,
        "temperature": temperature,
        "best_epoch": int(best_epoch),
        "epochs_run": len(history["train_loss"]),
        "history": history,
        "attention": attn,
        "n_params": int(sum(p.numel() for p in model.parameters())),
    }


def train_classical_model(
    name: str, cfg: Config, Xnp: np.ndarray, y: np.ndarray,
    train_idx: np.ndarray, val_idx: np.ndarray, test_idx: np.ndarray, seed: int,
) -> Dict[str, Any]:
    """Non-graph reference classifiers on the identical feature matrix."""
    cw = "balanced" if cfg.use_class_weights else None
    if name == "LogisticRegression":
        clf = LogisticRegression(max_iter=3000, C=1.0, class_weight=cw, random_state=seed, n_jobs=None)
    elif name == "SVM-RBF":
        clf = SVC(C=10.0, gamma="scale", probability=True, class_weight=cw, random_state=seed)
    elif name == "RandomForest":
        clf = RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=1,
            class_weight="balanced_subsample" if cfg.use_class_weights else None,
            random_state=seed, n_jobs=-1,
        )
    else:
        raise ValueError(f"'{name}' is not a classical model")

    classes_present = np.unique(y[train_idx])
    clf.fit(Xnp[train_idx], y[train_idx])

    def expand(prob: np.ndarray, n_cls: int) -> np.ndarray:
        """Re-insert zero columns for classes absent from this training fold."""
        if prob.shape[1] == n_cls:
            return prob
        full = np.zeros((prob.shape[0], n_cls), dtype=np.float64)
        for j, c in enumerate(clf.classes_):
            full[:, int(c)] = prob[:, j]
        return full

    n_cls = int(y.max()) + 1
    val_prob = expand(clf.predict_proba(Xnp[val_idx]), n_cls)
    test_prob = expand(clf.predict_proba(Xnp[test_idx]), n_cls)
    _ = classes_present
    return {
        "val_prob": val_prob,
        "test_prob": test_prob,
        "test_prob_uncal": test_prob,
        "temperature": 1.0,
        "best_epoch": -1,
        "epochs_run": 0,
        "history": {"train_loss": [], "val_loss": [], "val_bacc": []},
        "attention": None,
        "n_params": -1,
    }

### 1.9 PERFORMANCE METRICS AND UNCERTAINTY

In [ ]:
# =========================================================================== #
# SECTION 9 - PERFORMANCE METRICS AND UNCERTAINTY                             #
# =========================================================================== #
PRIMARY_METRIC = "balanced_accuracy"
METRIC_LABELS: "OrderedDict[str, str]" = OrderedDict(
    [
        ("balanced_accuracy", "Balanced accuracy"),
        ("macro_f1", "Macro F1"),
        ("accuracy", "Accuracy"),
        ("macro_precision", "Macro precision (PPV)"),
        ("macro_recall", "Macro recall (sensitivity)"),
        ("macro_specificity", "Macro specificity"),
        ("auroc_macro", "AUROC (macro, OvR)"),
        ("auprc_macro", "AUPRC (macro, OvR)"),
        ("mcc", "Matthews correlation"),
        ("kappa", "Cohen kappa"),
        ("brier", "Brier score (multiclass)"),
        ("ece", "Expected calibration error"),
    ]
)
LOWER_IS_BETTER = {"brier", "ece"}

# Natural range of each metric. Normal-theory confidence intervals can stray
# outside these bounds on small fold counts, which is meaningless in a table;
# intervals are clipped and the clipping is disclosed.
METRIC_BOUNDS: Dict[str, Tuple[float, float]] = {
    "accuracy": (0.0, 1.0),
    "balanced_accuracy": (0.0, 1.0),
    "macro_f1": (0.0, 1.0),
    "macro_precision": (0.0, 1.0),
    "macro_recall": (0.0, 1.0),
    "macro_specificity": (0.0, 1.0),
    "auroc_macro": (0.0, 1.0),
    "auprc_macro": (0.0, 1.0),
    "mcc": (-1.0, 1.0),
    "kappa": (-1.0, 1.0),
    "brier": (0.0, 2.0),
    "ece": (0.0, 1.0),
}


def expected_calibration_error(y_true: np.ndarray, prob: np.ndarray, n_bins: int = 15) -> float:
    """Equal-width binning ECE over the top-1 confidence."""
    if len(y_true) == 0:
        return float("nan")
    conf = prob.max(axis=1)
    pred = prob.argmax(axis=1)
    correct = (pred == y_true).astype(np.float64)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if sel.sum() == 0:
            continue
        ece += (sel.sum() / len(y_true)) * abs(correct[sel].mean() - conf[sel].mean())
    return float(ece)


def multiclass_brier(y_true: np.ndarray, prob: np.ndarray, n_cls: int) -> float:
    if len(y_true) == 0:
        return float("nan")
    onehot = np.zeros_like(prob)
    onehot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((prob - onehot) ** 2, axis=1)))


def macro_specificity(y_true: np.ndarray, y_pred: np.ndarray, n_cls: int) -> float:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    total = cm.sum()
    spec = []
    for c in range(n_cls):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = total - tp - fn - fp
        if (tn + fp) > 0:
            spec.append(tn / (tn + fp))
    return float(np.mean(spec)) if spec else float("nan")


def safe_auc(y_true: np.ndarray, prob: np.ndarray, n_cls: int, average: str = "macro") -> float:
    """One-vs-rest AUROC that survives folds where a class is absent."""
    present = np.unique(y_true)
    if len(present) < 2:
        return float("nan")
    try:
        if len(present) == n_cls:
            return float(roc_auc_score(y_true, prob, multi_class="ovr", average=average))
        aucs = []
        for c in present:
            yb = (y_true == c).astype(int)
            if yb.min() == yb.max():
                continue
            aucs.append(roc_auc_score(yb, prob[:, int(c)]))
        return float(np.mean(aucs)) if aucs else float("nan")
    except Exception:
        return float("nan")


def safe_auprc(y_true: np.ndarray, prob: np.ndarray, n_cls: int) -> float:
    present = np.unique(y_true)
    aps = []
    for c in present:
        yb = (y_true == c).astype(int)
        if yb.min() == yb.max():
            continue
        try:
            aps.append(average_precision_score(yb, prob[:, int(c)]))
        except Exception:
            continue
    return float(np.mean(aps)) if aps else float("nan")


def compute_metrics(y_true: np.ndarray, prob: np.ndarray, n_cls: int) -> Dict[str, float]:
    """The full endpoint panel for one set of predictions."""
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=np.float64)
    prob = prob / np.maximum(prob.sum(axis=1, keepdims=True), 1e-12)
    y_pred = prob.argmax(axis=1)
    if len(y_true) == 0:
        return {k: float("nan") for k in METRIC_LABELS}
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_specificity": macro_specificity(y_true, y_pred, n_cls),
        "auroc_macro": safe_auc(y_true, prob, n_cls),
        "auprc_macro": safe_auprc(y_true, prob, n_cls),
        "mcc": float(matthews_corrcoef(y_true, y_pred)) if len(np.unique(y_true)) > 1 else float("nan"),
        "kappa": float(cohen_kappa_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else float("nan"),
        "brier": multiclass_brier(y_true, prob, n_cls),
        "ece": expected_calibration_error(y_true, prob),
    }


def per_class_metrics(y_true: np.ndarray, prob: np.ndarray, classes: List[str]) -> pd.DataFrame:
    n_cls = len(classes)
    y_pred = prob.argmax(axis=1)
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_cls)), zero_division=0
    )
    rows = []
    for c in range(n_cls):
        yb = (y_true == c).astype(int)
        auc = float("nan")
        ap = float("nan")
        if yb.min() != yb.max():
            try:
                auc = float(roc_auc_score(yb, prob[:, c]))
                ap = float(average_precision_score(yb, prob[:, c]))
            except Exception:
                pass
        cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        rows.append(
            {
                "Class": classes[c],
                "Support": int(sup[c]),
                "Sensitivity": float(r[c]),
                "Specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan"),
                "PPV": float(p[c]),
                "NPV": float(tn / (tn + fn)) if (tn + fn) > 0 else float("nan"),
                "F1": float(f1[c]),
                "AUROC": auc,
                "AUPRC": ap,
            }
        )
    return pd.DataFrame(rows)


def bootstrap_ci(
    y_true: np.ndarray,
    prob: np.ndarray,
    n_cls: int,
    n_boot: int,
    alpha: float,
    seed: int = 0,
) -> Dict[str, Tuple[float, float, float]]:
    """
    Stratified non-parametric bootstrap over the pooled out-of-fold
    predictions. Returns {metric: (point_estimate, lo, hi)} using the
    percentile method.
    """
    rng = np.random.default_rng(seed)
    point = compute_metrics(y_true, prob, n_cls)
    if n_boot <= 0 or len(y_true) < 10:
        return {k: (v, float("nan"), float("nan")) for k, v in point.items()}

    by_class = {c: np.flatnonzero(y_true == c) for c in np.unique(y_true)}
    draws: Dict[str, List[float]] = defaultdict(list)
    for _ in range(n_boot):
        idx = np.concatenate(
            [rng.choice(ix, size=len(ix), replace=True) for ix in by_class.values() if len(ix)]
        )
        m = compute_metrics(y_true[idx], prob[idx], n_cls)
        for k, v in m.items():
            if not (isinstance(v, float) and math.isnan(v)):
                draws[k].append(v)

    out: Dict[str, Tuple[float, float, float]] = {}
    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    for k, v in point.items():
        vals = draws.get(k, [])
        if len(vals) >= 20:
            out[k] = (v, float(np.percentile(vals, lo_q)), float(np.percentile(vals, hi_q)))
        else:
            out[k] = (v, float("nan"), float("nan"))
    return out


# --------------------------------------------------------------------------- #
# DeLong test for correlated ROC curves                                        #
# --------------------------------------------------------------------------- #
def _midrank(x: np.ndarray) -> np.ndarray:
    order = np.argsort(x)
    z = x[order]
    n = len(x)
    t = np.zeros(n, dtype=np.float64)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n, dtype=np.float64)
    out[order] = t
    return out


def _fast_delong(preds_sorted: np.ndarray, m: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Sun & Xu (2014) fast DeLong. `preds_sorted` is [k, m+n] with the m positive
    cases first. Returns (aucs[k], covariance[k, k]).
    """
    k, total = preds_sorted.shape
    n = total - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    tx = np.vstack([_midrank(pos[r]) for r in range(k)])
    ty = np.vstack([_midrank(neg[r]) for r in range(k)])
    tz = np.vstack([_midrank(preds_sorted[r]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01).reshape(k, k)
    sy = np.cov(v10).reshape(k, k)
    return aucs, sx / m + sy / n


def delong_test(y_true_bin: np.ndarray, prob_a: np.ndarray, prob_b: np.ndarray) -> Dict[str, float]:
    """Two-sided DeLong test comparing two AUCs on the *same* binary sample."""
    y = np.asarray(y_true_bin).astype(int)
    if len(np.unique(y)) < 2:
        return {"auc_a": float("nan"), "auc_b": float("nan"), "z": float("nan"), "p": float("nan")}
    order = np.argsort(-y, kind="mergesort")  # positives first
    y_sorted = y[order]
    m = int(y_sorted.sum())
    preds = np.vstack([np.asarray(prob_a)[order], np.asarray(prob_b)[order]])
    try:
        aucs, cov = _fast_delong(preds, m)
    except Exception:
        return {"auc_a": float("nan"), "auc_b": float("nan"), "z": float("nan"), "p": float("nan")}
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        z = 0.0
        p = 1.0
    else:
        z = float((aucs[0] - aucs[1]) / math.sqrt(var))
        p = float(2.0 * st.norm.sf(abs(z)))
    return {"auc_a": float(aucs[0]), "auc_b": float(aucs[1]), "z": z, "p": p}


def delong_macro(
    y_true: np.ndarray, prob_a: np.ndarray, prob_b: np.ndarray, classes: List[str]
) -> pd.DataFrame:
    """
    Per-class one-vs-rest DeLong comparison with Holm-Bonferroni control.

    Caveat, stated here because it must also be stated in the manuscript:
    DeLong assumes the two AUCs come from one model each, evaluated on
    independent samples. Applied to cross-validated predictions - even
    single-repeat out-of-fold ones, where each sample is scored exactly once by
    a model that never saw it - the scores still originate from k different
    fitted models. The resulting p-values are therefore approximate and are
    reported as supportive of, not a substitute for, the fold-level paired
    Wilcoxon test, which respects the cross-validation structure.
    """
    rows = []
    for c, name in enumerate(classes):
        yb = (y_true == c).astype(int)
        res = delong_test(yb, prob_a[:, c], prob_b[:, c])
        rows.append({"Class": name, "AUC_reference": res["auc_a"], "AUC_comparator": res["auc_b"],
                     "Delta": res["auc_a"] - res["auc_b"], "z": res["z"], "p": res["p"]})
    df = pd.DataFrame(rows)
    df["p_holm"] = holm_bonferroni(df["p"].values)
    return df


def holm_bonferroni(pvals: np.ndarray) -> np.ndarray:
    """Step-down Holm adjustment; NaNs pass through untouched."""
    p = np.asarray(pvals, dtype=np.float64)
    finite = np.isfinite(p)
    adj = np.full_like(p, np.nan)
    idx = np.flatnonzero(finite)
    if idx.size == 0:
        return adj
    order = idx[np.argsort(p[idx])]
    m = len(order)
    running = 0.0
    for rank, i in enumerate(order):
        val = (m - rank) * p[i]
        running = max(running, val)
        adj[i] = min(running, 1.0)
    return adj


# --------------------------------------------------------------------------- #
# Decision curve analysis                                                      #
# --------------------------------------------------------------------------- #
def net_benefit(y_bin: np.ndarray, prob: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """Vickers & Elkin net benefit for a one-vs-rest decision rule."""
    n = len(y_bin)
    nb = np.zeros_like(thresholds, dtype=np.float64)
    for i, pt in enumerate(thresholds):
        pred = prob >= pt
        tp = float(np.sum(pred & (y_bin == 1)))
        fp = float(np.sum(pred & (y_bin == 0)))
        odds = pt / max(1.0 - pt, 1e-9)
        nb[i] = tp / n - (fp / n) * odds
    return nb


def decision_curve(y_true: np.ndarray, prob: np.ndarray, cls: int) -> Dict[str, np.ndarray]:
    y_bin = (y_true == cls).astype(int)
    th = np.linspace(0.01, 0.80, 80)
    prevalence = float(y_bin.mean())
    treat_all = np.array([prevalence - (1 - prevalence) * (t / max(1 - t, 1e-9)) for t in th])
    return {
        "thresholds": th,
        "model": net_benefit(y_bin, prob[:, cls], th),
        "treat_all": treat_all,
        "treat_none": np.zeros_like(th),
    }

### 1.10 INFERENTIAL COMPARISON ACROSS MODELS

In [ ]:
# =========================================================================== #
# SECTION 10 - INFERENTIAL COMPARISON ACROSS MODELS                           #
# =========================================================================== #
# Studentised range statistic q_alpha (alpha = 0.05, infinite df) for the
# Nemenyi post-hoc test, indexed by the number of compared models.
NEMENYI_Q05 = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031,
    9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268, 13: 3.313, 14: 3.354, 15: 3.391,
}


def friedman_nemenyi(fold_scores: Dict[str, List[float]], alpha: float = 0.05) -> Dict[str, Any]:
    """
    Omnibus Friedman test over models measured on the same folds, followed by
    the Nemenyi critical-difference post-hoc (Demsar, JMLR 2006) - the
    standard protocol for comparing classifiers over multiple datasets/folds.
    """
    models = [m for m, v in fold_scores.items() if len(v) > 0]
    if len(models) < 3:
        return {"applicable": False, "reason": "fewer than three models"}
    n_folds = min(len(fold_scores[m]) for m in models)
    mat = np.array([fold_scores[m][:n_folds] for m in models], dtype=np.float64)  # [k, N]
    if not np.all(np.isfinite(mat)) or n_folds < 3:
        return {"applicable": False, "reason": "insufficient or non-finite fold scores"}

    try:
        stat, p = st.friedmanchisquare(*[mat[i] for i in range(mat.shape[0])])
    except Exception as exc:
        return {"applicable": False, "reason": str(exc)}

    # Average ranks (rank 1 = best, hence rank the negated scores).
    ranks = np.apply_along_axis(st.rankdata, 0, -mat)
    avg_ranks = ranks.mean(axis=1)
    k = len(models)
    q = NEMENYI_Q05.get(k, 3.4)
    cd = float(q * math.sqrt(k * (k + 1) / (6.0 * n_folds)))
    return {
        "applicable": True,
        "statistic": float(stat),
        "p_value": float(p),
        "n_folds": int(n_folds),
        "models": models,
        "average_ranks": {m: float(r) for m, r in zip(models, avg_ranks)},
        "critical_difference": cd,
        "alpha": alpha,
    }


# The two-sided signed-rank test cannot reach p < 0.05 with fewer than six
# paired observations, so below this many folds we report the comparison as
# not-available rather than emitting an uninterpretable p-value.
WILCOXON_MIN_FOLDS = 6


def pairwise_wilcoxon(
    fold_scores: Dict[str, List[float]], reference: str
) -> pd.DataFrame:
    """
    Paired Wilcoxon signed-rank tests of every model against the reference,
    on fold-level scores, with Holm-Bonferroni multiplicity control and a
    rank-biserial effect size.
    """
    rows = []
    ref = np.asarray(fold_scores.get(reference, []), dtype=np.float64)
    for name, vals in fold_scores.items():
        if name == reference:
            continue
        arr = np.asarray(vals, dtype=np.float64)
        n = min(len(ref), len(arr))
        if n < WILCOXON_MIN_FOLDS or not np.all(np.isfinite(ref[:n])) or not np.all(np.isfinite(arr[:n])):
            # Still report the descriptive difference; withhold only the inference.
            delta = float(np.mean(ref[:n] - arr[:n])) if n > 0 else float("nan")
            rows.append({"Comparator": name, "Delta_mean": delta, "W": float("nan"),
                         "p": float("nan"), "effect_r": float("nan")})
            continue
        d = ref[:n] - arr[:n]
        if np.allclose(d, 0.0):
            rows.append({"Comparator": name, "Delta_mean": 0.0, "W": float("nan"),
                         "p": 1.0, "effect_r": 0.0})
            continue
        try:
            w, p = st.wilcoxon(ref[:n], arr[:n], zero_method="wilcox", alternative="two-sided")
        except Exception:
            w, p = float("nan"), float("nan")
        nz = d[d != 0]
        pos = float(np.sum(st.rankdata(np.abs(nz))[nz > 0]))
        total = float(np.sum(st.rankdata(np.abs(nz))))
        rows.append(
            {
                "Comparator": name,
                "Delta_mean": float(np.mean(d)),
                "W": float(w),
                "p": float(p),
                "effect_r": float(2 * pos / total - 1) if total > 0 else float("nan"),
            }
        )
    df = pd.DataFrame(rows)
    if not df.empty:
        df["p_holm"] = holm_bonferroni(df["p"].values)
        df.insert(0, "Reference", reference)
    return df

### 1.11 ALGORITHMIC FAIRNESS AUDIT (skin pigmentation strata)

In [ ]:
# =========================================================================== #
# SECTION 11 - ALGORITHMIC FAIRNESS AUDIT (skin pigmentation strata)          #
# =========================================================================== #
def ita_proxy_validity(df: pd.DataFrame, classes: List[str]) -> Dict[str, Any]:
    """
    Test whether the ITA skin-tone proxy is confounded by diagnosis.

    ITA is only a valid pigmentation measure on colour-calibrated photography.
    These images carry no colour reference card and were acquired on unknown
    devices under unknown illumination, so ITA may instead be tracking lesion
    chromaticity, white balance or exposure. If ITA differs strongly by
    diagnostic class, any "fairness across skin tone" claim is unsafe: the
    strata are partly disease strata.

    Kruskal-Wallis across classes, with epsilon-squared as the effect size.
    """
    out: Dict[str, Any] = {"available": False}
    if "ita" not in df.columns:
        return out
    groups = [df.loc[df["label"] == c, "ita"].dropna().to_numpy() for c in classes]
    groups = [g for g in groups if len(g) >= 5]
    if len(groups) < 2:
        return out
    try:
        stat, p = st.kruskal(*groups)
    except Exception as exc:
        return {"available": False, "reason": str(exc)}

    n = int(sum(len(g) for g in groups))
    k = len(groups)
    # epsilon-squared for Kruskal-Wallis; >= 0.14 is conventionally "large".
    eps2 = float((stat - k + 1) / max(n - k, 1)) if n > k else float("nan")
    confounded = bool(np.isfinite(eps2) and p < 0.05 and eps2 >= 0.06)
    out.update(
        {
            "available": True,
            "kruskal_H": float(stat),
            "p_value": float(p),
            "epsilon_squared": eps2,
            "median_ita_by_class": {
                c: (float(df.loc[df["label"] == c, "ita"].median())
                    if df.loc[df["label"] == c, "ita"].notna().any() else None)
                for c in classes
            },
            "confounded_by_diagnosis": confounded,
            "interpretation": (
                "ITA differs substantially by diagnosis, so the pigmentation strata are "
                "partly disease strata. Subgroup results must be read as exploratory and "
                "must not be described as a validated skin-tone fairness analysis."
                if confounded else
                "No strong association between ITA and diagnosis was detected, which is "
                "consistent with (but does not prove) ITA behaving as a pigmentation "
                "measure here. Colour calibration is still absent, so the proxy remains "
                "unvalidated."
            ),
        }
    )
    return out


def fairness_audit(
    y_true: np.ndarray, prob: np.ndarray, strata: np.ndarray, n_cls: int, min_n: int = 20
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    Subgroup performance across ITA-defined pigmentation strata.

    Reported because a model trained on a Bangladesh/Ghana/Kenya cohort must be
    shown to work across the full pigmentation range it claims to serve; the
    dermatology-AI literature's dominant failure mode is exactly this.
    """
    rows = []
    for group in sorted(set(strata.tolist())):
        sel = strata == group
        if sel.sum() < min_n:
            rows.append({"Stratum": group, "n": int(sel.sum()), "note": "too small to report"})
            continue
        m = compute_metrics(y_true[sel], prob[sel], n_cls)
        rows.append(
            {
                "Stratum": group,
                "n": int(sel.sum()),
                "Balanced accuracy": m["balanced_accuracy"],
                "Macro F1": m["macro_f1"],
                "Sensitivity": m["macro_recall"],
                "Specificity": m["macro_specificity"],
                "AUROC": m["auroc_macro"],
                "ECE": m["ece"],
                "note": "",
            }
        )
    df = pd.DataFrame(rows)
    gaps: Dict[str, float] = {}
    reportable = df[df.get("note", "") == ""] if "note" in df else df
    for col in ("Balanced accuracy", "Macro F1", "Sensitivity", "AUROC"):
        if col in reportable and reportable[col].notna().sum() >= 2:
            gaps[f"max_gap_{col.lower().replace(' ', '_')}"] = float(
                reportable[col].max() - reportable[col].min()
            )
    return df, gaps

### 1.12 CROSS-VALIDATED EXPERIMENT RUNNER

In [ ]:
# =========================================================================== #
# SECTION 12 - CROSS-VALIDATED EXPERIMENT RUNNER                              #
# =========================================================================== #
@dataclass
class ExperimentResult:
    classes: List[str]
    fold_metrics: pd.DataFrame                      # one row per (model, repeat, fold)
    # Out-of-fold probabilities from a SINGLE repeat: every sample is predicted
    # exactly once, by a model that never saw it. Averaging the repeats instead
    # would build an implicit ensemble and inflate every pooled metric, so the
    # repeat-averaged version is kept separately and never used for reporting.
    oof_prob: Dict[str, np.ndarray]                 # [n, C], single repeat
    oof_prob_uncal: Dict[str, np.ndarray]
    oof_prob_ensemble: Dict[str, np.ndarray]        # repeat-averaged; descriptive only
    histories: Dict[str, Dict[str, List[float]]]    # representative training curves
    graph_stats: Dict[str, Any]
    temperatures: Dict[str, float]
    n_params: Dict[str, int]
    runtime: Dict[str, float]


def stratified_val_split(
    idx: np.ndarray, y: np.ndarray, val_fraction: float, seed: int,
    groups: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carve a validation set out of the training portion.

    When `groups` is supplied the split is made over whole groups, so every
    image of a lesion or patient stays on one side. Otherwise it is stratified
    by class.
    """
    if groups is not None:
        rng = np.random.default_rng(seed)
        uniq = np.unique(groups[idx])
        rng.shuffle(uniq)
        n_val = max(int(round(len(uniq) * val_fraction)), 1)
        val_groups = set(uniq[:n_val].tolist())
        mask = np.array([g in val_groups for g in groups[idx]])
        tr, va = idx[~mask], idx[mask]
        if len(va) == 0 or len(tr) == 0:
            tr, va = idx[:-1], idx[-1:]
        return np.asarray(tr), np.asarray(va)
    try:
        tr, va = train_test_split(
            idx, test_size=val_fraction, stratify=y[idx], random_state=seed, shuffle=True
        )
    except ValueError:  # a class has too few members to stratify
        tr, va = train_test_split(idx, test_size=val_fraction, random_state=seed, shuffle=True)
    if len(va) == 0:  # degenerate tiny-fold guard
        tr, va = idx[:-1], idx[-1:]
    return np.asarray(tr), np.asarray(va)


def make_cv_splitter(cfg: Config, groups: Optional[np.ndarray], rep: int):
    """
    Return an object exposing .split(X, y, groups).

    With grouping, StratifiedGroupKFold keeps every image of a lesion or patient
    within one fold while still balancing classes. This is the difference
    between a defensible estimate and one inflated by near-identical images of
    the same lesion sitting on both sides of a split.
    """
    if groups is None:
        return StratifiedKFold(n_splits=cfg.folds, shuffle=True, random_state=cfg.seed + rep)
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        return StratifiedGroupKFold(n_splits=cfg.folds, shuffle=True, random_state=cfg.seed + rep)
    except ImportError:  # scikit-learn < 0.24
        from sklearn.model_selection import GroupKFold
        LOGGER.warning("StratifiedGroupKFold unavailable; falling back to GroupKFold "
                       "(class balance across folds is not guaranteed).")
        return GroupKFold(n_splits=cfg.folds)


def run_cross_validation(
    cfg: Config,
    X_np: np.ndarray,
    y: np.ndarray,
    S: np.ndarray,
    classes: List[str],
    device: torch.device,
    models: Optional[List[str]] = None,
    verbose: bool = True,
    raw_feats: Optional[np.ndarray] = None,
    image_paths: Optional[Sequence[str]] = None,
    groups: Optional[np.ndarray] = None,
) -> ExperimentResult:
    """
    Repeated stratified k-fold cross-validation.

    For each (repeat, fold) the data are partitioned into train / validation /
    test. Validation is used *only* for early stopping and temperature
    scaling; every reported number comes from the untouched test partition.

    When `cfg.feature_fit == "foldwise"` and `raw_feats` is supplied, the
    standardisation, PCA projection and similarity graph are all rebuilt from
    the fold's training rows alone, so nothing about the held-out partition
    influences the representation. `X_np` / `S` are then used only as the
    fallback for the "global" mode.
    """
    models = models or cfg.models
    n, n_cls = len(y), len(classes)
    foldwise = cfg.feature_fit == "foldwise" and raw_feats is not None
    X_t_global = torch.as_tensor(X_np, dtype=torch.float32, device=device)
    y_t = torch.as_tensor(y, dtype=torch.long, device=device)

    rows: List[Dict[str, Any]] = []
    oof_sum = {m: np.zeros((n, n_cls), dtype=np.float64) for m in models}
    oof_sum_uncal = {m: np.zeros((n, n_cls), dtype=np.float64) for m in models}
    oof_count = {m: np.zeros(n, dtype=np.float64) for m in models}
    # Predictions from repeat 0 only, kept separate so pooled metrics describe a
    # single model rather than an ensemble of repeats.
    oof_single = {m: np.full((n, n_cls), np.nan) for m in models}
    oof_single_uncal = {m: np.full((n, n_cls), np.nan) for m in models}
    histories: Dict[str, Dict[str, List[float]]] = {}
    temps: Dict[str, List[float]] = defaultdict(list)
    nparams: Dict[str, int] = {}
    runtime: Dict[str, float] = defaultdict(float)
    graph_stats: Dict[str, Any] = {}

    total = cfg.repeats * cfg.folds
    step = 0
    for rep in range(cfg.repeats):
        skf = make_cv_splitter(cfg, groups, rep)
        for fold, (trval_idx, test_idx) in enumerate(skf.split(np.zeros(n), y, groups)):
            step += 1
            tr_idx, val_idx = stratified_val_split(
                trval_idx, y, cfg.val_fraction, cfg.seed + rep * 100 + fold, groups
            )

            # Optionally adapt the encoder itself to this fold's training set,
            # then re-embed every image with it. Strictly inside the fold.
            fold_raw = raw_feats
            if cfg.finetune and image_paths is not None:
                try:
                    ft_model, ft_info = finetune_backbone(
                        cfg, image_paths, y, tr_idx, val_idx, n_cls, device,
                        cfg.seed + rep * 1000 + fold,
                    )
                    fold_raw = embed_with_model(ft_model, cfg, image_paths, device)
                    del ft_model
                    if device.type == "cuda":
                        torch.cuda.empty_cache()
                    if verbose:
                        LOGGER.info("    encoder fine-tuned: val bAcc=%.3f over %d epochs",
                                    ft_info["val_bacc"], ft_info["epochs"])
                except Exception as exc:
                    LOGGER.error("Fine-tuning failed on repeat %d fold %d (%s); "
                                 "falling back to frozen features.", rep, fold, exc)
                    fold_raw = raw_feats

            # Representation and graph are rebuilt from this fold's training
            # rows when foldwise fitting is enabled.
            if foldwise or fold_raw is not raw_feats:
                base = fold_raw if fold_raw is not None else raw_feats
                transform, _ = fit_reducer(base[tr_idx], cfg.pca_dim, cfg.seed)
                X_fold = transform(base)
                S_fold = similarity_matrix(X_fold, cfg.graph_metric)
                X_t = torch.as_tensor(X_fold, dtype=torch.float32, device=device)
            else:
                X_fold, S_fold, X_t = X_np, S, X_t_global

            graphs = build_fold_graphs(S_fold, cfg, y, tr_idx, val_idx, test_idx, device)
            if not graph_stats:
                graph_stats = dict(graphs.stats)

            for name in models:
                t0 = time.time()
                seed = cfg.seed + rep * 1000 + fold * 10
                try:
                    if name in TORCH_MODELS:
                        res = train_torch_model(
                            name, cfg, X_t, y_t, graphs, tr_idx, val_idx, test_idx,
                            n_cls, device, seed,
                        )
                    else:
                        res = train_classical_model(
                            name, cfg, X_fold, y, tr_idx, val_idx, test_idx, seed
                        )
                except Exception as exc:  # never let one model abort the study
                    LOGGER.error("Model %s failed on repeat %d fold %d: %s", name, rep, fold, exc)
                    continue
                runtime[name] += time.time() - t0

                met = compute_metrics(y[test_idx], res["test_prob"], n_cls)
                rows.append({"model": name, "repeat": rep, "fold": fold,
                             "n_test": len(test_idx), **met})
                oof_sum[name][test_idx] += res["test_prob"]
                oof_sum_uncal[name][test_idx] += res["test_prob_uncal"]
                oof_count[name][test_idx] += 1.0
                if rep == 0:
                    oof_single[name][test_idx] = res["test_prob"]
                    oof_single_uncal[name][test_idx] = res["test_prob_uncal"]
                temps[name].append(res["temperature"])
                if res["n_params"] > 0:
                    nparams[name] = res["n_params"]
                if name not in histories and res["history"]["train_loss"]:
                    histories[name] = res["history"]

            if verbose:
                ref = [r for r in rows if r["repeat"] == rep and r["fold"] == fold
                       and r["model"] == cfg.reference_model]
                extra = f" | {cfg.reference_model} bAcc={ref[-1]['balanced_accuracy']:.3f}" if ref else ""
                LOGGER.info(
                    "  split %2d/%2d (repeat %d, fold %d)  n_train=%d n_val=%d n_test=%d%s",
                    step, total, rep, fold, len(tr_idx), len(val_idx), len(test_idx), extra,
                )

    fold_df = pd.DataFrame(rows)

    def _normalise(p: np.ndarray) -> np.ndarray:
        p = np.nan_to_num(p, nan=1.0 / max(n_cls, 1))
        return p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)

    oof_prob, oof_uncal, oof_ens = {}, {}, {}
    for m in models:
        cnt = np.maximum(oof_count[m], 1.0)[:, None]
        oof_ens[m] = _normalise(oof_sum[m] / cnt)
        # Reporting uses repeat 0 alone; if a model produced nothing there,
        # fall back to the averaged version rather than returning NaNs.
        if np.isnan(oof_single[m]).all():
            oof_prob[m] = oof_ens[m]
            oof_uncal[m] = _normalise(oof_sum_uncal[m] / cnt)
        else:
            oof_prob[m] = _normalise(oof_single[m])
            oof_uncal[m] = _normalise(oof_single_uncal[m])

    return ExperimentResult(
        classes=classes,
        fold_metrics=fold_df,
        oof_prob=oof_prob,
        oof_prob_uncal=oof_uncal,
        oof_prob_ensemble=oof_ens,
        histories=histories,
        graph_stats=graph_stats,
        temperatures={m: float(np.mean(v)) if v else 1.0 for m, v in temps.items()},
        n_params=nparams,
        runtime=dict(runtime),
    )


def corrected_fold_ci(
    scores: Sequence[float], n_train: int, n_test: int, alpha: float = 0.05,
    bounds: Optional[Tuple[float, float]] = None,
) -> Tuple[float, float, float]:
    """
    Confidence interval for a repeated k-fold estimate, using the Nadeau &
    Bengio (2003) variance correction.

    Cross-validation folds overlap in their training data, so the naive
    standard error of the fold scores is badly optimistic. The correction
    inflates the variance by (1/k + n_test/n_train) to account for that
    dependence. This is the interval reported as primary, because it reflects
    the variability of the whole estimation procedure rather than of one
    particular partition.

    The interval is normal-theory, so with few folds it can extend past a
    metric's natural range (a balanced accuracy above 1, say). When `bounds`
    are supplied the interval is clipped to them; the clipping is disclosed in
    the report so the interval is not mistaken for a tighter estimate.
    """
    vals = np.asarray([v for v in scores if np.isfinite(v)], dtype=np.float64)
    k = len(vals)
    if k == 0:
        return float("nan"), float("nan"), float("nan")
    mean = float(vals.mean())
    if k < 2 or n_train <= 0:
        return mean, float("nan"), float("nan")
    var = float(vals.var(ddof=1))
    corrected = var * (1.0 / k + n_test / max(n_train, 1))
    if corrected <= 0:
        return mean, mean, mean
    half = float(st.t.ppf(1.0 - alpha / 2.0, df=k - 1) * math.sqrt(corrected))
    lo, hi = mean - half, mean + half
    if bounds is not None:
        lo = max(lo, bounds[0])
        hi = min(hi, bounds[1])
    return mean, lo, hi


def summarise_results(
    cfg: Config, res: ExperimentResult, y: np.ndarray
) -> Tuple[pd.DataFrame, Dict[str, Dict[str, Tuple[float, float, float]]],
           Dict[str, Dict[str, Tuple[float, float, float]]]]:
    """
    Build the headline results table.

    Two uncertainty estimates are produced, and the distinction matters:

      * **primary** - mean across cross-validation folds with a Nadeau-Bengio
        corrected 95% interval. This is what the tables and figures report,
        because it accounts for the dependence between overlapping training
        sets and therefore for the variability of the procedure itself.
      * **secondary** - percentile bootstrap over the pooled single-repeat
        out-of-fold predictions. Narrower by construction, since it conditions
        on one particular partition; retained for comparability with the
        literature but explicitly labelled as such.
    """
    n_cls = len(res.classes)
    boot: Dict[str, Dict[str, Tuple[float, float, float]]] = {}
    primary: Dict[str, Dict[str, Tuple[float, float, float]]] = {}
    rows = []
    n_total = len(y)
    for m in cfg.models:
        if m not in res.oof_prob or res.fold_metrics.empty:
            continue
        sub = res.fold_metrics[res.fold_metrics["model"] == m]
        if sub.empty:
            continue
        boot[m] = bootstrap_ci(y, res.oof_prob[m], n_cls, cfg.bootstrap, cfg.alpha, seed=cfg.seed)

        n_test = float(sub["n_test"].mean()) if "n_test" in sub else n_total / max(cfg.folds, 2)
        n_train = max(n_total - n_test, 1.0)
        primary[m] = {
            key: corrected_fold_ci(sub[key].tolist(), int(n_train), int(n_test), cfg.alpha,
                                   METRIC_BOUNDS.get(key))
            for key in METRIC_LABELS
        }

        row: Dict[str, Any] = {"Model": m, "n_folds": int(len(sub))}
        for key, label in METRIC_LABELS.items():
            pe, lo, hi = primary[m][key]
            row[label] = pe
            row[f"{label} 95% CI low"] = lo
            row[f"{label} 95% CI high"] = hi
            row[f"{label} fold SD"] = float(sub[key].std(ddof=1)) if len(sub) > 1 else 0.0
            bpe, blo, bhi = boot[m][key]
            row[f"{label} pooled OOF"] = bpe
            row[f"{label} pooled 95% CI low"] = blo
            row[f"{label} pooled 95% CI high"] = bhi
        row["Parameters"] = res.n_params.get(m, np.nan)
        row["Train time (s/fold)"] = res.runtime.get(m, np.nan) / max(len(sub), 1)
        rows.append(row)
    table = pd.DataFrame(rows)
    if not table.empty:
        table = table.sort_values(METRIC_LABELS[PRIMARY_METRIC], ascending=False).reset_index(drop=True)
    return table, primary, boot


def run_ablations(
    cfg: Config, X_np: np.ndarray, y: np.ndarray, S: np.ndarray,
    classes: List[str], device: torch.device, raw_feats: Optional[np.ndarray] = None,
) -> pd.DataFrame:
    """
    Systematic sensitivity analyses on the proposed model. Each setting is
    re-cross-validated with a reduced budget; all other hyperparameters are
    held at their defaults.
    """
    banner("Ablation studies")
    base = asdict(cfg)
    rows: List[Dict[str, Any]] = []

    def variant(tag: str, value: Any, **overrides) -> None:
        c = Config(**base)
        for k, v in overrides.items():
            setattr(c, k, v)
        c.repeats = 1
        c.folds = max(3, min(cfg.folds, 5))
        c.epochs = min(cfg.epochs, 150)
        c.bootstrap = 0
        c.models = [cfg.reference_model]
        c.reference_model = cfg.reference_model
        try:
            r = run_cross_validation(c, X_np, y, S, classes, device,
                                     models=[cfg.reference_model], verbose=False,
                                     raw_feats=raw_feats)
            sub = r.fold_metrics
            if sub.empty:
                return
            rows.append(
                {
                    "Ablation": tag,
                    "Setting": value,
                    "Balanced accuracy": float(sub["balanced_accuracy"].mean()),
                    "SD": float(sub["balanced_accuracy"].std(ddof=1)) if len(sub) > 1 else 0.0,
                    "Macro F1": float(sub["macro_f1"].mean()),
                    "AUROC": float(sub["auroc_macro"].mean()),
                    "Mean degree": float(r.graph_stats.get("mean_degree", np.nan)),
                    "Homophily": float(r.graph_stats.get("homophily", np.nan)),
                }
            )
            LOGGER.info("  %-22s %-14s bAcc=%.3f", tag, str(value), rows[-1]["Balanced accuracy"])
        except Exception as exc:
            LOGGER.error("  ablation %s=%s failed: %s", tag, value, exc)

    for k in [3, 5, 10, 20, 40]:
        variant("Neighbourhood size k", k, knn_k=k)
    for L in [1, 2, 3, 4]:
        variant("Propagation depth", L, num_layers=L)
    for mode in ["inductive", "transductive"]:
        variant("Graph protocol", mode, graph_mode=mode)
    for de in [0.0, 0.1, 0.3]:
        variant("DropEdge rate", de, drop_edge=de)
    for pca in [64, 128, 256]:
        variant("PCA dimensionality", pca, pca_dim=pca)
    for fit in ["foldwise", "global"]:
        variant("Feature-fit scope", fit, feature_fit=fit)
    variant("Class weighting", "off", use_class_weights=False)
    variant("Focal loss", "gamma=0 (plain CE)", focal_gamma=0.0)

    return pd.DataFrame(rows)

### 1.13 EXPLAINABILITY

In [ ]:
# =========================================================================== #
# SECTION 13 - EXPLAINABILITY                                                 #
# =========================================================================== #
class CentreMaskedDataset(ImageListDataset):
    """Image dataset that blanks the central square, leaving only periphery."""

    def __init__(self, paths: Sequence[str], transform, mask_fraction: float) -> None:
        super().__init__(paths, transform)
        self.mask_fraction = float(mask_fraction)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        x, i = super().__getitem__(idx)
        _, h, w = x.shape
        mh, mw = int(h * self.mask_fraction), int(w * self.mask_fraction)
        top, left = (h - mh) // 2, (w - mw) // 2
        x[:, top:top + mh, left:left + mw] = 0.0  # zero == ImageNet mean after normalisation
        return x, i


@torch.no_grad()
def extract_periphery_features(
    cfg: Config, df: pd.DataFrame, device: torch.device
) -> np.ndarray:
    """Embed images whose centre has been masked out (background/context only)."""
    model, dim = load_backbone(cfg.backbone.split(",")[0].strip())
    model = model.to(device)
    ds = CentreMaskedDataset(
        df["path"].tolist(), build_eval_transform(cfg.image_size, cfg.center_crop_frac), cfg.shortcut_mask_fraction
    )
    loader = torch.utils.data.DataLoader(
        ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
    )
    feats = np.zeros((len(ds), dim), dtype=np.float32)
    for batch, idx in tqdm(loader, desc="periphery embedding", leave=False):
        out = model(batch.to(device, non_blocking=True)).float().flatten(1)
        feats[idx.numpy()] = out.cpu().numpy()
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return feats


def shortcut_learning_test(
    cfg: Config, df: pd.DataFrame, y: np.ndarray, classes: List[str], device: torch.device
) -> Dict[str, Any]:
    """
    Quantify how much of the model's performance could come from acquisition
    context rather than the lesion.

    The centre of every image - where the lesion almost always sits - is masked
    out, and the reference model is re-cross-validated on what remains:
    background, framing, skin at the margins, lighting, device characteristics.
    Balanced accuracy meaningfully above chance means the corpus carries a
    shortcut, and the headline result is partly explained by it.

    This matters here because the healthy-skin images plausibly come from a
    different acquisition setting than the clinical disease photographs.
    """
    banner("Shortcut-learning probe (centre-masked images)")
    chance = 1.0 / max(len(classes), 1)
    try:
        raw = extract_periphery_features(cfg, df, device)
    except Exception as exc:
        LOGGER.error("Shortcut probe failed during feature extraction: %s", exc)
        return {"available": False, "reason": str(exc)}

    probe = Config(**asdict(cfg))
    probe.repeats = 1
    probe.folds = max(3, min(cfg.folds, 5))
    probe.epochs = min(cfg.epochs, 150)
    probe.bootstrap = 0
    probe.models = [cfg.reference_model]
    probe.shortcut_test = False

    X_probe, _ = reduce_features(raw, cfg.pca_dim, cfg.seed)
    S_probe = similarity_matrix(X_probe, cfg.graph_metric)
    try:
        res = run_cross_validation(probe, X_probe, y, S_probe, classes, device,
                                   models=[cfg.reference_model], verbose=False, raw_feats=raw)
    except Exception as exc:
        LOGGER.error("Shortcut probe failed during cross-validation: %s", exc)
        return {"available": False, "reason": str(exc)}

    sub = res.fold_metrics
    if sub.empty:
        return {"available": False, "reason": "no folds completed"}

    bacc = float(sub["balanced_accuracy"].mean())
    sd = float(sub["balanced_accuracy"].std(ddof=1)) if len(sub) > 1 else 0.0
    # One-sample t-test of the fold scores against chance.
    try:
        _, p = st.ttest_1samp(sub["balanced_accuracy"].to_numpy(), chance)
        p = float(p)
    except Exception:
        p = float("nan")
    above = bool(np.isfinite(p) and p < 0.05 and bacc > chance)

    severity = "none detected"
    if above:
        margin = (bacc - chance) / max(1.0 - chance, 1e-9)
        severity = "severe" if margin > 0.5 else "moderate" if margin > 0.25 else "mild"

    LOGGER.info("  balanced accuracy on background only: %.3f (chance %.3f), p = %.3g",
                bacc, chance, p)
    LOGGER.info("  shortcut signal: %s", severity)

    return {
        "available": True,
        "mask_fraction": cfg.shortcut_mask_fraction,
        "balanced_accuracy": bacc,
        "sd": sd,
        "chance": chance,
        "p_vs_chance": p,
        "above_chance": above,
        "severity": severity,
        "macro_f1": float(sub["macro_f1"].mean()),
        "interpretation": (
            f"Background alone reaches {bacc:.1%} balanced accuracy against a {chance:.1%} "
            f"chance level ({severity} shortcut). Part of the headline performance is "
            "attributable to acquisition context rather than the lesion, and the corpus "
            "should be re-curated so that all classes share an acquisition setting."
            if above else
            f"Background alone does not exceed chance ({bacc:.1%} vs {chance:.1%}, "
            f"p = {p:.3g}), which argues against acquisition context driving the result."
        ),
    }


def spatial_backbone(name: str) -> Tuple[nn.Module, int]:
    """Return the convolutional trunk (before global pooling) of a backbone."""
    model, dim = load_backbone(name)
    if name.startswith("resnet"):
        trunk = nn.Sequential(*list(model.children())[:-2])
    elif name.startswith("densenet"):
        trunk = nn.Sequential(model.features, nn.ReLU(inplace=False))
    elif name.startswith("efficientnet") or name.startswith("convnext"):
        trunk = model.features
    else:
        trunk = nn.Sequential(*list(model.children())[:-2])
    trunk.eval()
    return trunk, dim


@torch.no_grad()
def class_activation_maps(
    cfg: Config, df: pd.DataFrame, raw_feats: np.ndarray, y: np.ndarray,
    classes: List[str], device: torch.device, per_class: int = 2,
) -> List[Dict[str, Any]]:
    """
    Class activation mapping.

    Because the classifier head sits on globally average-pooled features, the
    CAM for class c is exactly the weighted sum of the final feature maps with
    the head's weights for c - no gradient approximation is required. We fit a
    multinomial logistic head on the frozen embeddings to obtain those weights.
    """
    LOGGER.info("Computing class activation maps ...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        head = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=cfg.seed)
        head.fit(StandardScaler().fit_transform(raw_feats), y)
    scaler = StandardScaler().fit(raw_feats)
    W = head.coef_ / np.maximum(scaler.scale_, 1e-8)  # undo standardisation
    if W.shape[0] == 1:  # binary edge case
        W = np.vstack([-W, W])

    trunk, _ = spatial_backbone(cfg.backbone.split(",")[0].strip())
    trunk = trunk.to(device)
    tf = build_eval_transform(cfg.image_size, cfg.center_crop_frac)

    picks: List[int] = []
    for c in range(len(classes)):
        idx = np.flatnonzero(y == c)
        picks.extend(idx[: per_class].tolist())

    out: List[Dict[str, Any]] = []
    for i in picks:
        path = df.iloc[i]["path"]
        try:
            with Image.open(path) as im:
                pil = im.convert("RGB")
            x = tf(pil).unsqueeze(0).to(device)
            fmap = trunk(x)  # [1, K, h, w]
            if fmap.dim() != 4:
                continue
            K = fmap.shape[1]
            w = W[int(y[i])]
            if w.shape[0] != K:  # dimensionality mismatch across pooling variants
                w = np.resize(w, K)
            cam = (fmap[0].cpu().numpy() * w[:, None, None]).sum(0)
            cam = np.maximum(cam, 0)
            cam = cam / max(cam.max(), 1e-8)
            disp = np.asarray(pil.resize((cfg.image_size, cfg.image_size), Image.BILINEAR)) / 255.0
            cam_img = np.asarray(
                Image.fromarray((cam * 255).astype(np.uint8)).resize(
                    (cfg.image_size, cfg.image_size), Image.BILINEAR
                )
            ) / 255.0
            out.append({"image": disp, "cam": cam_img, "label": classes[int(y[i])]})
        except Exception as exc:
            LOGGER.debug("CAM failed for %s: %s", path, exc)
            continue
    del trunk
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out


def embed_2d(X: np.ndarray, seed: int, method: str = "tsne") -> Tuple[np.ndarray, str]:
    """2-D projection for manifold visualisation, with a safe perplexity."""
    n = len(X)
    if n < 6:
        return np.zeros((n, 2)), "unavailable"
    if method == "umap" and _HAS_UMAP:
        try:
            Z = umap.UMAP(n_neighbors=min(15, n - 1), min_dist=0.1, random_state=seed).fit_transform(X)
            return np.asarray(Z), "UMAP"
        except Exception:
            pass
    perp = float(max(5, min(30, (n - 1) // 3)))
    try:
        Z = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto",
                 random_state=seed).fit_transform(X)
    except TypeError:  # older scikit-learn
        Z = TSNE(n_components=2, perplexity=perp, random_state=seed).fit_transform(X)
    return np.asarray(Z), "t-SNE"


@torch.no_grad()
def learned_node_embeddings(
    cfg: Config, X_np: np.ndarray, y: np.ndarray, S: np.ndarray,
    classes: List[str], device: torch.device,
) -> Optional[np.ndarray]:
    """Fit the reference model once on a single split and export node embeddings."""
    try:
        n = len(y)
        idx = np.arange(n)
        tr, te = train_test_split(idx, test_size=0.2, stratify=y, random_state=cfg.seed)
        tr, va = stratified_val_split(tr, y, cfg.val_fraction, cfg.seed)
        graphs = build_fold_graphs(S, cfg, y, tr, va, te, device)
        X_t = torch.as_tensor(X_np, dtype=torch.float32, device=device)
        y_t = torch.as_tensor(y, dtype=torch.long, device=device)
        set_seed(cfg.seed)
        model = build_model(cfg.reference_model, cfg, X_t.size(1), len(classes)).to(device)
        weights = class_weights_from(y[tr], len(classes)).to(device) if cfg.use_class_weights else None
        crit = FocalLoss(weights, cfg.focal_gamma, cfg.label_smoothing).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        tr_t = torch.as_tensor(tr, dtype=torch.long, device=device)
        ei, ew = graphs.train
        with torch.enable_grad():
            for _ in range(min(cfg.epochs, 200)):
                model.train()
                opt.zero_grad(set_to_none=True)
                loss = crit(model(X_t, ei, ew).index_select(0, tr_t), y_t.index_select(0, tr_t))
                loss.backward()
                opt.step()
        model.eval()
        ei_te, ew_te = graphs.test
        return model.embed(X_t, ei_te, ew_te).float().cpu().numpy()
    except Exception as exc:
        LOGGER.warning("Could not extract learned embeddings: %s", exc)
        return None

### 1.14 PUBLICATION FIGURES

In [ ]:
# =========================================================================== #
# SECTION 14 - PUBLICATION FIGURES                                            #
# =========================================================================== #
def fig_cohort(df: pd.DataFrame, flow: Dict[str, Any], classes: List[str], cfg: Config, outdir: Path) -> None:
    """Figure 1 - cohort composition, image geometry and pigmentation spread."""
    has_ita = "ita" in df.columns and df["ita"].notna().any()
    ncols = 3 if has_ita else 2
    fig, axes = plt.subplots(1, ncols, figsize=(3.6 * ncols, 3.2))

    ax = axes[0]
    counts = [int((df["label"] == c).sum()) for c in classes]
    bars = ax.barh(classes, counts, color=[PALETTE[i % len(PALETTE)] for i in range(len(classes))])
    for b, v in zip(bars, counts):
        ax.text(b.get_width() + max(counts) * 0.015, b.get_y() + b.get_height() / 2,
                str(v), va="center", fontsize=8)
    ax.set_xlabel("Images (n)")
    ax.set_title(f"A  Class distribution (N = {len(df)})", loc="left", fontweight="bold")
    ax.set_xlim(0, max(counts) * 1.18)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)

    ax = axes[1]
    ax.scatter(df["width"], df["height"], s=8, alpha=0.4, color=PALETTE[0], edgecolors="none")
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    ax.set_title("B  Native image geometry", loc="left", fontweight="bold")
    ax.set_xscale("log")
    ax.set_yscale("log")

    if has_ita:
        ax = axes[2]
        vals = df["ita"].dropna().to_numpy()
        ax.hist(vals, bins=30, color=PALETTE[2], alpha=0.85, edgecolor="white", linewidth=0.4)
        for thr, lab in [(41, "light"), (10, "intermediate")]:
            ax.axvline(thr, color="0.35", ls="--", lw=0.9)
            ax.text(thr, ax.get_ylim()[1] * 0.95, f" {lab}", fontsize=7, color="0.35", va="top")
        ax.set_xlabel("Individual Typology Angle (degrees)")
        ax.set_ylabel("Images")
        ax.set_title("C  Skin pigmentation spread", loc="left", fontweight="bold")

    fig.suptitle(
        "Figure 1 | Analysis cohort after quality control "
        f"({flow.get('files_found', 0)} files screened, {len(df)} analysed)",
        y=1.04, fontsize=10, fontweight="bold",
    )
    fig.tight_layout()
    savefig(fig, outdir, "fig01_cohort", cfg.dpi)


def fig_graph(S: np.ndarray, cfg: Config, y: np.ndarray, classes: List[str], outdir: Path) -> None:
    """Figure 2 - population-graph properties: degree, homophily, connectivity."""
    n = len(y)
    all_mask = np.ones(n, bool)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))

    ei, _ = build_knn_edges(S, cfg.knn_k, all_mask, all_mask, cfg.mutual_knn, cfg.edge_sim_threshold)
    deg = np.bincount(ei[1], minlength=n) if ei.shape[1] else np.zeros(n)
    axes[0].hist(deg, bins=max(10, int(deg.max()) + 1), color=PALETTE[0], edgecolor="white", linewidth=0.4)
    axes[0].set_xlabel("In-degree")
    axes[0].set_ylabel("Nodes")
    axes[0].set_title(f"A  Degree distribution (k = {cfg.knn_k})", loc="left", fontweight="bold")

    ks = [1, 2, 3, 5, 8, 10, 15, 20, 30, 40]
    homo = []
    for k in ks:
        e, _ = build_knn_edges(S, k, all_mask, all_mask, cfg.mutual_knn, cfg.edge_sim_threshold)
        st_ = graph_statistics(e, n, y, all_mask)
        homo.append(st_["homophily"])
    axes[1].plot(ks, homo, "o-", color=PALETTE[1], markersize=4)
    axes[1].axhline(1.0 / len(classes), color="0.4", ls="--", lw=0.9)
    axes[1].text(ks[-1], 1.0 / len(classes), " chance", fontsize=7, color="0.4", va="bottom", ha="right")
    axes[1].axvline(cfg.knn_k, color=PALETTE[3], ls=":", lw=1.2)
    axes[1].set_xscale("log")
    axes[1].set_xlabel("Neighbourhood size k")
    axes[1].set_ylabel("Edge homophily")
    axes[1].set_ylim(0, 1.02)
    axes[1].set_title("B  Label homophily vs k", loc="left", fontweight="bold")

    # Class-to-class edge affinity: where does the graph confuse conditions?
    if ei.shape[1]:
        M = np.zeros((len(classes), len(classes)))
        for s, d in zip(ei[0], ei[1]):
            M[y[d], y[s]] += 1
        M = M / np.maximum(M.sum(axis=1, keepdims=True), 1e-9)
    else:
        M = np.zeros((len(classes), len(classes)))
    im = axes[2].imshow(M, cmap="Blues", vmin=0, vmax=1)
    axes[2].set_xticks(range(len(classes)))
    axes[2].set_yticks(range(len(classes)))
    axes[2].set_xticklabels([c.split()[0] for c in classes], rotation=45, ha="right", fontsize=7)
    axes[2].set_yticklabels([c.split()[0] for c in classes], fontsize=7)
    axes[2].set_xlabel("Neighbour class")
    axes[2].set_ylabel("Node class")
    axes[2].set_title("C  Neighbourhood composition", loc="left", fontweight="bold")
    axes[2].grid(False)
    fig.colorbar(im, ax=axes[2], fraction=0.046, shrink=0.85, label="Proportion")

    fig.suptitle("Figure 2 | Structure of the k-nearest-neighbour population graph",
                 y=1.04, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig02_graph", cfg.dpi)


def fig_performance(
    table: pd.DataFrame, primary: Dict[str, Dict[str, Tuple[float, float, float]]],
    res: ExperimentResult, cfg: Config, outdir: Path,
) -> None:
    """Figure 3 - forest plot of the primary endpoint plus fold-level dispersion."""
    boot = primary  # intervals plotted are the corrected fold-level ones
    if table.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.15, 1]})

    label = METRIC_LABELS[PRIMARY_METRIC]
    order = table.sort_values(label)["Model"].tolist()
    ax = axes[0]
    for i, m in enumerate(order):
        pe, lo, hi = boot[m][PRIMARY_METRIC]
        col = model_color(m)
        lw = 2.4 if m == cfg.reference_model else 1.4
        if np.isfinite(lo) and np.isfinite(hi):
            ax.plot([lo, hi], [i, i], color=col, lw=lw, solid_capstyle="round")
        ax.plot(pe, i, "o", color=col, markersize=7 if m == cfg.reference_model else 5,
                markeredgecolor="white", markeredgewidth=0.8, zorder=3)
        ax.text(1.005, i, fmt_ci(pe, lo, hi), transform=ax.get_yaxis_transform(),
                va="center", fontsize=7.5, color="0.25")
    ax.axvline(1.0 / len(res.classes), color="0.5", ls="--", lw=0.9)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([f"$\\bf{{{m}}}$" if m == cfg.reference_model else m for m in order])
    ax.set_xlabel(f"{label} (fold mean, corrected 95% CI)")
    ax.set_xlim(min(0.15, 1.0 / len(res.classes) - 0.05), 1.0)
    ax.set_title("A  Primary endpoint", loc="left", fontweight="bold")
    ax.grid(axis="y", visible=False)

    ax = axes[1]
    data, labels, colors = [], [], []
    for m in order:
        sub = res.fold_metrics[res.fold_metrics["model"] == m]["macro_f1"].dropna()
        if len(sub):
            data.append(sub.to_numpy())
            labels.append(m)
            colors.append(model_color(m))
    if data:
        try:  # matplotlib >= 3.11 renamed `vert` to `orientation`
            bp = ax.boxplot(data, orientation="horizontal", widths=0.6,
                            patch_artist=True, showfliers=False)
        except (TypeError, AttributeError):
            bp = ax.boxplot(data, vert=False, widths=0.6, patch_artist=True, showfliers=False)
        for patch, col in zip(bp["boxes"], colors):
            patch.set_facecolor(col)
            patch.set_alpha(0.35)
            patch.set_edgecolor(col)
        for element in ("medians", "whiskers", "caps"):
            for art in bp[element]:
                art.set_color("0.3")
        for i, (vals, col) in enumerate(zip(data, colors), start=1):
            jitter = np.random.default_rng(0).normal(0, 0.055, len(vals))
            ax.plot(vals, i + jitter, ".", color=col, markersize=3.2, alpha=0.75)
        ax.set_yticklabels(labels)
    ax.set_xlabel("Macro F1 per cross-validation fold")
    ax.set_title(f"B  Fold-level dispersion ({cfg.repeats} x {cfg.folds}-fold)",
                 loc="left", fontweight="bold")
    ax.grid(axis="y", visible=False)

    fig.suptitle("Figure 3 | Discriminative performance across models",
                 y=1.04, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig03_performance", cfg.dpi)


def fig_confusion(y: np.ndarray, prob: np.ndarray, classes: List[str], cfg: Config, outdir: Path) -> None:
    """Figure 4 - confusion matrices (counts and row-normalised recall)."""
    pred = prob.argmax(1)
    cm = confusion_matrix(y, pred, labels=list(range(len(classes))))
    cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
    short = [c.replace(" ", "\n") for c in classes]
    for ax, mat, title, fmtf, cmap in [
        (axes[0], cm, "A  Counts", lambda v: f"{int(v):d}", "Blues"),
        (axes[1], cmn, "B  Row-normalised (sensitivity on diagonal)", lambda v: f"{v:.2f}", "Blues"),
    ]:
        im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=mat.max() if mat.max() > 0 else 1)
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, fmtf(mat[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if mat[i, j] > 0.6 * mat.max() else "0.15")
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(short, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(short, fontsize=7)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Reference (ground truth)")
        ax.set_title(title, loc="left", fontweight="bold")
        ax.grid(False)
        fig.colorbar(im, ax=ax, fraction=0.046, shrink=0.85)
    fig.suptitle(f"Figure 4 | Confusion structure - {cfg.reference_model} (pooled out-of-fold)",
                 y=1.03, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig04_confusion", cfg.dpi)


def fig_roc_pr(y: np.ndarray, prob: np.ndarray, classes: List[str], cfg: Config, outdir: Path) -> None:
    """Figure 5 - one-vs-rest ROC and precision-recall curves."""
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0))
    for c, name in enumerate(classes):
        yb = (y == c).astype(int)
        if yb.min() == yb.max():
            continue
        col = PALETTE[c % len(PALETTE)]
        fpr, tpr, _ = roc_curve(yb, prob[:, c])
        auc = roc_auc_score(yb, prob[:, c])
        axes[0].plot(fpr, tpr, color=col, label=f"{name} ({auc:.3f})")
        prec, rec, _ = precision_recall_curve(yb, prob[:, c])
        ap = average_precision_score(yb, prob[:, c])
        axes[1].plot(rec, prec, color=col, label=f"{name} ({ap:.3f})")
        axes[1].axhline(yb.mean(), color=col, ls=":", lw=0.7, alpha=0.5)
    axes[0].plot([0, 1], [0, 1], color="0.6", ls="--", lw=0.9)
    axes[0].set_xlabel("1 - specificity")
    axes[0].set_ylabel("Sensitivity")
    axes[0].set_title("A  ROC (AUROC)", loc="left", fontweight="bold")
    axes[0].legend(loc="lower right", fontsize=7)
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("B  Precision-recall (average precision)", loc="left", fontweight="bold")
    axes[1].legend(loc="lower left", fontsize=7)
    for ax in axes:
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal")
    fig.suptitle(f"Figure 5 | Class-wise discrimination - {cfg.reference_model}",
                 y=1.03, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig05_roc_pr", cfg.dpi)


def fig_calibration(
    y: np.ndarray, prob_uncal: np.ndarray, prob_cal: np.ndarray, cfg: Config, outdir: Path
) -> None:
    """Figure 6 - reliability diagram before and after temperature scaling."""
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0))
    bins = np.linspace(0, 1, 11)
    for ax, prob, title in [
        (axes[0], prob_uncal, "A  Uncalibrated"),
        (axes[1], prob_cal, "B  After temperature scaling"),
    ]:
        conf = prob.max(1)
        correct = (prob.argmax(1) == y).astype(float)
        xs, ys, ws = [], [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            sel = (conf > lo) & (conf <= hi)
            if sel.sum() >= 5:
                xs.append(conf[sel].mean())
                ys.append(correct[sel].mean())
                ws.append(sel.sum())
        ax.plot([0, 1], [0, 1], color="0.6", ls="--", lw=0.9, label="perfect calibration")
        if xs:
            sizes = 18 + 130 * np.asarray(ws) / max(ws)
            ax.plot(xs, ys, "-", color=PALETTE[0], lw=1.4)
            ax.scatter(xs, ys, s=sizes, color=PALETTE[0], alpha=0.8, edgecolors="white", zorder=3)
        ece = expected_calibration_error(y, prob)
        brier = multiclass_brier(y, prob, prob.shape[1])
        ax.text(0.04, 0.93, f"ECE = {ece:.3f}\nBrier = {brier:.3f}", fontsize=8,
                va="top", bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.8", lw=0.6))
        ax.set_xlabel("Mean predicted confidence")
        ax.set_ylabel("Observed accuracy")
        ax.set_title(title, loc="left", fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal")
    axes[0].legend(loc="lower right", fontsize=7)
    fig.suptitle(f"Figure 6 | Probability calibration - {cfg.reference_model} "
                 "(marker area proportional to bin count)",
                 y=1.03, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig06_calibration", cfg.dpi)


def fig_cd_diagram(fried: Dict[str, Any], cfg: Config, outdir: Path) -> None:
    """Figure 7 - Demsar critical-difference diagram of average ranks."""
    if not fried.get("applicable"):
        return
    ranks = fried["average_ranks"]
    cd = fried["critical_difference"]
    models = sorted(ranks, key=lambda m: ranks[m])
    vals = [ranks[m] for m in models]
    lo, hi = 1.0, float(len(models))

    fig, ax = plt.subplots(figsize=(8.4, 0.42 * len(models) + 2.0))
    ax.set_xlim(hi + 0.35, lo - 0.35)  # rank 1 (best) on the right-hand side
    ax.set_ylim(-0.6, len(models) * 0.55 + 0.9)
    ax.axis("off")
    ax.hlines(len(models) * 0.55 + 0.45, lo, hi, color="0.2", lw=1.1)
    for t in np.arange(math.floor(lo), math.ceil(hi) + 1):
        ax.vlines(t, len(models) * 0.55 + 0.45, len(models) * 0.55 + 0.58, color="0.2", lw=1.0)
        ax.text(t, len(models) * 0.55 + 0.72, f"{int(t)}", ha="center", fontsize=8)

    for i, (m, v) in enumerate(zip(models, vals)):
        yy = len(models) * 0.55 + 0.45 - (i + 1) * 0.55
        col = model_color(m)
        ax.plot([v, v], [yy, len(models) * 0.55 + 0.45], color=col, lw=1.1)
        ax.plot([v, hi + 0.2], [yy, yy], color=col, lw=1.1)
        ax.text(hi + 0.25, yy, f"{m}  ({v:.2f})", va="center", ha="left", fontsize=8,
                fontweight="bold" if m == cfg.reference_model else "normal", color=col)

    # Cliques of models that are NOT significantly different.
    ybar = len(models) * 0.55 + 0.30
    drawn = 0
    for i in range(len(models)):
        j = i
        while j + 1 < len(models) and (vals[j + 1] - vals[i]) <= cd:
            j += 1
        if j > i:
            ax.hlines(ybar - drawn * 0.10, vals[i] - 0.02, vals[j] + 0.02, color="0.15", lw=3.0)
            drawn += 1
    ax.plot([lo, lo + cd], [-0.25, -0.25], color="0.15", lw=1.6)
    ax.vlines([lo, lo + cd], -0.32, -0.18, color="0.15", lw=1.6)
    ax.text(lo + cd / 2, -0.45, f"CD = {cd:.2f}", ha="center", fontsize=8)
    ax.set_title(
        "Figure 7 | Critical-difference diagram of average ranks "
        f"(Friedman p = {fried['p_value']:.2e}, N = {fried['n_folds']} folds)\n"
        "Models joined by a bar are not significantly different (Nemenyi, alpha = 0.05)",
        fontsize=9.5, fontweight="bold", loc="left", pad=14,
    )
    fig.tight_layout()
    savefig(fig, outdir, "fig07_critical_difference", cfg.dpi)


def fig_dca(y: np.ndarray, prob: np.ndarray, classes: List[str], cfg: Config, outdir: Path) -> None:
    """Figure 8 - decision curve analysis for each disease vs the rest."""
    disease = [i for i, c in enumerate(classes) if "healthy" not in c.lower()]
    if not disease:
        return
    ncols = min(len(disease), 4)
    nrows = int(math.ceil(len(disease) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 2.9 * nrows), squeeze=False)
    for ax_i, c in enumerate(disease):
        ax = axes[ax_i // ncols][ax_i % ncols]
        dc = decision_curve(y, prob, c)
        ax.plot(dc["thresholds"], dc["model"], color=PALETTE[0], label=cfg.reference_model)
        ax.plot(dc["thresholds"], dc["treat_all"], color="0.45", ls="--", lw=1.0, label="Treat all")
        ax.plot(dc["thresholds"], dc["treat_none"], color="0.7", ls=":", lw=1.0, label="Treat none")
        ymin = float(np.nanmin(dc["model"]))
        ax.set_ylim(min(-0.02, ymin), max(0.02, float(np.nanmax(dc["model"])) * 1.2))
        ax.set_xlabel("Threshold probability")
        ax.set_ylabel("Net benefit")
        ax.set_title(classes[c], loc="left", fontsize=8.5, fontweight="bold")
        if ax_i == 0:
            ax.legend(fontsize=7, loc="upper right")
    for k in range(len(disease), nrows * ncols):
        axes[k // ncols][k % ncols].axis("off")
    fig.suptitle("Figure 8 | Decision curve analysis (one-vs-rest clinical utility)",
                 y=1.02, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig08_decision_curves", cfg.dpi)


def fig_fairness(fair: pd.DataFrame, cfg: Config, outdir: Path) -> None:
    """Figure 9 - subgroup performance across pigmentation strata."""
    if fair is None or fair.empty or "Balanced accuracy" not in fair:
        return
    sub = fair[fair["Balanced accuracy"].notna()]
    if sub.empty:
        return
    metrics = ["Balanced accuracy", "Macro F1", "Sensitivity", "Specificity"]
    metrics = [m for m in metrics if m in sub]
    x = np.arange(len(sub))
    width = 0.8 / max(len(metrics), 1)
    fig, ax = plt.subplots(figsize=(1.9 * len(sub) + 3.6, 3.6))
    for i, m in enumerate(metrics):
        ax.bar(x + i * width - 0.4 + width / 2, sub[m].to_numpy(), width * 0.92,
               label=m, color=PALETTE[i % len(PALETTE)], alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}\n(n = {n})" for s, n in zip(sub["Stratum"], sub["n"])], fontsize=8)
    ax.set_ylabel("Performance")
    ax.set_ylim(0, 1.05)
    ax.legend(ncol=len(metrics), fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, 1.14))
    ax.grid(axis="x", visible=False)
    ax.set_title(
        f"Figure 9 | Fairness audit across skin-pigmentation strata - {cfg.reference_model}",
        loc="left", fontsize=10, fontweight="bold", pad=26,
    )
    fig.tight_layout()
    savefig(fig, outdir, "fig09_fairness", cfg.dpi)


def fig_embeddings(
    X: np.ndarray, Z_learned: Optional[np.ndarray], y: np.ndarray,
    classes: List[str], cfg: Config, outdir: Path,
) -> None:
    """Figure 10 - manifold before and after graph-based representation learning."""
    panels = [("A  Frozen CNN embedding (input to the graph)", X)]
    if Z_learned is not None:
        panels.append((f"B  {cfg.reference_model} node embedding", Z_learned))
    fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.2), squeeze=False)
    method = "t-SNE"
    for ax, (title, mat) in zip(axes[0], panels):
        Z, method = embed_2d(np.asarray(mat, dtype=np.float64), cfg.seed)
        for c, name in enumerate(classes):
            sel = y == c
            ax.scatter(Z[sel, 0], Z[sel, 1], s=11, alpha=0.75, label=name,
                       color=PALETTE[c % len(PALETTE)], edgecolors="none")
        ax.set_title(title, loc="left", fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    axes[0][0].legend(fontsize=7.5, loc="best", markerscale=1.4)
    fig.suptitle(f"Figure 10 | {method} projection of the representation space",
                 y=1.02, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig10_embeddings", cfg.dpi)


def fig_training(histories: Dict[str, Dict[str, List[float]]], cfg: Config, outdir: Path) -> None:
    """Figure 11 - representative optimisation traces."""
    if not histories:
        return
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4))
    for name, h in histories.items():
        col = model_color(name)
        if h["train_loss"]:
            axes[0].plot(h["train_loss"], color=col, lw=1.2, label=name)
        if h["val_bacc"]:
            axes[1].plot(h["val_bacc"], color=col, lw=1.2, label=name)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Training loss (focal)")
    axes[0].set_title("A  Optimisation", loc="left", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Validation balanced accuracy")
    axes[1].set_title("B  Model selection criterion", loc="left", fontweight="bold")
    axes[1].legend(fontsize=7.5, loc="lower right")
    fig.suptitle("Figure 11 | Representative training dynamics (first fold)",
                 y=1.04, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig11_training", cfg.dpi)


def fig_ablation(abl: pd.DataFrame, cfg: Config, outdir: Path) -> None:
    """Figure 12 - sensitivity of the proposed model to its design choices."""
    if abl is None or abl.empty:
        return
    groups = list(dict.fromkeys(abl["Ablation"].tolist()))
    ncols = min(3, len(groups))
    nrows = int(math.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 2.9 * nrows), squeeze=False)
    for i, g in enumerate(groups):
        ax = axes[i // ncols][i % ncols]
        sub = abl[abl["Ablation"] == g]
        labels = [str(s) for s in sub["Setting"]]
        vals = sub["Balanced accuracy"].to_numpy(dtype=float)
        errs = sub["SD"].to_numpy(dtype=float)
        cols = [PALETTE[0]] * len(vals)
        if len(vals):
            cols[int(np.nanargmax(vals))] = PALETTE[1]
        ax.bar(range(len(vals)), vals, yerr=errs, capsize=2.5, color=cols, alpha=0.9,
               error_kw={"lw": 0.8, "ecolor": "0.4"})
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, fontsize=7, rotation=20, ha="right")
        ax.set_ylabel("Balanced accuracy")
        ax.set_ylim(0, 1.02)
        ax.set_title(g, loc="left", fontsize=8.5, fontweight="bold")
        ax.grid(axis="x", visible=False)
    for k in range(len(groups), nrows * ncols):
        axes[k // ncols][k % ncols].axis("off")
    fig.suptitle(f"Figure 12 | Ablation studies - {cfg.reference_model}",
                 y=1.02, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig12_ablation", cfg.dpi)


def fig_cam(cams: List[Dict[str, Any]], cfg: Config, outdir: Path) -> None:
    """Figure 13 - class activation maps over representative cases."""
    if not cams:
        return
    n = len(cams)
    ncols = min(6, n)
    nrows = int(math.ceil(n / ncols)) * 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.85 * ncols, 1.95 * nrows), squeeze=False)
    for ax_row in axes:
        for ax in ax_row:
            ax.axis("off")
    for i, item in enumerate(cams):
        block = (i // ncols) * 2
        col = i % ncols
        axes[block][col].imshow(np.clip(item["image"], 0, 1))
        axes[block][col].set_title(item["label"], fontsize=7)
        axes[block][col].axis("off")
        axes[block + 1][col].imshow(np.clip(item["image"], 0, 1))
        axes[block + 1][col].imshow(item["cam"], cmap="jet", alpha=0.45)
        axes[block + 1][col].axis("off")
    fig.suptitle("Figure 13 | Class activation mapping (top: clinical image; bottom: model evidence)",
                 y=1.005, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "fig13_activation_maps", cfg.dpi)

### 1.15 MANUSCRIPT-READY REPORTING

In [ ]:
# =========================================================================== #
# SECTION 15 - MANUSCRIPT-READY REPORTING                                     #
# =========================================================================== #
def export_table(df: pd.DataFrame, outdir: Path, name: str, caption: str, float_fmt: str = "%.3f") -> None:
    """Write a table as CSV and as a booktabs LaTeX fragment."""
    tdir = outdir / "tables"
    tdir.mkdir(parents=True, exist_ok=True)
    df.to_csv(tdir / f"{name}.csv", index=False)
    try:
        body = df.to_latex(index=False, float_format=lambda v: float_fmt % v,
                           escape=True, longtable=False, na_rep="--")
    except Exception:
        body = df.to_string(index=False)
    tex = (
        "\\begin{table}[htbp]\n\\centering\n\\small\n"
        f"\\caption{{{caption}}}\n\\label{{tab:{name}}}\n{body}\\end{{table}}\n"
    )
    (tdir / f"{name}.tex").write_text(tex, encoding="utf-8")
    LOGGER.info("  table saved: tables/%s.{csv,tex}", name)


def cohort_table(df: pd.DataFrame, classes: List[str]) -> pd.DataFrame:
    """Table 1 - baseline characteristics of the analysis cohort."""
    rows = []
    total = len(df)
    for c in classes:
        sub = df[df["label"] == c]
        row = {
            "Class": c,
            "n": len(sub),
            "% of cohort": 100.0 * len(sub) / max(total, 1),
            "Median width (px)": float(sub["width"].median()) if len(sub) else np.nan,
            "Median height (px)": float(sub["height"].median()) if len(sub) else np.nan,
        }
        if "ita" in df.columns:
            row["Median ITA (deg)"] = float(sub["ita"].median()) if sub["ita"].notna().any() else np.nan
            for band in sorted(df["ita_band"].unique()):
                row[band] = int((sub["ita_band"] == band).sum())
        rows.append(row)
    overall = {
        "Class": "All", "n": total, "% of cohort": 100.0,
        "Median width (px)": float(df["width"].median()),
        "Median height (px)": float(df["height"].median()),
    }
    if "ita" in df.columns:
        overall["Median ITA (deg)"] = float(df["ita"].median()) if df["ita"].notna().any() else np.nan
        for band in sorted(df["ita_band"].unique()):
            overall[band] = int((df["ita_band"] == band).sum())
    rows.append(overall)
    return pd.DataFrame(rows)


def headline_table(
    table: pd.DataFrame, primary: Dict[str, Dict[str, Tuple[float, float, float]]]
) -> pd.DataFrame:
    """
    Table 2 - primary results as 'fold mean (Nadeau-Bengio corrected 95% CI)'.
    """
    rows = []
    for _, r in table.iterrows():
        m = r["Model"]
        row: Dict[str, Any] = {"Model": m}
        for key, label in METRIC_LABELS.items():
            pe, lo, hi = primary[m][key]
            pct = key not in {"brier", "ece", "mcc", "kappa"}
            row[label] = fmt_ci(pe, lo, hi, pct=pct)
        rows.append(row)
    return pd.DataFrame(rows)


def tripod_ai_checklist(cfg: Config, flow: Dict[str, Any], res: ExperimentResult) -> pd.DataFrame:
    """
    TRIPOD+AI-aligned reporting table. Each row states where in this analysis
    the corresponding item is addressed - designed to be pasted into the
    supplementary material of the manuscript.
    """
    items = [
        ("Title & abstract", "Study identified as diagnostic model development using AI",
         "Manuscript title; module docstring."),
        ("Background", "Clinical need for rare-dermatosis triage in skin of colour",
         "Introduction; cohort drawn from Bangladesh, Ghana and Kenya."),
        ("Source of data", "Retrospective, consented, anonymised clinical photography",
         "Rare_Skin_Disease_Dataset (Kaggle, DERM-Net Dual)."),
        ("Participants", f"{flow.get('analysed', 0)} images across {len(res.classes)} classes",
         "Table 1; Figure 1."),
        ("Outcome", "Five-class dermatological diagnosis; reference standard = curated label",
         "Section 3 (data QC)."),
        ("Predictors", f"Frozen {cfg.backbone} embedding, PCA to {cfg.pca_dim} dimensions",
         "Section 4."),
        ("Sample size", "No a priori calculation; all available images analysed",
         "Limitation - stated explicitly in the report."),
        ("Missing data", "Unreadable and near-duplicate images removed before splitting",
         "Flow counts in results.json."),
        ("Analytical methods", f"{cfg.repeats} x {cfg.folds}-fold stratified CV, {cfg.graph_mode} graph protocol",
         "Section 12."),
        ("Class imbalance", "Inverse-frequency class weights + focal loss; balanced accuracy primary",
         "Section 8; Table 2."),
        ("Model performance", "Discrimination, calibration and clinical utility all reported",
         "Figures 3-6, 8; Tables 2-3."),
        ("Model updating", "Temperature scaling fitted on the validation split only",
         "Section 8; Figure 6."),
        ("Uncertainty", "Bootstrap 95% CIs; fold-level SD; Friedman/Nemenyi; Wilcoxon; DeLong",
         "Tables 2, 4; Figure 7."),
        ("Fairness", "Subgroup analysis by Individual Typology Angle strata",
         "Table 5; Figure 9."),
        ("Explainability", "Class activation maps and graph-neighbourhood attribution",
         "Figures 10, 13."),
        ("Limitations", "Single retrospective corpus; no external or prospective validation",
         "Report - Limitations."),
        ("Data & code availability", "Dataset public on Kaggle; analysis code released with fixed seeds",
         "This file; requirements.txt."),
    ]
    return pd.DataFrame(items, columns=["TRIPOD+AI item", "How it was addressed", "Where reported"])


def write_report(
    cfg: Config, flow: Dict[str, Any], res: ExperimentResult,
    table: pd.DataFrame,
    primary: Dict[str, Dict[str, Tuple[float, float, float]]],
    boot: Dict[str, Dict[str, Tuple[float, float, float]]],
    per_cls: pd.DataFrame, fried: Dict[str, Any], wilcox: pd.DataFrame,
    delong: Optional[pd.DataFrame], fair: Optional[pd.DataFrame],
    fair_gaps: Dict[str, float], ita_check: Dict[str, Any],
    shortcut: Dict[str, Any], abl: Optional[pd.DataFrame],
    feature_desc: str, y_ref_true: np.ndarray, outdir: Path,
) -> None:
    """Emit REPORT.md - a drafted Results section with every number filled in."""
    ref = cfg.reference_model
    L: List[str] = []
    A = L.append

    A("# DERM-GNN: Graph Neural Network Analysis of a Rare Skin Disease Cohort\n")
    A(f"_Generated {time.strftime('%Y-%m-%d %H:%M')} - seed {cfg.seed} - device {cfg.device}_\n")

    A("## 1. Cohort and data flow\n")
    A(f"- Image files screened: **{flow.get('files_found', 0)}**")
    A(f"- Excluded, unmapped directory: {flow.get('files_unmapped', 0)}")
    A(f"- Excluded, unreadable/degenerate: {flow.get('removed_unreadable', 0)}")
    A(f"- Excluded, perceptual near-duplicates: {flow.get('removed_near_duplicates', 0)}")
    A(f"- Excluded, classes below the minimum size: {flow.get('removed_small_classes', [])}")
    A(f"- **Analysis cohort: {flow.get('analysed', 0)} images, {len(res.classes)} classes**")
    A(f"- Class imbalance ratio (max/min): {flow.get('imbalance_ratio', float('nan'))}\n")
    A("| Class | n |")
    A("|---|---|")
    for c, n in flow.get("class_counts", {}).items():
        A(f"| {c} | {n} |")
    A("")

    A("## 2. Representation and graph\n")
    A(f"- Encoder: {feature_desc}, test-time flip augmentation = {cfg.feature_tta}")
    A(f"- Dimensionality reduction: PCA to {cfg.pca_dim} whitened components")
    A(f"- Graph: {cfg.graph_metric} k-NN, k = {cfg.knn_k}, protocol = **{cfg.graph_mode}**")
    gs = res.graph_stats or {}
    A(f"- Edges = {gs.get('num_edges', 'n/a')}, mean in-degree = {gs.get('mean_degree', float('nan')):.2f}, "
      f"isolated nodes = {gs.get('isolated_nodes', 'n/a')}")
    homo = gs.get("homophily", float("nan"))
    A(f"- **Edge homophily = {homo:.3f}** (chance = {1.0 / max(len(res.classes), 1):.3f}); "
      "values well above chance confirm that visual similarity carries diagnostic signal, "
      "which is the premise of the graph formulation.\n")

    A("## 3. Primary results\n")
    A("Estimates are the mean across cross-validation folds with a Nadeau-Bengio corrected "
      "95% confidence interval. The correction is applied because cross-validation folds share "
      "training data, which makes the naive standard error of fold scores optimistic. A "
      "percentile bootstrap over pooled out-of-fold predictions is reported alongside for "
      "comparability with the literature; it is narrower by construction because it conditions "
      "on a single partition. The corrected interval is normal-theory and is clipped to each "
      "metric's natural range, so a bound sitting exactly at 0 or 1 reflects that clipping "
      "rather than a precise estimate.\n")
    A("Pooled out-of-fold predictions come from a **single repeat**, so each image is scored "
      "exactly once by a model that never saw it. Averaging predictions across repeats would "
      "form an implicit ensemble and inflate every pooled metric; that averaged version is "
      "retained in `results.json` for reference but is not reported.\n")
    if not table.empty:
        best = table.iloc[0]["Model"]
        pe, lo, hi = primary[ref][PRIMARY_METRIC]
        f1 = primary[ref]["macro_f1"]
        auc = primary[ref]["auroc_macro"]
        A(f"Across {cfg.repeats} x {cfg.folds}-fold cross-validation, **{ref}** achieved a balanced "
          f"accuracy of **{fmt_ci(pe, lo, hi)}%**, macro F1 {fmt_ci(*f1)}% and macro AUROC "
          f"{fmt_ci(*auc)}%. The best-ranked model on the primary endpoint was **{best}**.\n")
        A(headline_table(table, primary).to_markdown(index=False))
        A("")
        bpe, blo, bhi = boot[ref][PRIMARY_METRIC]
        A(f"_Secondary estimate for {ref}: pooled single-repeat out-of-fold balanced accuracy "
          f"{fmt_ci(bpe, blo, bhi)}% (percentile bootstrap, {cfg.bootstrap} resamples)._\n")

    A(f"### 3.1 Class-wise performance ({ref})\n")
    if per_cls is not None and not per_cls.empty:
        A(per_cls.round(3).to_markdown(index=False))
        A("")

    A("## 4. Statistical comparison\n")
    if fried.get("applicable"):
        A(f"Friedman omnibus test across {len(fried['models'])} models on {fried['n_folds']} folds: "
          f"chi-square = {fried['statistic']:.2f}, **p = {fried['p_value']:.3e}**. "
          f"Nemenyi critical difference = {fried['critical_difference']:.2f} rank units.\n")
        A("| Model | Average rank |")
        A("|---|---|")
        for m, r in sorted(fried["average_ranks"].items(), key=lambda kv: kv[1]):
            A(f"| {m} | {r:.2f} |")
        A("")
    else:
        A(f"Friedman test not applicable ({fried.get('reason', 'unknown')}).\n")

    if wilcox is not None and not wilcox.empty:
        A(f"### 4.1 Paired Wilcoxon signed-rank vs {ref} (fold-level {METRIC_LABELS[PRIMARY_METRIC]})\n")
        A(wilcox.round(4).to_markdown(index=False))
        A("")
        if wilcox["p"].isna().all():
            A(f"> Only {cfg.repeats * cfg.folds} cross-validation splits were run, below the "
              f"{WILCOXON_MIN_FOLDS} paired observations the two-sided signed-rank test needs to "
              "reach any conventional significance level. The differences above are descriptive "
              "only; increase `--repeats`/`--folds` before drawing inferential conclusions.\n")

    if delong is not None and not delong.empty:
        A("### 4.2 DeLong test on pooled out-of-fold AUCs\n")
        A(delong.round(4).to_markdown(index=False))
        A("")

    A("## 5. Calibration\n")
    t = res.temperatures.get(ref, 1.0)
    ece_pre = expected_calibration_error(y_ref_true, res.oof_prob_uncal[ref]) if ref in res.oof_prob_uncal else float("nan")
    ece_post = expected_calibration_error(y_ref_true, res.oof_prob[ref]) if ref in res.oof_prob else float("nan")
    A(f"Mean fitted temperature for {ref} was T = {t:.3f} "
      f"({'sharpening' if t < 1 else 'softening'} the raw logits). "
      f"Expected calibration error fell from {ece_pre:.3f} to {ece_post:.3f}. "
      "Reliability diagrams before and after scaling are shown in Figure 6.\n")

    A("## 6. Fairness across skin-pigmentation strata\n")
    if ita_check.get("available"):
        A(f"**Proxy-validity check.** ITA requires colour-calibrated photography; these images "
          f"carry no colour reference and were acquired on unknown devices, so ITA may track "
          f"lesion chromaticity, white balance or exposure rather than constitutive pigmentation. "
          f"Testing ITA across diagnostic classes: Kruskal-Wallis H = {ita_check['kruskal_H']:.2f}, "
          f"p = {ita_check['p_value']:.3g}, epsilon-squared = {ita_check['epsilon_squared']:.3f}.\n")
        A(f"> {ita_check['interpretation']}\n")
        if ita_check.get("confounded_by_diagnosis"):
            A("> **The subgroup analysis below is therefore exploratory and must not be "
              "described as a validated skin-tone fairness analysis.** Establishing fairness "
              "across pigmentation requires either colour-calibrated acquisition or recorded "
              "Fitzpatrick phototype.\n")
    if fair is not None and not fair.empty:
        A(fair.round(3).to_markdown(index=False))
        A("")
        if fair_gaps:
            worst = max(fair_gaps.items(), key=lambda kv: kv[1])
            A(f"Largest observed between-stratum gap: **{worst[0].replace('_', ' ')} = {worst[1]:.3f}**. "
              "Gaps should be interpreted alongside the stratum sample sizes, which are small "
              "for the lighter-pigmentation groups in this cohort.\n")
    else:
        A("Fairness audit not run.\n")

    A("## 7. Shortcut-learning probe\n")
    if shortcut.get("available"):
        A(f"The central {shortcut['mask_fraction']:.0%} of every image - where the lesion almost "
          f"always sits - was masked out and {ref} was re-cross-validated on the remaining "
          f"background, framing and marginal skin.\n")
        A(f"- Balanced accuracy on background alone: **{shortcut['balanced_accuracy']:.3f}** "
          f"(SD {shortcut['sd']:.3f}) against a chance level of {shortcut['chance']:.3f}")
        A(f"- One-sample t-test vs chance: p = {shortcut['p_vs_chance']:.3g}")
        A(f"- Shortcut signal: **{shortcut['severity']}**\n")
        A(f"> {shortcut['interpretation']}\n")
        if shortcut.get("above_chance"):
            A("> This is a material threat to validity. The healthy-skin images plausibly "
              "originate from a different acquisition setting than the clinical disease "
              "photographs, and a classifier can exploit that difference without learning "
              "anything dermatological. The headline result must be interpreted in this "
              "light, and re-curation with matched acquisition is the appropriate remedy.\n")
    else:
        A("Not run.\n")

    if abl is not None and not abl.empty:
        A("## 8. Ablation studies\n")
        A(abl.round(3).to_markdown(index=False))
        A("")

    A("## 9. Interpretation\n")
    graph_models = [m for m in cfg.models if m in GRAPH_MODELS and m in primary]
    if graph_models and "MLP" in primary:
        gain = primary[ref][PRIMARY_METRIC][0] - primary["MLP"][PRIMARY_METRIC][0]
        A(f"- The graph contributes **{gain * 100:+.1f} percentage points** of balanced accuracy over an "
          "identically optimised graph-free MLP on the same features, isolating the benefit of "
          "message passing from that of the imaging encoder.")
    A(f"- The encoder is frozen and label-agnostic, and with `--feature-fit {cfg.feature_fit}` the "
      "standardisation, PCA projection and similarity graph are "
      + ("refit on each fold's training rows alone, so the held-out partition influences neither "
         "the representation nor the graph." if cfg.feature_fit == "foldwise" else
         "fit once on the whole cohort, which lets the held-out partition's distribution inform "
         "the projection - a reviewer may reasonably object; prefer `foldwise`."))
    A("- The inductive graph protocol additionally prevents test images from informing one another.")
    A("- Near-duplicate removal before splitting is essential in scraped clinical corpora; without it, "
      "performance estimates are optimistically biased.\n")

    A("## 10. Limitations\n")
    A("1. **Patient-level independence cannot be verified.** The corpus carries no patient "
      "identifiers, so several images of the same individual may fall on both sides of a split. "
      "Perceptual near-duplicate removal does not solve this: different photographs of the same "
      "patient are not near-duplicates. Performance may therefore be optimistic by an unknown "
      "margin. This is the single most serious limitation of the study.")
    A("2. **No adjudicated reference standard.** Labels derive from the dataset curation, without "
      "histopathological confirmation, specialist re-review or any inter-rater agreement estimate.")
    A("3. Single retrospective corpus of modest size; no external, temporal, prospective or "
      "multi-reader validation, and no comparison against clinician performance.")
    if shortcut.get("available") and shortcut.get("above_chance"):
        A(f"4. **Acquisition shortcut detected.** Background alone reaches "
          f"{shortcut['balanced_accuracy']:.1%} balanced accuracy against "
          f"{shortcut['chance']:.1%} chance, so part of the discrimination reflects "
          f"acquisition context rather than skin pathology.")
    else:
        A("4. Residual confounding by acquisition setting cannot be excluded, although the "
          "background-only probe did not detect it.")
    A("5. Pigmentation strata rest on an image-derived ITA proxy computed from uncalibrated "
      "photographs, not recorded Fitzpatrick phototype; several strata are small. Fairness "
      "findings are exploratory.")
    A("6. Class prevalence in the corpus does not reflect population prevalence, so predictive "
      "values would differ in deployment.")
    A("7. Hyperparameters were fixed a priori rather than tuned in a nested loop; the ablations "
      "are reported for sensitivity, not used for selection.")
    A("8. No sample-size justification, and the protocol was not pre-registered.")
    A("9. The model is a triage aid, not a diagnostic device. Prospective evaluation is required "
      "before any clinical use.\n")

    A("## 11. Reproducibility\n")
    A("```bash")
    A(f"python dermgnn_analysis.py --data-root <path> --backbone {cfg.backbone} \\")
    A(f"    --knn-k {cfg.knn_k} --graph-mode {cfg.graph_mode} --repeats {cfg.repeats} "
      f"--folds {cfg.folds} --seed {cfg.seed}")
    A("```\n")
    A("All randomness is seeded; `config.json`, `results.json`, `tables/` and `figures/` in this "
      "directory constitute the complete analytic record.\n")

    (outdir / "REPORT.md").write_text("\n".join(L), encoding="utf-8")
    LOGGER.info("Report written: %s", outdir / "REPORT.md")

### 1.16 COMMAND-LINE INTERFACE

In [ ]:
# =========================================================================== #
# SECTION 16 - COMMAND-LINE INTERFACE                                         #
# =========================================================================== #
def parse_args(argv: Optional[Sequence[str]] = None) -> Config:
    p = argparse.ArgumentParser(
        prog="dermgnn_analysis.py",
        description="GNN analysis pipeline for the Rare Skin Disease Dataset.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    d = Config()

    g = p.add_argument_group("data")
    g.add_argument("--data-root", type=str, default=d.data_root,
                   help="Folder containing the extracted dataset, or archive.zip")
    g.add_argument("--manifest", type=str, default=d.manifest,
                   help="CSV with columns path,label[,group]; bypasses folder discovery")
    g.add_argument("--group-column", type=str, default=d.group_column,
                   help="manifest column holding the lesion/patient id to keep within a fold")
    g.add_argument("--skin-tone-column", type=str, default=d.skin_tone_column,
                   help="manifest column with a recorded skin-tone label; used for the "
                        "fairness audit instead of the image-derived ITA proxy")
    g.add_argument("--max-per-class", type=int, default=d.max_per_class,
                   help="cap images per class (0 = no cap); samples whole groups")
    g.add_argument("--download", action="store_true", help="Fetch the dataset via kagglehub")
    g.add_argument("--synthetic", action="store_true",
                   help="Run on a generated stand-in corpus (pipeline smoke test)")
    g.add_argument("--synthetic-n", type=int, default=d.synthetic_n)
    g.add_argument("--image-size", type=int, default=d.image_size)
    g.add_argument("--center-crop-frac", type=float, default=d.center_crop_frac,
                   help="crop to this fraction of the shorter side before resizing; "
                        "values near 0.7 strip most background")
    g.add_argument("--no-dedupe", dest="dedupe", action="store_false", default=d.dedupe)
    g.add_argument("--dedupe-hamming", type=int, default=d.dedupe_hamming)

    g = p.add_argument_group("representation")
    g.add_argument("--backbone", type=str, default=d.backbone,
                   help="one backbone, or a comma-separated list to fuse their embeddings "
                        f"(choices: {', '.join(BACKBONE_REGISTRY)})")
    g.add_argument("--finetune", action="store_true",
                   help="fine-tune the encoder inside every fold (slow, but the largest "
                        "legitimate accuracy gain available)")
    g.add_argument("--finetune-epochs", type=int, default=d.finetune_epochs)
    g.add_argument("--finetune-patience", type=int, default=d.finetune_patience)
    g.add_argument("--finetune-lr", type=float, default=d.finetune_lr)
    g.add_argument("--finetune-batch", type=int, default=d.finetune_batch)
    g.add_argument("--no-tta", dest="feature_tta", action="store_false", default=d.feature_tta)
    g.add_argument("--pca-dim", type=int, default=d.pca_dim, help="0 disables PCA")
    g.add_argument("--feature-fit", type=str, default=d.feature_fit,
                   choices=["foldwise", "global"],
                   help="refit scaler/PCA/graph per fold (foldwise) or once on all data (global)")
    g.add_argument("--batch-size", type=int, default=d.batch_size)
    g.add_argument("--num-workers", type=int, default=d.num_workers)

    g = p.add_argument_group("graph")
    g.add_argument("--knn-k", type=int, default=d.knn_k)
    g.add_argument("--graph-mode", type=str, default=d.graph_mode,
                   choices=["inductive", "transductive"])
    g.add_argument("--graph-metric", type=str, default=d.graph_metric,
                   choices=["cosine", "euclidean"])
    g.add_argument("--mutual-knn", action="store_true", default=d.mutual_knn)
    g.add_argument("--edge-sim-threshold", type=float, default=d.edge_sim_threshold)

    g = p.add_argument_group("model / optimisation")
    g.add_argument("--hidden-dim", type=int, default=d.hidden_dim)
    g.add_argument("--num-layers", type=int, default=d.num_layers)
    g.add_argument("--heads", type=int, default=d.heads)
    g.add_argument("--dropout", type=float, default=d.dropout)
    g.add_argument("--drop-edge", type=float, default=d.drop_edge)
    g.add_argument("--lr", type=float, default=d.lr)
    g.add_argument("--weight-decay", type=float, default=d.weight_decay)
    g.add_argument("--epochs", type=int, default=d.epochs)
    g.add_argument("--patience", type=int, default=d.patience)
    g.add_argument("--focal-gamma", type=float, default=d.focal_gamma)
    g.add_argument("--label-smoothing", type=float, default=d.label_smoothing)
    g.add_argument("--no-class-weights", dest="use_class_weights", action="store_false",
                   default=d.use_class_weights)

    g = p.add_argument_group("evaluation")
    g.add_argument("--folds", type=int, default=d.folds)
    g.add_argument("--repeats", type=int, default=d.repeats)
    g.add_argument("--val-fraction", type=float, default=d.val_fraction)
    g.add_argument("--bootstrap", type=int, default=d.bootstrap)
    g.add_argument("--alpha", type=float, default=d.alpha)
    g.add_argument("--no-calibration", dest="calibrate", action="store_false", default=d.calibrate)
    g.add_argument("--models", type=str, nargs="+", default=d.models, choices=ALL_MODELS)
    g.add_argument("--reference-model", type=str, default=d.reference_model, choices=ALL_MODELS)
    g.add_argument("--ablations", action="store_true")
    g.add_argument("--no-fairness", dest="fairness", action="store_false", default=d.fairness)
    g.add_argument("--no-explain", dest="explain", action="store_false", default=d.explain)
    g.add_argument("--no-shortcut-test", dest="shortcut_test", action="store_false",
                   default=d.shortcut_test,
                   help="skip the background-only probe for acquisition shortcuts")
    g.add_argument("--shortcut-mask-fraction", type=float, default=d.shortcut_mask_fraction)

    g = p.add_argument_group("infrastructure")
    g.add_argument("--outdir", type=str, default=d.outdir)
    g.add_argument("--seed", type=int, default=d.seed)
    g.add_argument("--device", type=str, default=d.device)
    g.add_argument("--quick", action="store_true", help="Fast reduced-budget run for debugging")
    g.add_argument("--dpi", type=int, default=d.dpi)

    ns = p.parse_args(argv)
    cfg = Config(**{k: v for k, v in vars(ns).items() if k in Config.__dataclass_fields__})
    return cfg.resolve()

### 1.17 MAIN

In [ ]:
# =========================================================================== #
# SECTION 17 - MAIN                                                           #
# =========================================================================== #
def main(argv: Optional[Sequence[str]] = None) -> int:
    cfg = parse_args(argv)
    outdir = Path(cfg.outdir).expanduser().resolve()
    setup_logging(outdir)
    set_publication_style()
    set_seed(cfg.seed)

    device = resolve_device(cfg.device)
    cfg.device = str(device)
    t_start = time.time()

    banner("DERM-GNN | Rare Skin Disease Dataset | graph-based diagnostic modelling")
    LOGGER.info("Output directory : %s", outdir)
    LOGGER.info("Device           : %s", device)
    LOGGER.info("Torch            : %s", torch.__version__)
    LOGGER.info("Seed             : %d", cfg.seed)
    save_json(asdict(cfg), outdir / "config.json")

    # ---- 1. data --------------------------------------------------------
    banner("Step 1/9 | Data acquisition and quality control")
    if cfg.manifest:
        root = None
    elif cfg.synthetic:
        root = make_synthetic_dataset(outdir / "synthetic_data", cfg.synthetic_n, cfg.seed)
    elif cfg.download:
        root = download_dataset(outdir / "download")
    elif cfg.data_root:
        root = Path(cfg.data_root).expanduser().resolve()
        if not root.exists():
            LOGGER.error("--data-root does not exist: %s", root)
            return 2
    else:
        LOGGER.error(
            "No data source given. Use one of --data-root <path>, --download, or --synthetic."
        )
        return 2
    if cfg.manifest:
        mpath = Path(cfg.manifest).expanduser().resolve()
        if not mpath.exists():
            LOGGER.error("--manifest does not exist: %s", mpath)
            return 2
        df, flow = load_manifest_dataset(cfg, mpath)
        df, flow = _finalise_cohort(cfg, df, flow)
    else:
        root = maybe_extract_zip(root)
        df, flow = discover_dataset(cfg, root)
    if cfg.fairness:
        if cfg.skin_tone_column and cfg.skin_tone_column in df.columns:
            df = df.copy()
            df["ita_band"] = df[cfg.skin_tone_column].astype(str).str.strip()
            df["ita"] = np.nan
            df["skin_tone_source"] = "recorded"
            LOGGER.info("Fairness strata from recorded column '%s': %s",
                        cfg.skin_tone_column,
                        df["ita_band"].value_counts().to_dict())
        else:
            df = annotate_skin_tone(df)
            df["skin_tone_source"] = "ita_proxy"
    classes = flow["classes"]
    y = df["y"].to_numpy().astype(int)
    df.to_csv(outdir / "cohort_manifest.csv", index=False)

    # ---- 2. representation ----------------------------------------------
    banner("Step 2/9 | Image representation")
    raw_feats, feature_desc = extract_features(cfg, df, device)
    np.save(outdir / "embeddings_raw.npy", raw_feats)
    X, feat_info = reduce_features(raw_feats, cfg.pca_dim, cfg.seed)

    # ---- 3. graph -------------------------------------------------------
    banner("Step 3/9 | Population-graph construction")
    S = similarity_matrix(X, cfg.graph_metric)
    all_mask = np.ones(len(y), bool)
    ei_all, _ = build_knn_edges(S, cfg.knn_k, all_mask, all_mask, cfg.mutual_knn, cfg.edge_sim_threshold)
    gstat_all = graph_statistics(ei_all, len(y), y, all_mask)
    LOGGER.info(
        "Full-cohort graph: %d edges, mean degree %.2f, homophily %.3f (chance %.3f)",
        gstat_all["num_edges"], gstat_all["mean_degree"], gstat_all["homophily"],
        1.0 / len(classes),
    )

    # ---- 4. cross-validated benchmark -----------------------------------
    banner(f"Step 4/9 | Cross-validation ({cfg.repeats} x {cfg.folds}-fold, {cfg.graph_mode}, "
           f"{cfg.feature_fit} feature fitting)")
    if cfg.finetune:
        LOGGER.info("Encoder fine-tuning is ON: the backbone is retrained inside every "
                    "one of the %d splits. Expect a substantially longer run.",
                    cfg.repeats * cfg.folds)
    groups = df["group"].to_numpy() if cfg.group_column and "group" in df else None
    if groups is not None:
        LOGGER.info("Group-aware cross-validation: %d groups over %d images "
                    "(no group spans a split).", len(np.unique(groups)), len(df))
    res = run_cross_validation(cfg, X, y, S, classes, device, raw_feats=raw_feats,
                               image_paths=df["path"].tolist(), groups=groups)
    if res.fold_metrics.empty:
        LOGGER.error("No model produced results; aborting.")
        return 1
    res.fold_metrics.to_csv(outdir / "fold_metrics.csv", index=False)
    table, primary, boot = summarise_results(cfg, res, y)

    LOGGER.info("")
    LOGGER.info("Primary estimates: fold mean (Nadeau-Bengio corrected 95%% CI)")
    LOGGER.info("%-20s %-28s %-28s", "Model", METRIC_LABELS[PRIMARY_METRIC], "Macro F1")
    LOGGER.info("-" * 78)
    for _, r in table.iterrows():
        m = r["Model"]
        LOGGER.info("%-20s %-28s %-28s", m,
                    fmt_ci(*primary[m][PRIMARY_METRIC]), fmt_ci(*primary[m]["macro_f1"]))
    LOGGER.info("")

    ref = cfg.reference_model
    if ref not in res.oof_prob:
        ref = table.iloc[0]["Model"]
        cfg.reference_model = ref
        LOGGER.warning("Reference model unavailable; using %s instead.", ref)
    ref_prob = res.oof_prob[ref]

    # ---- 5. inference ---------------------------------------------------
    banner("Step 5/9 | Inferential comparison")
    fold_primary = {
        m: res.fold_metrics[res.fold_metrics["model"] == m][PRIMARY_METRIC].dropna().tolist()
        for m in cfg.models if m in res.oof_prob
    }
    fried = friedman_nemenyi(fold_primary, cfg.alpha)
    if fried.get("applicable"):
        LOGGER.info("Friedman chi2 = %.2f, p = %.3e, CD = %.2f",
                    fried["statistic"], fried["p_value"], fried["critical_difference"])
    wilcox = pairwise_wilcoxon(fold_primary, ref)
    n_splits = cfg.repeats * cfg.folds
    if n_splits < WILCOXON_MIN_FOLDS:
        LOGGER.warning(
            "Only %d cross-validation splits: the paired Wilcoxon test needs at least %d "
            "to reach any attainable significance level, so pairwise p-values are reported "
            "as not-available. Increase --repeats / --folds for inferential comparison.",
            n_splits, WILCOXON_MIN_FOLDS,
        )
    elif not wilcox.empty:
        for _, r in wilcox.iterrows():
            LOGGER.info("  %s vs %-20s delta=%+.3f  p=%.4g  p_holm=%.4g",
                        ref, r["Comparator"], r["Delta_mean"], r["p"], r["p_holm"])

    delong_df = None
    comparator = next((m for m in ["MLP", "LogisticRegression", "SVM-RBF"] if m in res.oof_prob and m != ref), None)
    if comparator:
        delong_df = delong_macro(y, ref_prob, res.oof_prob[comparator], classes)
        delong_df.insert(0, "Comparator", comparator)
        delong_df.insert(0, "Reference", ref)
        LOGGER.info("DeLong (%s vs %s): min p_holm = %.4g", ref, comparator,
                    float(np.nanmin(delong_df["p_holm"].values)) if len(delong_df) else float("nan"))

    # ---- 6. fairness ----------------------------------------------------
    fair_df, fair_gaps, ita_check = None, {}, {"available": False}
    if cfg.fairness and "ita_band" in df.columns:
        banner("Step 6/9 | Fairness audit across pigmentation strata")
        recorded = str(df.get("skin_tone_source", pd.Series(["ita_proxy"])).iloc[0]) == "recorded"
        if recorded:
            ita_check = {"available": False, "recorded_skin_tone": True,
                         "column": cfg.skin_tone_column,
                         "interpretation": (
                             "Strata come from a recorded skin-tone variable, not the ITA "
                             "proxy, so no proxy-validity test is required and the subgroup "
                             "analysis is not limited by the absence of colour calibration.")}
            LOGGER.info("Skin tone is recorded, not inferred; proxy-validity test skipped.")
        else:
            ita_check = ita_proxy_validity(df, classes)
        if ita_check.get("available"):
            LOGGER.info("ITA proxy validity: Kruskal-Wallis H = %.2f, p = %.3g, "
                        "epsilon-squared = %.3f",
                        ita_check["kruskal_H"], ita_check["p_value"],
                        ita_check["epsilon_squared"])
            if ita_check["confounded_by_diagnosis"]:
                LOGGER.warning(
                    "ITA is strongly associated with diagnosis in this cohort. The "
                    "pigmentation strata are partly disease strata, so the subgroup "
                    "results below are EXPLORATORY and must not be presented as a "
                    "validated skin-tone fairness analysis."
                )
        fair_df, fair_gaps = fairness_audit(y, ref_prob, df["ita_band"].to_numpy(), len(classes))
        LOGGER.info("\n%s", fair_df.to_string(index=False))

    # ---- 6b. shortcut-learning probe -------------------------------------
    shortcut = {"available": False}
    if cfg.shortcut_test:
        shortcut = shortcut_learning_test(cfg, df, y, classes, device)

    # ---- 7. ablations ---------------------------------------------------
    abl_df = None
    if cfg.ablations:
        abl_df = run_ablations(cfg, X, y, S, classes, device, raw_feats=raw_feats)

    # ---- 8. figures -----------------------------------------------------
    banner("Step 7/9 | Figures")
    per_cls = per_class_metrics(y, ref_prob, classes)
    try:
        fig_cohort(df, flow, classes, cfg, outdir)
        fig_graph(S, cfg, y, classes, outdir)
        fig_performance(table, primary, res, cfg, outdir)
        fig_confusion(y, ref_prob, classes, cfg, outdir)
        fig_roc_pr(y, ref_prob, classes, cfg, outdir)
        fig_calibration(y, res.oof_prob_uncal[ref], ref_prob, cfg, outdir)
        fig_cd_diagram(fried, cfg, outdir)
        fig_dca(y, ref_prob, classes, cfg, outdir)
        if fair_df is not None:
            fig_fairness(fair_df, cfg, outdir)
        fig_training(res.histories, cfg, outdir)
        if abl_df is not None:
            fig_ablation(abl_df, cfg, outdir)
    except Exception as exc:
        LOGGER.error("Figure generation problem: %s", exc)

    if cfg.explain:
        banner("Step 8/9 | Explainability")
        try:
            Z_learned = learned_node_embeddings(cfg, X, y, S, classes, device)
            fig_embeddings(X, Z_learned, y, classes, cfg, outdir)
        except Exception as exc:
            LOGGER.error("Embedding figure failed: %s", exc)
        try:
            cams = class_activation_maps(cfg, df, raw_feats, y, classes, device)
            fig_cam(cams, cfg, outdir)
        except Exception as exc:
            LOGGER.error("Activation-map figure failed: %s", exc)

    # ---- 9. tables and report -------------------------------------------
    banner("Step 9/9 | Tables and manuscript report")
    export_table(cohort_table(df, classes), outdir, "table1_cohort",
                 "Baseline characteristics of the analysis cohort.")
    export_table(headline_table(table, primary), outdir, "table2_primary_results",
                 f"Cross-validated diagnostic performance: mean across folds with "
                 f"Nadeau-Bengio corrected {int((1 - cfg.alpha) * 100)}% confidence interval.")
    export_table(per_cls, outdir, "table3_per_class",
                 f"Class-wise performance of {ref} on pooled out-of-fold predictions.")
    if not wilcox.empty:
        export_table(wilcox, outdir, "table4_statistical_tests",
                     f"Paired Wilcoxon signed-rank tests against {ref}, Holm-adjusted.")
    if delong_df is not None:
        export_table(delong_df, outdir, "table4b_delong",
                     "DeLong tests on pooled out-of-fold one-vs-rest AUCs.")
    if fair_df is not None:
        export_table(fair_df, outdir, "table5_fairness",
                     "Performance stratified by Individual Typology Angle band.")
    if abl_df is not None:
        export_table(abl_df, outdir, "table6_ablations",
                     f"Ablation studies on {ref}.")
    export_table(tripod_ai_checklist(cfg, flow, res), outdir, "table_s1_tripod_ai",
                 "TRIPOD+AI reporting checklist mapping.", float_fmt="%s")
    table.to_csv(outdir / "tables" / "table2_primary_results_numeric.csv", index=False)

    results_blob = {
        "config": asdict(cfg),
        "feature_extraction": {"description": feature_desc, **feat_info},
        "data_flow": flow,
        "graph_full_cohort": gstat_all,
        "graph_fold_example": res.graph_stats,
        "classes": classes,
        "reference_model": ref,
        "temperatures": res.temperatures,
        "n_parameters": res.n_params,
        "runtime_seconds": res.runtime,
        "metrics_primary_fold_corrected": {
            m: {k: {"estimate": v[0], "ci_low": v[1], "ci_high": v[2]} for k, v in d.items()}
            for m, d in primary.items()
        },
        "metrics_secondary_pooled_bootstrap": {
            m: {k: {"estimate": v[0], "ci_low": v[1], "ci_high": v[2]} for k, v in d.items()}
            for m, d in boot.items()
        },
        "ita_proxy_validity": ita_check,
        "shortcut_learning_probe": shortcut,
        "metrics_fold_summary": (
            res.fold_metrics.groupby("model")[list(METRIC_LABELS)].agg(["mean", "std"]).round(4)
            .to_dict() if not res.fold_metrics.empty else {}
        ),
        "friedman_nemenyi": fried,
        "wilcoxon_vs_reference": wilcox.to_dict(orient="records") if not wilcox.empty else [],
        "delong": delong_df.to_dict(orient="records") if delong_df is not None else [],
        "fairness": {
            "table": fair_df.to_dict(orient="records") if fair_df is not None else [],
            "gaps": fair_gaps,
        },
        "ablations": abl_df.to_dict(orient="records") if abl_df is not None else [],
        "total_runtime_seconds": round(time.time() - t_start, 1),
    }
    save_json(results_blob, outdir / "results.json")

    write_report(cfg, flow, res, table, primary, boot, per_cls, fried, wilcox, delong_df,
                 fair_df, fair_gaps, ita_check, shortcut, abl_df, feature_desc, y, outdir)

    banner("Complete")
    LOGGER.info("Total runtime : %.1f s", time.time() - t_start)
    LOGGER.info("Artefacts     : %s", outdir)
    LOGGER.info("  REPORT.md            drafted Results section")
    LOGGER.info("  results.json         machine-readable record")
    LOGGER.info("  tables/*.csv|.tex    manuscript tables")
    LOGGER.info("  figures/*.png|.pdf   publication figures (%d dpi)", cfg.dpi)
    return 0

## 2. Benchmark engine

The DERM-Net architecture, its ablations and the benchmark runner.

In [ ]:
from __future__ import annotations

import argparse
import json
import logging
import math
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- everything below is reused rather than reimplemented -------------------

try:
    import timm
    _HAS_TIMM = True
except Exception:
    timm = None
    _HAS_TIMM = False

try:
    import torchvision
    from torchvision import transforms as T
    _HAS_TV = True
except Exception:
    torchvision = None
    T = None
    _HAS_TV = False

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x=None, **k):
        return x if x is not None else []


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)



# The graph pipeline and the benchmark share one namespace in this notebook.
GraphConfig = Config

### 2.1 DERM-Net ARCHITECTURE

In [ ]:
# =========================================================================== #
# SECTION 1 - DERM-Net ARCHITECTURE                                           #
# =========================================================================== #
class ChannelAttention1D(nn.Module):
    """Squeeze-and-excitation over channels, using both mean and max pooling."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        att = self.sigmoid(self.fc(torch.mean(x, dim=2)) +
                           self.fc(torch.max(x, dim=2).values)).unsqueeze(2)
        return x * att


class MSCABlock1D(nn.Module):
    """
    Multi-scale channel-attention fusion.

    Three parallel 1-D convolutions (kernel 1, 3, 5) view the concatenated
    backbone descriptor at different scales; a learned softmax over three
    scalars weights them; channel attention reweights the fused result; a
    residual path preserves the input. This is the block whose contribution the
    "no MSCA" ablation isolates.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        mid = max(1, out_channels // 3)
        self.branch1 = nn.Sequential(nn.Conv1d(in_channels, mid, 1, bias=False),
                                     nn.BatchNorm1d(mid), nn.GELU())
        self.branch3 = nn.Sequential(nn.Conv1d(in_channels, mid, 3, padding=1, bias=False),
                                     nn.BatchNorm1d(mid), nn.GELU())
        self.branch5 = nn.Sequential(nn.Conv1d(in_channels, mid, 5, padding=2, bias=False),
                                     nn.BatchNorm1d(mid), nn.GELU())
        fused_ch = mid * 3
        self.channel_att = ChannelAttention1D(fused_ch)
        self.project = nn.Sequential(nn.Conv1d(fused_ch, out_channels, 1, bias=False),
                                     nn.BatchNorm1d(out_channels))
        self.scale = nn.Parameter(torch.ones(3) / 3)
        self.residual = (nn.Conv1d(in_channels, out_channels, 1, bias=False)
                         if in_channels != out_channels else nn.Identity())
        self.act = nn.GELU()

    def forward(self, x):
        x = x.unsqueeze(2)
        w = F.softmax(self.scale, dim=0)
        fused = torch.cat([self.branch1(x) * w[0], self.branch3(x) * w[1],
                           self.branch5(x) * w[2]], dim=1)
        fused = self.channel_att(fused)
        return self.act(self.project(fused) + self.residual(x)).squeeze(2)


def _make_backbone(name: str, pretrained: bool) -> Tuple[nn.Module, int]:
    """Create a headless backbone, preferring timm and falling back to torchvision."""
    if _HAS_TIMM:
        m = timm.create_model(name, pretrained=pretrained, num_classes=0)
        return m, int(m.num_features)
    if not _HAS_TV:
        raise RuntimeError("Neither timm nor torchvision is available.")
    LOGGER.warning("timm unavailable; substituting a torchvision backbone for '%s'.", name)
    m = torchvision.models.resnet50(weights=None)
    dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, dim


class DERMNet(nn.Module):
    """
    The proposed architecture, and its three ablations, behind one flag set.

    variant:
      "full"     EfficientNet-B4 + ViT-B/16, MSCA fusion   (as published)
      "no_msca"  same backbones, concatenation + linear    (isolates MSCA)
      "eff"      EfficientNet-B4 alone                     (isolates dual-backbone)
      "vit"      ViT-B/16 alone                            (isolates dual-backbone)
    """

    def __init__(self, num_classes: int, pretrained: bool = True, variant: str = "full",
                 common_dim: int = 512, dropout: float = 0.4):
        super().__init__()
        self.variant = variant
        self.use_eff = variant in {"full", "no_msca", "eff"}
        self.use_vit = variant in {"full", "no_msca", "vit"}

        if self.use_eff:
            self.eff_features, eff_dim = _make_backbone("efficientnet_b4", pretrained)
            self.eff_proj = nn.Sequential(nn.Linear(eff_dim, common_dim),
                                          nn.LayerNorm(common_dim), nn.GELU())
        if self.use_vit:
            self.vit_features, vit_dim = _make_backbone("vit_base_patch16_224", pretrained)
            self.vit_proj = nn.Sequential(nn.Linear(vit_dim, common_dim),
                                          nn.LayerNorm(common_dim), nn.GELU())

        n_streams = int(self.use_eff) + int(self.use_vit)
        fused_in = common_dim * n_streams
        if variant == "full":
            self.fusion = MSCABlock1D(fused_in, common_dim)
        elif variant == "no_msca":
            # Matched-capacity control: same parameters budget, no multi-scale
            # attention. Any DERM-Net advantage over this row is the block.
            self.fusion = nn.Sequential(nn.Linear(fused_in, common_dim),
                                        nn.LayerNorm(common_dim), nn.GELU())
        else:
            self.fusion = nn.Identity()

        head_in = common_dim
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))

    def forward(self, x, return_features: bool = False):
        parts = []
        if self.use_eff:
            parts.append(self.eff_proj(self.eff_features(x)))
        if self.use_vit:
            parts.append(self.vit_proj(self.vit_features(x)))
        fused = self.fusion(torch.cat(parts, dim=1) if len(parts) > 1 else parts[0])
        fused = self.dropout(fused)
        logits = self.classifier(fused)
        return (logits, fused) if return_features else logits


class TorchvisionBaseline(nn.Module):
    """Standard single-backbone comparator (ResNet-50, DenseNet-121)."""

    def __init__(self, arch: str, num_classes: int, pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        if not _HAS_TV:
            raise RuntimeError("torchvision is required for the baseline models.")
        ctor = getattr(torchvision.models, arch)
        try:
            enum_name = {"resnet50": "ResNet50_Weights", "densenet121": "DenseNet121_Weights"}[arch]
            weights = getattr(torchvision.models, enum_name).DEFAULT if pretrained else None
            self.body = ctor(weights=weights)
        except Exception:
            self.body = ctor(pretrained=pretrained)
        if arch.startswith("resnet"):
            dim = self.body.fc.in_features
            self.body.fc = nn.Identity()
        else:
            dim = self.body.classifier.in_features
            self.body.classifier = nn.Identity()
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, num_classes))

    def forward(self, x, return_features: bool = False):
        f = self.body(x).flatten(1)
        logits = self.head(f)
        return (logits, f) if return_features else logits


MODEL_ZOO: "Dict[str, Dict[str, Any]]" = {
    "DERM-Net":            {"kind": "dermnet", "variant": "full",
                            "note": "proposed: EfficientNet-B4 + ViT-B/16, MSCA fusion"},
    "DERM-Net (no MSCA)":  {"kind": "dermnet", "variant": "no_msca",
                            "note": "ablation: same backbones, plain concatenation"},
    "DERM-Net (Eff only)": {"kind": "dermnet", "variant": "eff",
                            "note": "ablation: EfficientNet-B4 alone"},
    "DERM-Net (ViT only)": {"kind": "dermnet", "variant": "vit",
                            "note": "ablation: ViT-B/16 alone"},
    "ResNet-50":           {"kind": "tv", "arch": "resnet50", "note": "baseline"},
    "DenseNet-121":        {"kind": "tv", "arch": "densenet121", "note": "baseline"},
}
DEFAULT_MODELS = list(MODEL_ZOO)


def build_model(name: str, num_classes: int, pretrained: bool, dropout: float) -> nn.Module:
    spec = MODEL_ZOO[name]
    if spec["kind"] == "dermnet":
        return DERMNet(num_classes, pretrained=pretrained, variant=spec["variant"], dropout=dropout)
    return TorchvisionBaseline(spec["arch"], num_classes, pretrained=pretrained)

### 2.2 CONFIGURATION

In [ ]:
# =========================================================================== #
# SECTION 2 - CONFIGURATION                                                   #
# =========================================================================== #
@dataclass
class BenchConfig:
    manifest: Optional[str] = None
    data_root: Optional[str] = None
    dataset_name: str = "cohort"
    group_column: Optional[str] = None
    skin_tone_column: Optional[str] = None
    max_per_class: int = 0
    min_class_size: int = 10
    dedupe: bool = True
    dedupe_hamming: int = 4

    image_size: int = 224
    batch_size: int = 16
    num_workers: int = 2

    models: List[str] = field(default_factory=lambda: list(DEFAULT_MODELS))
    reference_model: str = "DERM-Net"
    pretrained: bool = True
    dropout: float = 0.4

    epochs: int = 20
    patience: int = 5
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    use_class_weights: bool = True
    amp: bool = True

    folds: int = 3
    repeats: int = 1
    val_fraction: float = 0.15
    bootstrap: int = 1000
    alpha: float = 0.05
    calibrate: bool = True

    outdir: str = "benchmark_results"
    seed: int = 42
    device: str = "auto"
    quick: bool = False
    dpi: int = 300

    def resolve(self) -> "BenchConfig":
        if self.quick:
            self.folds = 2
            self.repeats = 1
            self.epochs = 2
            self.patience = 1
            self.bootstrap = 100
        unknown = [m for m in self.models if m not in MODEL_ZOO]
        if unknown:
            raise ValueError(f"unknown model(s): {unknown}; choose from {list(MODEL_ZOO)}")
        if self.reference_model not in self.models:
            self.reference_model = self.models[0]
        return self

    def as_graph_config(self) -> GraphConfig:
        """Adapter so the shared quality-control code can be reused unchanged."""
        g = GraphConfig()
        g.group_column = self.group_column
        g.max_per_class = self.max_per_class
        g.min_class_size = self.min_class_size
        g.dedupe = self.dedupe
        g.dedupe_hamming = self.dedupe_hamming
        g.folds = self.folds
        g.seed = self.seed
        return g

### 2.3 DATA

In [ ]:
# =========================================================================== #
# SECTION 3 - DATA                                                            #
# =========================================================================== #
def build_transforms(image_size: int):
    """Training augmentation and deterministic evaluation transform."""
    train = T.Compose([
        T.RandomResizedCrop(image_size, scale=(0.65, 1.0), ratio=(0.8, 1.25)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(p=0.2),
        T.RandomApply([T.RandomRotation(20)], p=0.5),
        T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.03),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.10)),
    ])
    evl = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train, evl


class SkinDataset(torch.utils.data.Dataset):
    def __init__(self, paths: Sequence[str], labels: Sequence[int], transform):
        self.paths, self.labels, self.transform = list(paths), list(labels), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as im:
                img = im.convert("RGB")
        except Exception:
            img = Image.new("RGB", (256, 256), (128, 128, 128))
        return self.transform(img), int(self.labels[i])


def load_cohort(cfg: BenchConfig) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Load and quality-control the cohort, reusing the shared implementation."""
    g = cfg.as_graph_config()
    if not cfg.manifest:
        raise ValueError("--manifest is required; build one in the dataset loader cell.")
    df, flow = load_manifest_dataset(g, Path(cfg.manifest).expanduser().resolve())
    df, flow = _finalise_cohort(g, df, flow)
    flow["dataset_name"] = cfg.dataset_name
    return df, flow

### 2.4 TRAINING

In [ ]:
# =========================================================================== #
# SECTION 4 - TRAINING                                                        #
# =========================================================================== #
def train_one(
    name: str, cfg: BenchConfig, df: pd.DataFrame, y: np.ndarray,
    tr_idx: np.ndarray, va_idx: np.ndarray, te_idx: np.ndarray,
    n_cls: int, device: torch.device, seed: int,
) -> Dict[str, Any]:
    """
    Fine-tune one model on one fold.

    Every model gets exactly the same schedule, augmentation, budget and early
    stopping criterion. That equality is the point: it is what makes the
    comparison attributable to architecture.
    """
    set_seed(seed)
    model = build_model(name, n_cls, cfg.pretrained, cfg.dropout).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    train_tf, eval_tf = build_transforms(cfg.image_size)
    paths = df["path"].tolist()
    mk = lambda idx, tf: SkinDataset([paths[i] for i in idx], y[idx], tf)
    common = dict(num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"))
    tr_ld = torch.utils.data.DataLoader(mk(tr_idx, train_tf), batch_size=cfg.batch_size,
                                        shuffle=True, drop_last=len(tr_idx) > cfg.batch_size, **common)
    va_ld = torch.utils.data.DataLoader(mk(va_idx, eval_tf), batch_size=cfg.batch_size, **common)
    te_ld = torch.utils.data.DataLoader(mk(te_idx, eval_tf), batch_size=cfg.batch_size, **common)

    weights = class_weights_from(y[tr_idx], n_cls).to(device) if cfg.use_class_weights else None
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=cfg.label_smoothing)

    # Backbone weights are already useful; the head is random. Different rates.
    head_names = ("classifier", "head", "fusion", "eff_proj", "vit_proj")
    head_p = [p for n, p in model.named_parameters() if any(h in n for h in head_names)]
    body_p = [p for n, p in model.named_parameters() if not any(h in n for h in head_names)]
    opt = torch.optim.AdamW(
        [{"params": body_p, "lr": cfg.lr_backbone}, {"params": head_p, "lr": cfg.lr_head}],
        weight_decay=cfg.weight_decay)
    steps = max(cfg.epochs * max(len(tr_ld), 1), 1)
    # OneCycle needs enough steps for its warm-up phase to have non-zero length;
    # below that it divides by zero. Short runs (small cohorts, few epochs, a
    # --quick smoke test) fall back to cosine annealing.
    if steps >= 8:
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=[cfg.lr_backbone, cfg.lr_head], total_steps=steps, pct_start=0.25)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    @torch.no_grad()
    def logits_for(loader):
        model.eval()
        L, Y = [], []
        for xb, yb in loader:
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(xb.to(device, non_blocking=True))
            L.append(out.float().cpu())
            Y.append(yb)
        return torch.cat(L), torch.cat(Y)

    from sklearn.metrics import balanced_accuracy_score
    best, best_state, stale = -np.inf, None, 0
    history = {"train_loss": [], "val_bacc": []}
    for epoch in range(cfg.epochs):
        model.train()
        losses = []
        for xb, yb in tr_ld:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(float(loss.detach().item()))
        vl, vy = logits_for(va_ld)
        vb = balanced_accuracy_score(vy.numpy(), vl.argmax(1).numpy())
        history["train_loss"].append(float(np.mean(losses)) if losses else float("nan"))
        history["val_bacc"].append(float(vb))
        if vb > best + 1e-6:
            best, stale = vb, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_logits, val_y = logits_for(va_ld)
    test_logits, _ = logits_for(te_ld)
    temp = 1.0
    if cfg.calibrate and len(va_idx) >= max(2 * n_cls, 10):
        temp = fit_temperature(val_logits, val_y)
    test_prob = F.softmax(test_logits / temp, dim=1).numpy()
    test_prob_uncal = F.softmax(test_logits, dim=1).numpy()

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {"test_prob": test_prob, "test_prob_uncal": test_prob_uncal,
            "temperature": float(temp), "history": history, "n_params": int(n_params),
            "epochs_run": len(history["train_loss"]), "best_val_bacc": float(best)}

### 2.5 BENCHMARK RUNNER

In [ ]:
# =========================================================================== #
# SECTION 5 - BENCHMARK RUNNER                                                #
# =========================================================================== #
def run_benchmark(cfg: BenchConfig, df: pd.DataFrame, device: torch.device) -> Dict[str, Any]:
    y = df["y"].to_numpy().astype(int)
    n, n_cls = len(y), int(y.max()) + 1
    groups = df["group"].to_numpy() if (cfg.group_column and "group" in df) else None
    if groups is not None:
        LOGGER.info("Group-aware cross-validation: %d groups over %d images.",
                    len(np.unique(groups)), n)

    rows: List[Dict[str, Any]] = []
    oof = {m: np.full((n, n_cls), np.nan) for m in cfg.models}
    oof_unc = {m: np.full((n, n_cls), np.nan) for m in cfg.models}
    hist: Dict[str, Dict[str, List[float]]] = {}
    nparams: Dict[str, int] = {}
    runtime: Dict[str, float] = {m: 0.0 for m in cfg.models}

    total = cfg.repeats * cfg.folds
    step = 0
    g_cfg = cfg.as_graph_config()
    for rep in range(cfg.repeats):
        splitter = make_cv_splitter(g_cfg, groups, rep)
        for fold, (trval, te) in enumerate(splitter.split(np.zeros(n), y, groups)):
            step += 1
            tr, va = stratified_val_split(trval, y, cfg.val_fraction,
                                          cfg.seed + rep * 100 + fold, groups)
            LOGGER.info("--- split %d/%d (repeat %d, fold %d): train=%d val=%d test=%d ---",
                        step, total, rep, fold, len(tr), len(va), len(te))
            for name in cfg.models:
                t0 = time.time()
                try:
                    res = train_one(name, cfg, df, y, tr, va, te, n_cls, device,
                                    cfg.seed + rep * 1000 + fold * 10)
                except Exception as exc:
                    LOGGER.error("  %s failed on this fold: %s", name, exc)
                    continue
                dt = time.time() - t0
                runtime[name] += dt
                met = compute_metrics(y[te], res["test_prob"], n_cls)
                rows.append({"model": name, "repeat": rep, "fold": fold,
                             "n_test": len(te), "seconds": dt, **met})
                if rep == 0:
                    oof[name][te] = res["test_prob"]
                    oof_unc[name][te] = res["test_prob_uncal"]
                nparams[name] = res["n_params"]
                hist.setdefault(name, res["history"])
                LOGGER.info("  %-22s bAcc=%.3f  F1=%.3f  (%d epochs, %.0fs)",
                            name, met["balanced_accuracy"], met["macro_f1"],
                            res["epochs_run"], dt)

    fold_df = pd.DataFrame(rows)
    for m in cfg.models:
        for store in (oof, oof_unc):
            p = np.nan_to_num(store[m], nan=1.0 / max(n_cls, 1))
            store[m] = p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)

    return {"fold_metrics": fold_df, "oof": oof, "oof_uncal": oof_unc,
            "histories": hist, "n_params": nparams, "runtime": runtime,
            "y": y, "n_classes": n_cls}


def summarise(cfg: BenchConfig, res: Dict[str, Any]) -> Tuple[pd.DataFrame, Dict[str, Any], Dict[str, Any]]:
    fold_df, y, n_cls = res["fold_metrics"], res["y"], res["n_classes"]
    primary, boot, rows = {}, {}, []
    n_total = len(y)
    for m in cfg.models:
        sub = fold_df[fold_df["model"] == m]
        if sub.empty:
            continue
        n_test = float(sub["n_test"].mean())
        n_train = max(n_total - n_test, 1.0)
        primary[m] = {k: corrected_fold_ci(sub[k].tolist(), int(n_train), int(n_test),
                                           cfg.alpha, METRIC_BOUNDS.get(k))
                      for k in METRIC_LABELS}
        boot[m] = bootstrap_ci(y, res["oof"][m], n_cls, cfg.bootstrap, cfg.alpha, seed=cfg.seed)
        row: Dict[str, Any] = {"Model": m, "n_folds": int(len(sub)),
                               "Parameters (M)": round(res["n_params"].get(m, 0) / 1e6, 1),
                               "Train time (s/fold)": round(res["runtime"].get(m, 0) / max(len(sub), 1), 1)}
        for k, label in METRIC_LABELS.items():
            pe, lo, hi = primary[m][k]
            row[label] = pe
            row[f"{label} CI low"] = lo
            row[f"{label} CI high"] = hi
            row[f"{label} SD"] = float(sub[k].std(ddof=1)) if len(sub) > 1 else 0.0
        rows.append(row)
    table = pd.DataFrame(rows)
    if not table.empty:
        table = table.sort_values(METRIC_LABELS[PRIMARY_METRIC], ascending=False).reset_index(drop=True)
    return table, primary, boot


def formatted_table(table: pd.DataFrame, primary: Dict[str, Any]) -> pd.DataFrame:
    """Publication table: point estimate with corrected 95% CI for every metric."""
    out = []
    for _, r in table.iterrows():
        m = r["Model"]
        row = {"Model": m, "Params (M)": r["Parameters (M)"], "s/fold": r["Train time (s/fold)"]}
        for k, label in METRIC_LABELS.items():
            pe, lo, hi = primary[m][k]
            row[label] = fmt_ci(pe, lo, hi, pct=k not in {"brier", "ece", "mcc", "kappa"})
        out.append(row)
    return pd.DataFrame(out)

### 2.6 FIGURES

In [ ]:
# =========================================================================== #
# SECTION 6 - FIGURES                                                         #
# =========================================================================== #
def fig_benchmark(table: pd.DataFrame, primary: Dict[str, Any], res: Dict[str, Any],
                  cfg: BenchConfig, outdir: Path, chance: float) -> None:
    if table.empty:
        return
    label = METRIC_LABELS[PRIMARY_METRIC]
    order = table.sort_values(label)["Model"].tolist()
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), gridspec_kw={"width_ratios": [1.2, 1]})

    ax = axes[0]
    for i, m in enumerate(order):
        pe, lo, hi = primary[m][PRIMARY_METRIC]
        col = PALETTE[list(MODEL_ZOO).index(m) % len(PALETTE)] if m in MODEL_ZOO else "#666"
        lw = 2.6 if m == cfg.reference_model else 1.4
        if np.isfinite(lo) and np.isfinite(hi):
            ax.plot([lo, hi], [i, i], color=col, lw=lw, solid_capstyle="round")
        ax.plot(pe, i, "o", color=col, markersize=8 if m == cfg.reference_model else 5,
                markeredgecolor="white", markeredgewidth=0.8, zorder=3)
        ax.text(1.005, i, fmt_ci(pe, lo, hi), transform=ax.get_yaxis_transform(),
                va="center", fontsize=7.5, color="0.25")
    ax.axvline(chance, color="0.5", ls="--", lw=0.9)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([f"$\\bf{{{m.replace(' ', chr(92)+' ')}}}$" if m == cfg.reference_model else m
                        for m in order], fontsize=8)
    ax.set_xlabel(f"{label} (fold mean, corrected 95% CI)")
    ax.set_title("A  Benchmark", loc="left", fontweight="bold")
    ax.grid(axis="y", visible=False)

    ax = axes[1]
    data, labels, cols = [], [], []
    for m in order:
        v = res["fold_metrics"][res["fold_metrics"]["model"] == m]["macro_f1"].dropna()
        if len(v):
            data.append(v.to_numpy())
            labels.append(m)
            cols.append(PALETTE[list(MODEL_ZOO).index(m) % len(PALETTE)] if m in MODEL_ZOO else "#666")
    if data:
        try:
            bp = ax.boxplot(data, orientation="horizontal", widths=0.6,
                            patch_artist=True, showfliers=False)
        except (TypeError, AttributeError):
            bp = ax.boxplot(data, vert=False, widths=0.6, patch_artist=True, showfliers=False)
        for patch, c in zip(bp["boxes"], cols):
            patch.set_facecolor(c); patch.set_alpha(0.35); patch.set_edgecolor(c)
        for el in ("medians", "whiskers", "caps"):
            for a in bp[el]:
                a.set_color("0.3")
        ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Macro F1 per fold")
    ax.set_title("B  Fold-level dispersion", loc="left", fontweight="bold")
    ax.grid(axis="y", visible=False)

    fig.suptitle(f"Benchmark on {cfg.dataset_name}", y=1.03, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "bench01_models", cfg.dpi)


def fig_efficiency(table: pd.DataFrame, cfg: BenchConfig, outdir: Path) -> None:
    """Accuracy against parameter count: is the extra capacity buying anything?"""
    if table.empty or "Parameters (M)" not in table:
        return
    label = METRIC_LABELS[PRIMARY_METRIC]
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for _, r in table.iterrows():
        m = r["Model"]
        col = PALETTE[list(MODEL_ZOO).index(m) % len(PALETTE)] if m in MODEL_ZOO else "#666"
        ax.scatter(r["Parameters (M)"], r[label] * 100, s=110 if m == cfg.reference_model else 60,
                   color=col, edgecolors="white", linewidth=0.9, zorder=3)
        ax.annotate(m, (r["Parameters (M)"], r[label] * 100), fontsize=7.5,
                    xytext=(6, 4), textcoords="offset points")
    ax.set_xlabel("Parameters (millions)")
    ax.set_ylabel(label + " (%)")
    ax.set_title("Accuracy versus model size", loc="left", fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "bench02_efficiency", cfg.dpi)


def fig_training(hist: Dict[str, Dict[str, List[float]]], cfg: BenchConfig, outdir: Path) -> None:
    if not hist:
        return
    fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.4))
    for name, h in hist.items():
        col = PALETTE[list(MODEL_ZOO).index(name) % len(PALETTE)] if name in MODEL_ZOO else "#666"
        if h["train_loss"]:
            axes[0].plot(h["train_loss"], color=col, lw=1.3, label=name)
        if h["val_bacc"]:
            axes[1].plot(h["val_bacc"], color=col, lw=1.3, label=name)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Training loss")
    axes[0].set_title("A  Optimisation", loc="left", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation balanced accuracy")
    axes[1].set_title("B  Model selection criterion", loc="left", fontweight="bold")
    axes[1].legend(fontsize=7, loc="lower right")
    fig.suptitle("Training dynamics (first fold)", y=1.04, fontsize=10, fontweight="bold")
    fig.tight_layout()
    savefig(fig, outdir, "bench03_training", cfg.dpi)

### 2.7 ENTRY POINT

In [ ]:
# =========================================================================== #
# SECTION 7 - ENTRY POINT                                                     #
# =========================================================================== #
def parse_args(argv: Optional[Sequence[str]] = None) -> BenchConfig:
    d = BenchConfig()
    ap = argparse.ArgumentParser(description="DERM-Net cross-dataset benchmark")
    ap.add_argument("--manifest", type=str, default=d.manifest)
    ap.add_argument("--dataset-name", type=str, default=d.dataset_name)
    ap.add_argument("--group-column", type=str, default=d.group_column)
    ap.add_argument("--skin-tone-column", type=str, default=d.skin_tone_column)
    ap.add_argument("--max-per-class", type=int, default=d.max_per_class)
    ap.add_argument("--min-class-size", type=int, default=d.min_class_size)
    ap.add_argument("--no-dedupe", dest="dedupe", action="store_false", default=d.dedupe)
    ap.add_argument("--image-size", type=int, default=d.image_size)
    ap.add_argument("--batch-size", type=int, default=d.batch_size)
    ap.add_argument("--num-workers", type=int, default=d.num_workers)
    ap.add_argument("--models", type=str, nargs="+", default=d.models, choices=list(MODEL_ZOO))
    ap.add_argument("--reference-model", type=str, default=d.reference_model, choices=list(MODEL_ZOO))
    ap.add_argument("--no-pretrained", dest="pretrained", action="store_false", default=d.pretrained)
    ap.add_argument("--epochs", type=int, default=d.epochs)
    ap.add_argument("--patience", type=int, default=d.patience)
    ap.add_argument("--lr-backbone", type=float, default=d.lr_backbone)
    ap.add_argument("--lr-head", type=float, default=d.lr_head)
    ap.add_argument("--folds", type=int, default=d.folds)
    ap.add_argument("--repeats", type=int, default=d.repeats)
    ap.add_argument("--bootstrap", type=int, default=d.bootstrap)
    ap.add_argument("--no-amp", dest="amp", action="store_false", default=d.amp)
    ap.add_argument("--outdir", type=str, default=d.outdir)
    ap.add_argument("--seed", type=int, default=d.seed)
    ap.add_argument("--device", type=str, default=d.device)
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--dpi", type=int, default=d.dpi)
    ns = ap.parse_args(argv)
    cfg = BenchConfig(**{k: v for k, v in vars(ns).items() if k in BenchConfig.__dataclass_fields__})
    return cfg.resolve()


def main(argv: Optional[Sequence[str]] = None) -> int:
    cfg = parse_args(argv)
    outdir = Path(cfg.outdir).expanduser().resolve()
    setup_logging(outdir)
    set_publication_style()
    set_seed(cfg.seed)
    device = resolve_device(cfg.device)
    cfg.device = str(device)
    t0 = time.time()

    banner(f"DERM-Net benchmark | {cfg.dataset_name} | device {device}")
    LOGGER.info("Models: %s", ", ".join(cfg.models))
    if not _HAS_TIMM:
        LOGGER.warning("timm is not installed: EfficientNet-B4 and ViT-B/16 cannot be built "
                       "as published, and results will NOT be comparable.")
    save_json(asdict(cfg), outdir / "config.json")

    banner("Step 1/4 | Cohort")
    df, flow = load_cohort(cfg)
    df.to_csv(outdir / "cohort_manifest.csv", index=False)
    n_cls = int(df["y"].max()) + 1
    chance = 1.0 / n_cls

    banner(f"Step 2/4 | Benchmark ({cfg.repeats} x {cfg.folds}-fold, {cfg.epochs} epochs max)")
    res = run_benchmark(cfg, df, device)
    if res["fold_metrics"].empty:
        LOGGER.error("No model completed a fold; aborting.")
        return 1
    res["fold_metrics"].to_csv(outdir / "fold_metrics.csv", index=False)
    table, primary, boot = summarise(cfg, res)

    LOGGER.info("")
    LOGGER.info("%-24s %-26s %-26s %8s", "Model", METRIC_LABELS[PRIMARY_METRIC], "Macro F1", "Params")
    LOGGER.info("-" * 92)
    for _, r in table.iterrows():
        m = r["Model"]
        LOGGER.info("%-24s %-26s %-26s %7.1fM", m,
                    fmt_ci(*primary[m][PRIMARY_METRIC]), fmt_ci(*primary[m]["macro_f1"]),
                    r["Parameters (M)"])
    LOGGER.info("")

    banner("Step 3/4 | Statistical comparison")
    fold_primary = {m: res["fold_metrics"][res["fold_metrics"]["model"] == m][PRIMARY_METRIC].dropna().tolist()
                    for m in cfg.models}
    fried = friedman_nemenyi(fold_primary, cfg.alpha)
    if fried.get("applicable"):
        LOGGER.info("Friedman chi2=%.2f p=%.3e CD=%.2f",
                    fried["statistic"], fried["p_value"], fried["critical_difference"])
    wilcox = pairwise_wilcoxon(fold_primary, cfg.reference_model)
    if not wilcox.empty:
        for _, r in wilcox.iterrows():
            LOGGER.info("  %s vs %-22s delta=%+.3f p_holm=%.4g",
                        cfg.reference_model, r["Comparator"], r["Delta_mean"], r["p_holm"])

    banner("Step 4/4 | Tables, figures and report")
    ref = cfg.reference_model if cfg.reference_model in res["oof"] else table.iloc[0]["Model"]
    classes = sorted(df["label"].unique(), key=lambda c: df.loc[df["label"] == c, "y"].iloc[0])
    per_cls = per_class_metrics(res["y"], res["oof"][ref], classes)

    export_table(formatted_table(table, primary), outdir, "benchmark_main",
                 f"Cross-validated benchmark on {cfg.dataset_name}: point estimate "
                 f"({int((1 - cfg.alpha) * 100)}% Nadeau-Bengio corrected CI).", "%s")
    export_table(per_cls, outdir, "benchmark_per_class",
                 f"Class-wise performance of {ref} on {cfg.dataset_name}.")
    if not wilcox.empty:
        export_table(wilcox, outdir, "benchmark_statistics",
                     f"Paired Wilcoxon tests against {cfg.reference_model}, Holm-adjusted.")
    table.to_csv(outdir / "tables" / "benchmark_main_numeric.csv", index=False)

    try:
        fig_benchmark(table, primary, res, cfg, outdir, chance)
        fig_efficiency(table, cfg, outdir)
        fig_training(res["histories"], cfg, outdir)
    except Exception as exc:
        LOGGER.error("Figure generation problem: %s", exc)

    summary = {
        "dataset": cfg.dataset_name,
        "config": asdict(cfg),
        "data_flow": flow,
        "classes": classes,
        "chance": chance,
        "reference_model": ref,
        "n_parameters": res["n_params"],
        "runtime_seconds": res["runtime"],
        "metrics_primary_fold_corrected": {
            m: {k: {"estimate": v[0], "ci_low": v[1], "ci_high": v[2]} for k, v in d.items()}
            for m, d in primary.items()},
        "metrics_secondary_pooled_bootstrap": {
            m: {k: {"estimate": v[0], "ci_low": v[1], "ci_high": v[2]} for k, v in d.items()}
            for m, d in boot.items()},
        "friedman_nemenyi": fried,
        "wilcoxon_vs_reference": wilcox.to_dict(orient="records") if not wilcox.empty else [],
        "total_runtime_seconds": round(time.time() - t0, 1),
    }
    save_json(summary, outdir / "benchmark_summary.json")

    banner("Complete")
    LOGGER.info("Runtime : %.1f s", time.time() - t0)
    LOGGER.info("Output  : %s", outdir)
    return 0

## 3. Render figures inline

In [ ]:
import io
from IPython.display import Image as IPyImage, display

SCREEN_DPI = 110


def savefig(fig, outdir, name, dpi=600):
    figdir = Path(outdir) / "figures"
    figdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figdir / f"{name}.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(figdir / f"{name}.pdf", bbox_inches="tight")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=SCREEN_DPI, bbox_inches="tight")
    plt.close(fig)
    display(IPyImage(data=buf.getvalue()))
    LOGGER.info("  figure saved: figures/%s.{png,pdf}", name)


print("Figures will render inline as well as being saved.")

## 4. Load Diverse Dermatology Images (DDI) and build the manifest

In [ ]:
DATASET_NAME  = "DDI"
MODALITY      = "clinical photography"
GROUP_COLUMN  = ""           # one image per lesion; no grouping variable published
MAX_PER_CLASS = 0
LABEL_MODE    = "malignant"  # "malignant" (binary, recommended) or "disease" (~78 classes)
SKIN_TONE_COLUMN = ""        # set below if the metadata records skin tone

# ---- known Kaggle mount for this dataset --------------------------------
# Layout:  <base>/ddidiversedermatologyimages/Images/000001.png ...
#          <base>/ddidiversedermatologyimages/ddi_metadata.csv
# Note the CSV sits one level ABOVE the Images folder.
_BASE = "/kaggle/input/datasets/souvikda/ddidiversedermatologyimages-multimodal-dataset"
IMAGES_ROOT  = f"{_BASE}/ddidiversedermatologyimages/Images"
METADATA_CSV = f"{_BASE}/ddidiversedermatologyimages/ddi_metadata.csv"

import glob as _glob

IMG_EXT = {".png", ".jpg", ".jpeg"}


def _n_images(d):
    try:
        return sum(1 for p in Path(d).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)
    except Exception:
        return 0


# ---- 1. locate the images -----------------------------------------------
if not (IMAGES_ROOT and Path(IMAGES_ROOT).exists() and _n_images(IMAGES_ROOT)):
    print(f"Stated image path not usable, searching /kaggle/input ...")
    cands = ([IMAGES_ROOT, f"{_BASE}/ddidiversedermatologyimages", _BASE,
              "/kaggle/input/ddidiversedermatologyimages"]
             + sorted(_glob.glob("/kaggle/input/*"))
             + sorted(_glob.glob("/kaggle/input/*/*"))
             + sorted(_glob.glob("/kaggle/input/*/*/*"))
             + ["./ddi", "./DDI"])
    best, n = None, 0
    for c in cands:
        if c and Path(c).exists():
            k = _n_images(c)
            if k > n:
                best, n = c, k
    IMAGES_ROOT = best

if not IMAGES_ROOT:
    raise FileNotFoundError("No DDI images found. Attach the dataset and re-run.")
root = Path(IMAGES_ROOT)
print(f"images root : {root}   ({_n_images(root)} image files)")

# ---- 2. locate the metadata CSV -----------------------------------------
# Labels live only in this file; the filenames carry no diagnosis.
def _looks_like_ddi_meta(path):
    try:
        head = pd.read_csv(path, nrows=5, dtype=str)
    except Exception:
        return False
    cols = {c.strip().lower() for c in head.columns}
    return ("ddi_file" in cols) or ({"malignant", "skin_tone"} <= cols) or (
        "disease" in cols and any("file" in c for c in cols))


search = []
if METADATA_CSV and Path(METADATA_CSV).exists():
    search.append(Path(METADATA_CSV))
# the images folder, then each parent up to the mount point
for up in [root] + list(root.parents)[:4]:
    try:
        search += sorted(up.glob("*.csv"))
    except Exception:
        pass
if Path("/kaggle/input").exists():
    for pat in ("/kaggle/input/*/*.csv", "/kaggle/input/*/*/*.csv", "/kaggle/input/*/*/*/*.csv"):
        search += [Path(x) for x in sorted(_glob.glob(pat))]

seen, meta_csv = set(), None
for c in search:
    if str(c) in seen:
        continue
    seen.add(str(c))
    if _looks_like_ddi_meta(c):
        meta_csv = c
        break

if meta_csv is None:
    raise FileNotFoundError(
        "\n" + "!" * 72 +
        "\nDDI metadata CSV not found, so there are no labels to train on."
        "\n\nThe images (000001.png ...) carry no diagnosis, malignancy flag or"
        "\nskin-tone information - all of that lives in ddi_metadata.csv."
        f"\n\nCSV files seen: {[str(x) for x in list(seen)[:8]] or 'none'}"
        "\n\nSet METADATA_CSV at the top of this cell to the correct path and re-run."
        "\n" + "!" * 72)

meta = pd.read_csv(meta_csv, dtype=str)   # dtype=str: keep filenames verbatim
print(f"metadata    : {meta_csv}   rows={len(meta)}")
print(f"columns     : {list(meta.columns)}")

# ---- 3. match metadata rows to image files ------------------------------
index = {}
for p in root.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMG_EXT:
        index[p.name] = p
        index[p.stem] = p

fcol = next((c for c in meta.columns
             if c.strip().lower() in {"ddi_file", "file", "filename", "image", "image_id", "path"}),
            meta.columns[0])
print(f"join column : {fcol}")


def _resolve(v):
    v = str(v).strip()
    for key in (v, Path(v).name, Path(v).stem):
        if key in index:
            return str(index[key])
    return None


meta["path"] = meta[fcol].map(_resolve)
found = meta["path"].notna()
print(f"matched     : {found.sum()} / {len(meta)} metadata rows resolved to an image")
if found.sum() == 0:
    raise RuntimeError(
        "No metadata row matched an image file.\n"
        f"  first join values: {meta[fcol].head(3).tolist()}\n"
        f"  first image names: {sorted(k for k in index if '.' in k)[:3]}\n"
        "Set fcol above to the column holding the image filename.")
meta = meta[found].copy()

# ---- 4. labels -----------------------------------------------------------
mcol = next((c for c in meta.columns if c.strip().lower() == "malignant"), None)
dcol = next((c for c in meta.columns if c.strip().lower() in {"disease", "diagnosis"}), None)
if LABEL_MODE == "malignant" and mcol:
    meta["label"] = meta[mcol].map(
        lambda v: "Malignant" if str(v).strip().lower() in {"1", "true", "yes", "t"} else "Benign")
elif dcol:
    meta["label"] = meta[dcol].astype(str)
    print("NOTE: fine-grained disease labels; most have very few images and classes")
    print("      below the minimum size will be dropped during quality control.")
else:
    raise RuntimeError(f"No usable label column in {list(meta.columns)}")
print(f"label mode  : {LABEL_MODE}")

cols = ["path", "label"]
scol = next((c for c in meta.columns
             if c.strip().lower() in {"skin_tone", "skin_type", "fitzpatrick"}), None)
if scol:
    meta["skin_tone"] = meta[scol]
    cols.append("skin_tone")
    print("\nSkin tone distribution (Fitzpatrick groups):")
    print(meta["skin_tone"].value_counts().sort_index().to_string())

MANIFEST = "ddi_manifest.csv"
meta[cols].to_csv(MANIFEST, index=False)
print(f"\nwrote {MANIFEST}: {len(meta)} rows, {meta['label'].nunique()} classes")
display(meta["label"].value_counts().rename("images").to_frame())

## 5. Run the benchmark

Six models, 1-2 hours on a T4. Reduce `FOLDS` or `EPOCHS` if the session is short.

In [ ]:
# ---- benchmark budget ----------------------------------------------------
# DERM-Net is ~107M parameters and six models are trained per fold, so this is
# the setting that determines whether the run finishes. On a Kaggle T4 the
# defaults below take roughly 1-2 hours for this dataset.
FOLDS      = 3
EPOCHS     = 20
BATCH      = 16
MODELS     = ["DERM-Net", "DERM-Net (no MSCA)", "DERM-Net (Eff only)",
              "DERM-Net (ViT only)", "ResNet-50", "DenseNet-121"]
OUTDIR     = ("/kaggle/working/bench_ddi" if Path("/kaggle/working").exists()
              else "bench_ddi")

args = [
    "--manifest", MANIFEST,
    "--dataset-name", DATASET_NAME,
    "--outdir", OUTDIR,
    "--folds", str(FOLDS),
    "--repeats", "1",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH),
    "--num-workers", "2",
    "--dpi", "300",
]
args += ["--models"] + MODELS
if GROUP_COLUMN:
    args += ["--group-column", GROUP_COLUMN]
if MAX_PER_CLASS:
    args += ["--max-per-class", str(MAX_PER_CLASS)]

print("dermnet_benchmark.py " + " ".join(args) + "\n")
exit_code = main(args)
print(f"\nFinished with exit code {exit_code}")

## 6. Result summary for this dataset

In [ ]:
import json

out = Path(OUTDIR)
blob = json.loads((out / "benchmark_summary.json").read_text())

ref = blob["reference_model"]
prim = blob["metrics_primary_fold_corrected"]
rank = sorted(prim, key=lambda m: -prim[m]["balanced_accuracy"]["estimate"])

row = {
    "dataset": blob["dataset"],
    "n_images": blob["data_flow"]["analysed"],
    "n_classes": len(blob["classes"]),
    "chance": blob["chance"],
    "best_model": rank[0],
    "best_bacc": prim[rank[0]]["balanced_accuracy"]["estimate"],
    "dermnet_bacc": prim.get("DERM-Net", {}).get("balanced_accuracy", {}).get("estimate"),
    "dermnet_rank": rank.index("DERM-Net") + 1 if "DERM-Net" in rank else None,
    "n_models": len(rank),
    "friedman_p": blob.get("friedman_nemenyi", {}).get("p_value"),
}
# Did DERM-Net beat its own ablation without the fusion block?
w = {r["Comparator"]: r for r in blob.get("wilcoxon_vs_reference", [])}
if "DERM-Net (no MSCA)" in w:
    row["msca_gain_pp"] = round(100 * w["DERM-Net (no MSCA)"]["Delta_mean"], 2)
    row["msca_p_holm"] = w["DERM-Net (no MSCA)"]["p_holm"]

dest = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
(dest / f"benchmark_row_{DATASET_NAME.replace(' ', '_')}.json").write_text(json.dumps(row, indent=2))

print(json.dumps(row, indent=2))
print()
print("=" * 74)
print(f"{blob['dataset']}: DERM-Net ranked {row['dermnet_rank']} of {row['n_models']}; "
      f"best was {row['best_model']}.")
if "msca_gain_pp" in row:
    verdict = ("supported" if (row["msca_gain_pp"] > 0 and (row["msca_p_holm"] or 1) < 0.05)
               else "not supported")
    print(f"MSCA fusion block vs plain concatenation: {row['msca_gain_pp']:+.2f} pp, "
          f"Holm p = {row['msca_p_holm']}  ->  {verdict}")
print("=" * 74)

## 7. Cross-dataset comparison

In [ ]:
import glob, json

rows = []
for f in sorted(glob.glob("benchmark_row_*.json") + glob.glob("/kaggle/input/*/benchmark_row_*.json")):
    try:
        rows.append(json.loads(Path(f).read_text()))
    except Exception:
        continue

if rows:
    cols = ["dataset", "n_images", "n_classes", "chance", "dermnet_bacc",
            "dermnet_rank", "n_models", "best_model", "best_bacc",
            "msca_gain_pp", "msca_p_holm"]
    cmp_df = pd.DataFrame(rows)
    cmp_df = cmp_df[[c for c in cols if c in cmp_df.columns]]
    display(cmp_df.round(4))
    print("\nRead this table two ways:")
    print("  dermnet_rank  - where the proposed model placed among all models tested")
    print("  msca_gain_pp  - what the fusion block contributed over plain concatenation")
    print("\nA claim that the architecture advances the state of the art needs the")
    print("second column to be positive and significant on more than one dataset.")
else:
    print("Only this dataset has been run so far.")
    print("Run the other benchmark notebooks and gather their benchmark_row_*.json")
    print("files here to build the cross-dataset table.")

---

## Interpreting the benchmark

Two questions, and they have different answers:

**Did DERM-Net win?** Read `dermnet_rank`. Rank 1 on a dataset means it beat every other
model there under an identical budget. Rank 1 on all four would be a strong result.

**Did the architecture's novel component earn its place?** Read `msca_gain_pp` and its
Holm-adjusted p value. This is the comparison a reviewer will look for, because it is the
only one that tests the paper's actual contribution rather than the value of large
pretrained backbones.

A result where DERM-Net ranks first but does not significantly beat its own no-MSCA
ablation says the backbones are doing the work. That is still a usable paper, but the
claim has to change to match it.

## Caveats to state in any write-up

- Absolute numbers here are not comparable to published leaderboard results for these
  datasets, which use different splits, budgets and often no cross-validation.
- Fitzpatrick17k and DDI publish no lesion or patient identifier, so splitting there is at
  image level and may be optimistic. Only HAM10000 supports group-aware splitting.
- A single training budget is applied to every model. A larger budget could favour the
  larger models; this is stated rather than tuned away, because per-model tuning on the
  same data is how benchmarks become unfair.